<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap07/cap07_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 7 Classificazione delle Immagini e Riconoscimento di Pattern

Nel **Capitolo 6**, la transizione dalla Parte I alla Parte II è stata presentata attraverso due applicazioni che richiedevano già decisioni automatizzate: il riconoscimento dei segni nei fogli di risposta (OMR) e il rilevamento dei difetti nell'ispezione industriale. In entrambi i casi, tuttavia, le decisioni dipendevano da regole geometriche e soglie definite manualmente, come determinare se un disco fosse sufficientemente circolare o se una regione fosse abbastanza scura.

Questo capitolo formalizza il problema più generale alla base di queste applicazioni: dato un insieme di esempi etichettati, come addestrare un sistema per **classificare automaticamente** nuove immagini o regioni di interesse? Questa questione è al centro del **Riconoscimento di Pattern**, disciplina che fonda gran parte dei compiti moderni della Visione Artificiale, dalla classificazione delle immagini al rilevamento degli oggetti e alla segmentazione semantica, esplorati nei prossimi capitoli.

Verranno studiati i principali **descrittori classici dell'immagine** (colore, tessitura e forma/gradiente) e il classificatore **k-Nearest Neighbors** (*k-Nearest Neighbors* — k-NN), scelto per la sua semplicità concettuale e per evidenziare, in modo diretto, la relazione tra lo spazio delle caratteristiche, le metriche di distanza e le frontiere di decisione — concetti che rimangono centrali anche nei classificatori basati su reti neurali profonde, studiati nel capitolo finale di questa parte.

## 7.1 Obiettivi del Capitolo

Al termine di questo capitolo, lo studente dovrebbe essere in grado di:

* **Comprendere il *pipeline* classico di riconoscimento dei pattern**: acquisizione, pre-elaborazione, estrazione dei descrittori, classificazione e valutazione;
* **Estrarre e interpretare descrittori classici** di colore, tessitura (*Local Binary Patterns* — LBP) e forma/gradiente (*Histogram of Oriented Gradients* — HOG);
* **Implementare e addestrare un classificatore k-NN** per compiti di classificazione delle immagini;
* **Valutare i classificatori** attraverso metriche come accuratezza, matrice di confusione, precisione e recall;
* **Analizzare l'effetto del parametro k** e della dimensionalità dello spazio delle caratteristiche sulle prestazioni del classificatore;
* **Riconoscere i limiti dei descrittori artigianali** (*hand-crafted features*) e comprendere la motivazione per la transizione, nei prossimi capitoli, verso descrittori appresi automaticamente.

## 7.2 Configurazione dell'Ambiente

Gli esempi di questo capitolo utilizzano librerie ampiamente impiegate
nell'Elaborazione Digitale delle Immagini, nella Visione Artificiale e nell'Apprendimento
Automatico. Il blocco seguente installa i pacchetti necessari; in ambienti che già
li possiedono, l'esecuzione può essere ignorata.

In [1]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup()
from morph import mm

import importlib
import subprocess
import sys

def setup_cap07():
    """Installa le librerie mancanti necessarie per questo capitolo
    (visione computazionale e apprendimento automatico)."""
    pacotes = {
        "cv2": "opencv-python",
        "skimage": "scikit-image",
        "numpy": "numpy",
        "sklearn": "scikit-learn",
        "matplotlib": "matplotlib",
        "pandas": "pandas",
        "seaborn": "seaborn",
        "tabulate": "tabulate",
        "kaleido": "kaleido",
    }
    for modulo, pacote in pacotes.items():
        if importlib.util.find_spec(modulo) is None:
            resultado = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", pacote]
            )
            if resultado.returncode != 0:
                print(f"[AVVISO] Impossibile installare {pacote} (necessario per il modulo {modulo}).")


setup_cap07()

# ==========================================================
# Librerie
# ==========================================================

# Calcolo scientifico
import numpy as np
import pandas as pd

# Visualizzazione
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import seaborn as sns

# Visione computazionale
import cv2
from skimage import data as skdata
from skimage.feature import hog, local_binary_pattern

# Apprendimento automatico
from sklearn.datasets import load_digits, make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

✅ Ambiente pronto. Morph: 1.1.9 | OpenCV: 5.0.0


## 7.3 Un Problema Concreto: Classificazione della Frutta

Prima di presentare i fondamenti teorici, si consideri il seguente problema,
che verrà utilizzato come esempio lungo tutto questo capitolo per illustrare i
principali concetti del riconoscimento di pattern.

**Lo scenario:** Una fattoria automatizzata utilizza un sistema di Visione
Artificiale per separare mele, banane e arance sulle linee di
confezionamento.

**La sfida:** I frutti arrivano sul nastro in diverse posizioni e
orientamenti, in condizioni di illuminazione che possono variare. Inoltre,
foglie, ombre e piccole occlusioni possono rendere difficile la loro
identificazione. Come sviluppare un sistema in grado di classificarli
correttamente?

**Un possibile approccio:**

1. Estrarre descrittori che rappresentino caratteristiche rilevanti dei
   frutti:
   - **Colore:** distribuzione predominante dei colori;
   - **Texture:** differenze nella superficie della buccia;
   - **Forma:** caratteristiche geometriche del contorno.

2. Addestrare un classificatore utilizzando esempi precedentemente etichettati.

3. Utilizzare il modello addestrato per classificare automaticamente nuovi
   frutti.

La [Figura 7.1](#fig-07-frutas-motivacao) illustra, in modo concettuale, come diversi
frutti possano essere rappresentati in uno spazio delle caratteristiche
tridimensionale.

> ### 💡 Rifletti prima di continuare
>
> Se ogni frutto fosse rappresentato solo dai valori di intensità dei
> suoi pixel, sarebbe possibile distinguerli in modo affidabile? Che tipi di
> informazioni potrebbero essere estratti dall'immagine per facilitare questo
> compito?

In [2]:
np.random.seed(42)

centros = {
    "Maçã":    [0.8, 0.2, 0.9],
    "Banana":  [0.3, 0.1, 0.2],
    "Laranja": [0.9, 0.8, 0.8],
}

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="3d")

for fruta, centro, cor, marcador in zip(
    centros,
    centros.values(),
    ["red", "gold", "orange"],
    ["o", "s", "^"],
):
    X = np.clip(np.random.normal(centro, 0.08, (70, 3)), 0, 1)
    ax.scatter(X[:, 0], X[:, 1], X[:, 2],
               c=cor, marker=marcador, s=35,
               alpha=0.7, label=fruta)

ax.set(
    xlim=(0,1), ylim=(0,1), zlim=(0,1),
    xlabel="Intensidade de cor",
    ylabel="Textura",
    zlabel="Forma",
    title="Espaço de Características"
)
ax.view_init(elev=25, azim=-60)
ax.legend(title="Frutas")
ax.zaxis.labelpad = 0.01

plt.tight_layout()
plt.show()

<Figure size 1500x1500 with 1 Axes>

**Figura 7.1:** Esempio motivazionale: diversi frutti che formano raggruppamenti distinti in uno spazio delle caratteristiche.


## 7.4 🗺️ Panoramica del Capitolo: Il *Pipeline* Classico di Classificazione delle Immagini

Prima di procedere, è utile presentare una visione integrata di ciò che verrà
studiato. La [Figura 7.2](#fig-07-infografo) mostra il flusso generale di un sistema
classico di classificazione delle immagini, dall'immagine di input fino alla
fase di assegnazione dell'etichetta finale. Nelle prossime sezioni, ciascuna delle
fasi di questo processo verrà studiata in dettaglio.

<figure id="fig-07-infografo" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-07-infografo.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figura 7.2:</strong> Panoramica del *pipeline* classico di classificazione delle immagini: estrazione dei descrittori (LBP, HOG), formazione dello spazio delle caratteristiche e classificazione tramite k-NN. Non si applica ai modelli di Deep Learning (CNN, YOLO), che apprendono *feature end-to-end* direttamente dai pixel. **Fonte:** elaborato con l'ausilio di Gemini Notebook ({GOOGLE}, 2025).</figcaption>
</figure>

> ### 📝 Ambito di questo capitolo
>
> In questo capitolo, l'attenzione ricade esclusivamente sul classificatore k-NN,
> per la sua semplicità didattica e per illustrare in modo intuitivo il
> concetto di spazio delle caratteristiche. Altri classificatori
> tradizionali ampiamente utilizzati nel Riconoscimento di Pattern — come
> Alberi Decisionali, Regole di Classificazione e Macchine a Vettori di
> Supporto (SVM) — sono discussi in profondità in
> Quilici-gonzalez (2014), in particolare nella sua 2ª edizione, attualmente
> in produzione (QUILICI-GONZALEZ, 2026).

## 7.5 Fondamenti del Riconoscimento di Pattern

Un sistema di **riconoscimento di pattern** ha come obiettivo assegnare una
categoria (etichetta) a un'osservazione — un'immagine intera, una regione di
interesse o un segnale — basandosi su esempi precedentemente etichettati. In
generale, questo processo è organizzato nelle seguenti fasi:

1. **Acquisizione:** ottenimento dell'immagine o del segnale da classificare;
2. **Pre-elaborazione:** normalizzazione, rimozione del rumore, correzione
   geometrica o dell'illuminazione — fasi già studiate nei capitoli
   precedenti;
3. **Estrazione di descrittori (*feature*):** trasformazione dell'immagine in
   un **vettore di caratteristiche** di dimensione fissa, che rappresenta le
   proprietà rilevanti per il compito di classificazione;
4. **Classificazione:** applicazione di un modello che associa il vettore di
   caratteristiche a una classe;
5. **Valutazione:** analisi delle prestazioni del modello su un insieme di dati
   indipendente da quello utilizzato per l'addestramento.

L'insieme di tutti i vettori di caratteristiche possibili costituisce lo
**spazio delle caratteristiche** (*feature space*). Un buon descrittore produce
rappresentazioni che avvicinano, in questo spazio, osservazioni della stessa
classe e allontanano osservazioni di classi distinte. Questa proprietà favorisce
metodi di classificazione basati sulla prossimità, come il **k-NN**, e
avvantaggia anche diversi altri classificatori.

La [Figura 7.1](#fig-07-frutas-motivacao) illustra questo concetto in modo schematico:
ogni frutto è rappresentato da un punto in uno spazio delle caratteristiche a
tre dimensioni (colore, consistenza e forma). Sebbene questo spazio sia solo una
semplificazione didattica, esso mostra come i campioni della stessa classe
tendano a formare raggruppamenti, mentre classi diverse occupano regioni
distinte, facilitando il compito di classificazione.

## 7.6 Estrazione di Descrittori Classici

Prima della popolarizzazione delle reti neurali profonde, i descrittori
di immagine erano, nella maggior parte dei casi, progettati manualmente
da esperti (*hand-crafted features*), sulla base di proprietà
statistiche o geometriche note. Tre famiglie classiche sono
particolarmente rilevanti:

* **Descrittori di colore:** istogrammi di intensità o di tinta,
  che catturano la distribuzione dei valori cromatici di una regione,
  già introdotti nel **Capitolo 3** tramite la funzione `mm.hist`;
* **Descrittori di texture:** catturano pattern locali di ripetizione,
  rugosità o orientamento, come il *Local Binary Patterns* (LBP),
  studiato in seguito;
* **Descrittori di forma/gradiente:** descrivono la distribuzione dei
  bordi e delle orientazioni del gradiente, come l'*Histogram of
  Oriented Gradients* (HOG), ampiamente impiegato nel rilevamento di
  persone e altri oggetti.

Per confrontare l'informazione catturata da ciascun approccio, la [Figura 7.3](#fig-07-visualizacao-descritores)
mostra come diverse tecniche "vedono" la stessa immagine.

In [3]:
# Caricare immagine di esempio
imagem = skdata.camera()

# Applicare descrittori
lbp_img = local_binary_pattern(
    imagem, 
    P=8, 
    R=1, 
    method="uniform"
    )

# Convertire l'LBP in RGB solo per facilitare la visualizzazione
lbp_norm = (lbp_img - lbp_img.min()) / (lbp_img.max() - lbp_img.min() + 1e-8)
lbp_rgb = (cm.nipy_spectral(lbp_norm)[..., :3] * 255).astype("uint8")

hog_features, hog_img = hog(
    imagem,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,
)

# Visualizzazione standard
mm.show(
    [imagem, lbp_rgb, hog_img],
    titles=[
        "Immagine Originale\n(come la vede l'uomo)",
        "LBP: Texture\n(ogni colore = un codice LBP)",
        "HOG: Gradient e Contorni\n(regioni chiare = maggiore intensità)",
    ],
    cols=3,
    figsize=(12, 4),
)

print("Osserva come ogni descrittore evidenzia proprietà diverse:")
print("• LBP: evidenzia pattern locali di texture.")
print("• HOG: evidenzia contorni e orientamenti dei bordi.")
print("• Immagine originale: contiene solo i valori di intensità.")

<Figure size 1800x600 with 3 Axes>

**Figura 7.3:** Confronto visivo di diversi descrittori applicati alla stessa immagine. Ogni descrittore rivela aspetti distinti della scena.


Osserva come ogni descrittore evidenzia proprietà diverse:
• LBP: evidenzia pattern locali di texture.
• HOG: evidenzia contorni e orientamenti dei bordi.
• Immagine originale: contiene solo i valori di intensità.


### 7.6.1 *Local Binary Patterns* (LBP)

L'LBP è un descrittore di tessitura che codifica, per ogni pixel centrale
$g_c$, la relazione tra la sua intensità e quella dei $P$ vicini disposti in
un intorno circolare di raggio $R$:

$$
\mathrm{LBP}_{P,R}(x_c, y_c) = \sum_{p=0}^{P-1} s(g_p - g_c)\, 2^p,
\qquad
s(z) =
\begin{cases}
1, & z \geq 0 \\
0, & z < 0
\end{cases}
$$

dove:

- $(x_c, y_c)$ sono le coordinate del pixel centrale;
- $g_c$ è l'intensità del pixel centrale;
- $g_p$ è l'intensità del $p$-esimo pixel vicino;
- $P$ è il numero di vicini considerati;
- $R$ è il raggio dell'intorno circolare;
- $p$ è l'indice del vicino, con $p = 0, 1, \ldots, P-1$;
- $s(z)$ è la funzione soglia definita nell'equazione, dove $z = g_p - g_c$;
  essa assume valore 1 quando $z \geq 0$ e 0 quando $z < 0$;
- $2^p$ corrisponde al peso binario associato al $p$-esimo vicino.

Il codice LBP ottenuto descrive il pattern locale di contrasto attorno al pixel. L'istogramma di questi codici forma un vettore di caratteristiche compatto per rappresentare la tessitura dell'immagine ([Figura 7.4](#fig-07-vetor-lbp)). In questo capitolo si utilizza la variante **uniforme**, che raggruppa i pattern non uniformi in un'unica categoria, riducendo la dimensionalità e aumentando la robustezza del descrittore.

In [4]:
plt.figure(figsize=(6, 4))

plt.hist(
    lbp_img.ravel(),
    bins=np.arange(-0.5, lbp_img.max() + 1.5, 1),
    density=True,
    edgecolor="black",
)

plt.title("Histograma dos códigos LBP")
plt.xlabel("Código LBP")
plt.ylabel("Frequência relativa")
plt.xticks(range(int(lbp_img.max()) + 1))
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 1800x1200 with 1 Axes>

**Figura 7.4:** Istogramma dei codici LBP dell


> ### 💡 Funzione `local_binary_pattern`
>
> L'implementazione utilizzata in questo capitolo è fornita dalla libreria
> `scikit-image`:
>
> ```python
> local_binary_pattern(
>     imagem,
>     P=8,
>     R=1,
>     method="uniform"
> )
> ```
>
> dove:
>
> * `image`: immagine in scala di grigi;
> * `P`: numero di vicini equidistanti nell'intorno circolare;
> * `R`: raggio dell'intorno, in pixel;
> * `method`: strategia di codifica. In questo capitolo si utilizza il valore
>   `"uniform"`.
>
> L'equazione presentata in precedenza descrive l'**LBP originale**. Nell'
> implementazione adottata in questo capitolo, l'opzione `method="uniform"`
> calcola inizialmente tale codice e, successivamente, rimappa i pattern non
> uniformi in un'unica categoria, riducendo la dimensionalità del
> descrittore e rendendolo più robusto a piccole variazioni locali.

La [Figura 7.3](#fig-07-visualizacao-descritores) presenta la rappresentazione visiva dell'LBP, mentre la [Figura 7.4](#fig-07-vetor-lbp) mostra l'istogramma dei codici LBP utilizzato come vettore di caratteristiche.

Il **Progetto Pratico 2** (sezione **Confronto di Descrittori per la Classificazione di Trame**) impiega l'LBP nella classificazione di diversi tipi di trama sintetica.

### 7.6.2 *Histogram of Oriented Gradients* (HOG)

L'HOG è un descrittore che rappresenta la forma di un oggetto tramite la distribuzione delle orientazioni del gradiente locale. Come nell'operatore di Canny (**Capitolo 6**), si calcola inizialmente il gradiente:

$$
|\nabla f(x,y)| =
\sqrt{\left(\frac{\partial f}{\partial x}\right)^2 +
      \left(\frac{\partial f}{\partial y}\right)^2},
\qquad
\theta(x,y) =
\operatorname{atan2}\!\left(
\frac{\partial f}{\partial y},
\frac{\partial f}{\partial x}
\right).
$$

dove:

- $f(x,y)$ è l'intensità dell'immagine nel pixel $(x,y)$;
- $\frac{\partial f}{\partial x}$ e $\frac{\partial f}{\partial y}$ sono, rispettivamente, le derivate parziali dell'immagine nelle direzioni orizzontale e verticale;
- $|\nabla f(x,y)|$ è la magnitudine del vettore gradiente nel pixel $(x,y)$, che indica l'intensità della variazione locale dell'immagine;
- $\theta(x,y)$ è l'orientazione del vettore gradiente nel pixel $(x,y)$, calcolata tramite la funzione $\operatorname{atan2}$, il cui risultato appartiene all'intervallo $(-\pi,\pi]$.

Sebbene $\theta(x,y)$, come calcolata dalla funzione $\operatorname{atan2}$, appartenga all'intervallo $(-\pi,\pi]$, l'implementazione standard dell'HOG utilizza il **gradiente non firmato** (*unsigned*): orientazioni opposte (ad esempio, $0$ e $\pi$) vengono trattate come equivalenti, e gli angoli vengono mappati nell'intervallo $[0,\pi)$ prima della costruzione dell'istogramma. Questa scelta rende il descrittore invariante alla direzione del contrasto (ad esempio, un bordo chiaro-scuro e un bordo scuro-chiaro producono la stessa orientazione).

L'immagine viene quindi suddivisa in **celle** (*cells*). Per ciascuna cella, si costruisce un istogramma delle orientazioni del gradiente, ponderato dalla magnitudine corrispondente. La concatenazione degli istogrammi di tutte le celle forma il vettore di caratteristiche HOG, che rappresenta la distribuzione spaziale delle orientazioni del gradiente e cattura informazioni sulla forma e sui contorni dell'oggetto ([Figura 7.5](#fig-07-vetor-hog)).

In [5]:
n = 100

plt.figure(figsize=(8, 3))
plt.bar(
    range(n),
    hog_features[:n],
    width=0.9
)

plt.title("Primeiros componentes do vetor HOG")
plt.xlabel(f"Índice do componente (0–{n-1}, de um total de {hog_features.shape[0]})")
plt.ylabel("Valor normalizado")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 2400x900 with 1 Axes>

**Figura 7.5:** Primi 100 componenti del vettore di caratteristiche HOG.


> ### 💡 Funzione `hog`
>
> L'estrazione del descrittore HOG è realizzata dalla funzione:
>
> ```python
> hog(
>     image,
>     orientations=9,
>     pixels_per_cell=(8, 8),
>     cells_per_block=(2, 2),
>     visualize=True,
> )
> ```
>
> I parametri principali sono:
>
> * `image`: immagine di input;
> * `orientations`: numero di suddivisioni angolari dell'istogramma delle orientazioni in ogni cella;
> * `pixels_per_cell`: dimensione, in pixel, di ogni cella in cui viene calcolato l'istogramma;
> * `cells_per_block`: numero di celle utilizzate per la normalizzazione del descrittore;
> * `visualize`: quando `True`, restituisce anche un'immagine che illustra i gradienti utilizzati dall'HOG.
>
> L'equazione presentata in precedenza descrive il calcolo della magnitudine e dell'orientazione del gradiente, che costituiscono la base del descrittore HOG. Nell'implementazione adottata in questo capitolo, la funzione `hog()` utilizza queste informazioni per costruire istogrammi delle orientazioni in ogni cella dell'immagine e, successivamente, esegue la normalizzazione a blocchi (`cells_per_block`), riducendo la sensibilità del descrittore alle variazioni di illuminazione e contrasto.


La [Figura 7.3](#fig-07-visualizacao-descritores) presenta la rappresentazione visiva dell'HOG, mentre la [Figura 7.5](#fig-07-vetor-hog) illustra le prime componenti del vettore di caratteristiche estratto dall'immagine.

Il **Progetto Pratico 1** (sezione **Classificazione di Cifre Manoscritte con k-NN**) confronta le prestazioni dei descrittori HOG con l'uso diretto delle intensità dei pixel come vettore di caratteristiche.

### 7.6.3 L'Impatto della Scala e la Normalizzazione delle Caratteristiche

Il classificatore $k$-NN prende le sue decisioni basandosi sulla distanza tra i
vettori di caratteristiche. Per questo motivo, la scala di ciascuna caratteristica
influenza direttamente il risultato della classificazione. Se una variabile
presenta valori molto maggiori rispetto alle altre (ad esempio, un'
intensità di colore che varia da $0$ a $255$, mentre un indice di
circolarità varia da $0$ a $1$), essa tende a dominare il calcolo della
distanza, riducendo l'influenza degli altri descrittori.

Per evitare questo problema, si applica una fase di **normalizzazione delle
caratteristiche**, generalmente tramite la standardizzazione (*Z-score
standardization*). In questa procedura, ciascuna caratteristica assume
media pari a zero e deviazione standard pari a uno, rendendo comparabili
grandezze originariamente misurate su scale diverse.

La standardizzazione viene effettuata tramite la trasformazione

$$
z = \frac{x - \mu}{\sigma},
$$

in cui:

- $x$ è il valore originale della caratteristica;
- $\mu$ è la media di tale caratteristica calcolata sull'insieme di addestramento;
- $\sigma$ è la deviazione standard della caratteristica;
- $z$ è il valore standardizzato.

Dopo questa trasformazione, tutte le caratteristiche possiedono media
pari a zero e deviazione standard pari a uno, consentendo loro di contribuire in
modo equilibrato al calcolo delle distanze.

> ### 💡 Classe `StandardScaler`
>
> La standardizzazione utilizzata in questo capitolo viene effettuata tramite la classe
> `StandardScaler`, della libreria `scikit-learn`:
>
> ```python
> from sklearn.preprocessing import StandardScaler
>
> scaler = StandardScaler()
> X_norm = scaler.fit_transform(X)
> ```
>
> in cui:
>
> * `StandardScaler()`: crea l'oggetto responsabile della standardizzazione;
> * `fit_transform(X)`: calcola la media e la deviazione standard di ciascuna
>   caratteristica dell'insieme `X` e restituisce la matrice standardizzata.
>
> In pratica, il metodo `fit_transform()` esegue due fasi: prima
> (`fit`), stima la media ($\mu$) e la deviazione standard ($\sigma$) di ogni
> caratteristica; successivamente (`transform`), applica la trasformazione di standardizzazione presentata in precedenza a tutti i valori della matrice di input.

La [Figura 7.6](#fig-07-normalizacao-features) mostra l'effetto della normalizzazione.
Visivamente, la distribuzione dei punti rimane la stessa; ciò che cambia è la
scala degli assi. Senza normalizzazione, la caratteristica di maggiore
magnitudine domina il calcolo delle distanze tra i campioni. Dopo la standardizzazione, tutte le caratteristiche contribuiscono in
modo equilibrato al calcolo delle distanze utilizzate dal
classificatore $k$-NN.

In [6]:
# Dati sintetici con scale molto diverse
np.random.seed(42)
X_demo = np.random.randn(20, 2) * [100, 1]
y_demo = np.array([0] * 10 + [1] * 10)

print("Effetto della normalizzazione:")
print("  Caratteristica 1: scala ≈ 100")
print("  Caratteristica 2: scala ≈ 1")
print("\nSenza normalizzazione, la prima caratteristica domina il calcolo delle distanze.")
print("Con la normalizzazione, entrambe contribuiscono in modo equilibrato.")
print("\nLa normalizzazione è essenziale quando le caratteristiche hanno scale diverse.")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Senza normalizzazione
axes[0].scatter(
    X_demo[y_demo == 0, 0], X_demo[y_demo == 0, 1],
    c="blue", label="Classe 0"
)
axes[0].scatter(
    X_demo[y_demo == 1, 0], X_demo[y_demo == 1, 1],
    c="red", label="Classe 1"
)
axes[0].set_title("Sem Normalização\n(escalas diferentes)")
axes[0].set_xlabel("Característica 1 (escala 100)")
axes[0].set_ylabel("Característica 2 (escala 1)")
axes[0].legend()

# Con normalizzazione
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_demo)

axes[1].scatter(
    X_norm[y_demo == 0, 0], X_norm[y_demo == 0, 1],
    c="blue", label="Classe 0"
)
axes[1].scatter(
    X_norm[y_demo == 1, 0], X_norm[y_demo == 1, 1],
    c="red", label="Classe 1"
)
axes[1].set_title("Com Normalização\n(características balanceadas)")
axes[1].set_xlabel("Característica 1")
axes[1].set_ylabel("Característica 2")
axes[1].legend()

plt.tight_layout()
plt.show()

Effetto della normalizzazione:
  Caratteristica 1: scala ≈ 100
  Caratteristica 2: scala ≈ 1

Senza normalizzazione, la prima caratteristica domina il calcolo delle distanze.
Con la normalizzazione, entrambe contribuiscono in modo equilibrato.

La normalizzazione è essenziale quando le caratteristiche hanno scale diverse.


<Figure size 3000x1200 with 2 Axes>

**Figura 7.6:** Importanza della normalizzazione delle caratteristiche per il classificatore k-NN.


## 7.7 📌 Mappa Concettuale

Fino a qui sono stati presentati i descrittori classici (LBP, HOG, pixel grezzi) e il modo in cui organizzano i campioni in uno spazio delle caratteristiche. La [Figura 7.7](#fig-07-mapa-conceitual) sintetizza questo percorso e colloca tali fasi all'interno del flusso generale di un sistema classico di classificazione delle immagini, indicando anche le fasi successive — classificazione (k-NN) e valutazione dei risultati — che verranno formalizzate nelle sezioni seguenti.

<figure id="fig-07-mapa-conceitual" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-07-mapa-conceitual.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figura 7.7:</strong> Mappa concettuale del processo di classificazione delle immagini utilizzando descrittori classici (LBP, HOG, pixel grezzi) e classificatore tradizionale (k-NN). Non si applica a modelli di Apprendimento Profondo (CNN, YOLO), che apprendono *feature end-to-end* direttamente dai pixel.</figcaption>
</figure>

## 7.8 Descrittori nella Pratica

Dopo aver conosciuto i principali descrittori classici, è naturale chiedersi
come essi influenzino le prestazioni di un classificatore in situazioni
vicine a quelle riscontrabili nella pratica.

In questa sezione, si confronta l'uso di tre diverse rappresentazioni delle
stesse immagini: intensità dei pixel, descrittori LBP e descrittori
HOG. Per rendere l'esperimento più realistico, si aggiunge rumore sintetico
ai dati, con intensità diverse per ciascun descrittore — una forma
semplificata di simulare il fatto che, nella pratica, diverse
rappresentazioni tollerano in modo diseguale le imperfezioni dell'acquisizione
(rumore del sensore, piccole variazioni di posizione, ecc.).

La [Figura 7.8](#fig-07-matrizes-confusao) presenta le matrici di confusione ottenute
per ciascun descrittore, consentendo di identificare in quali classi si verificano i
principali errori di classificazione. 
L'interpretazione di queste matrici è stata introdotta nel **Capitolo 1**, quando
sono stati presentati i concetti di **Vero Positivo (VP)**,
**Falso Positivo (FP)**, **Vero Negativo (VN)** e **Falso Negativo (FN)**.
Questi concetti sono stati approfonditi negli **EP 01_02** (metriche di classificazione)
e **01_03** (*mean Average Precision* – mAP), disponibili su:

- <https://fzampirolli.github.io/pdi-vc/eps/py.pt/EP01_02.html>
- <https://fzampirolli.github.io/pdi-vc/eps/py.pt/EP01_03.html>

In questo capitolo, le matrici di confusione sono impiegate per analizzare
come diversi descrittori influenzino le prestazioni del classificatore.

Successivamente, la [Figura 7.9](#fig-07-comparacao-descritores-detalhada) riassume l'accuratezza
globale ottenuta da ciascun descrittore.

I risultati mostrano che le prestazioni del classificatore dipendono
direttamente dalla rappresentazione scelta per descrivere le immagini.
Mentre l'uso diretto delle intensità dei pixel è più sensibile alle
degradazioni introdotte, i descrittori LBP e HOG preservano meglio le
informazioni rilevanti per la classificazione, risultando in maggiori
prestazioni in questo scenario. È importante sottolineare che i livelli di rumore
applicati a ciascun descrittore sono stati scelti solo a scopo didattico,
in modo da illustrare il principio generale secondo cui descrittori più elaborati
*possono* essere più robusti alle degradazioni — il che non significa che questa
relazione si verifichi sempre, come dimostrerà il caso di studio della prossima
sezione.

> ### 💡 Come viene eseguito l'esperimento
>
> Poiché l'obiettivo di questa sezione è confrontare esclusivamente l'effetto dei descrittori,
> viene generato un insieme di dati sintetico semplice: tre nubi di punti
> gaussiani, centrati sugli stessi valori di "colore, texture e forma" già
> utilizzati nella [Figura 7.1](#fig-07-frutas-motivacao) — lo stesso schema impiegato fin dall'inizio
> del capitolo per rappresentare le tre classi di frutta.
>
> ```python
> import numpy as np
> from sklearn.model_selection import train_test_split
> from sklearn.neighbors import KNeighborsClassifier
> from sklearn.metrics import accuracy_score, confusion_matrix
> ```
>
> ```python
> centri = {
>     "Mela":     [0.8, 0.2, 0.9],
>     "Banana":   [0.3, 0.1, 0.2],
>     "Arancia":  [0.9, 0.8, 0.8],
> }
>
> n_per_classe = 100
> X = np.vstack([
>     np.random.normal(centro, 0.12, (n_per_classe, 3))
>     for centro in centri.values()
> ])
> y = np.repeat(list(centri.keys()), n_per_classe)
> ```
>
> dove:
>
> * `centri`: dizionario con il punto medio di ciascuna classe nello spazio delle caratteristiche (colore, texture, forma);
> * `n_per_classe`: numero di campioni generati per classe;
> * `np.random.normal(centro, 0.12, (n_per_classe, 3))`: genera `n_per_classe` campioni attorno a ciascun centro, con deviazione standard 0,12 in ogni dimensione;
> * `np.repeat(list(centri.keys()), n_per_classe)`: genera il vettore delle etichette corrispondenti, nello stesso ordine dei centri.
>
> Successivamente, si utilizza il flusso di addestramento e valutazione:
>
> * `train_test_split(X, y, test_size=0.3)`: divide i dati in addestramento (70%) e test (30%);
> * `KNeighborsClassifier(n_neighbors=5)`: crea un classificatore $k$-NN con $k=5$ vicini;
> * `fit(X_train, y_train)`: adatta il modello ai dati di addestramento;
> * `predict(X_test)`: classifica i campioni di test;
> * `accuracy_score(y_test, y_pred)`: calcola l'accuratezza;
> * `confusion_matrix(y_test, y_pred)`: genera la matrice di confusione.
>
> In questo esperimento, il set di addestramento, il classificatore e il metodo
> di valutazione rimangono esattamente gli stessi. L'unica differenza tra gli
> esperimenti è la rappresentazione utilizzata per ciascuna immagine (pixel
> grezzi, LBP o HOG), consentendo di valutare esclusivamente l'influenza del
> descrittore sulle prestazioni del classificatore.

In [7]:
classes = ["Maçã", "Banana", "Laranja"]

np.random.seed(42)

# Stessi centri di classe (colore, consistenza, forma) utilizzati nel
# esempio precedente, ora riutilizzati per generare i dati
# sintetici di addestramento e test di questo esperimento.
centros = {
    "Maçã":    [0.8, 0.2, 0.9],
    "Banana":  [0.3, 0.1, 0.2],
    "Laranja": [0.9, 0.8, 0.8],
}

n_por_classe = 100
X = np.vstack([
    np.random.normal(centro, 0.12, (n_por_classe, 3))
    for centro in centros.values()
])
y = np.repeat(list(centros.keys()), n_por_classe)

# Simulare descrittori con diversi livelli di sensibilità al rumore.
# Maggiore è il rumore aggiunto, peggiore tende a essere la rappresentazione.
descritores = {
    "Pixels Brutos": X + 0.5 * np.random.randn(*X.shape),
    "LBP":           X + 0.3 * np.random.randn(*X.shape),
    "HOG":           X + 0.2 * np.random.randn(*X.shape),
}

# Creare un'unica figura con 3 sottotrame affiancate per le matrici
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
resultados = {}

for idx, (nome, Xd) in enumerate(descritores.items()):
    X_train, X_test, y_train, y_test = train_test_split(Xd, y, test_size=0.3, random_state=42)
    
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    # Matrice di confusione nel sottotrama corrispondente

    cm = confusion_matrix(y_test, y_pred, labels=classes)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=classes,
        yticklabels=classes,
        ax=axes[idx],
    )

    axes[idx].set_title(
        f"{nome}\nAcurácia: {acc:.3f}",
        fontsize=11,
        fontweight="bold"
    )
    axes[idx].set_xlabel("Classe Predita")
    axes[idx].set_ylabel("Classe Real")


plt.tight_layout()
plt.show()


<Figure size 4200x1200 with 3 Axes>

**Figura 7.8:** Matrici di confusione ottenute dal classificatore k-NN utilizzando tre descrittori diversi. Le righe rappresentano la classe reale (Mela, Banana e Arancia) e le colonne la classe predetta. Maggiore è la concentrazione di valori sulla diagonale principale, migliore è la prestazione del descrittore.


In [8]:
# Confronto visivo in una figura isolata
plt.figure(figsize=(6, 3.5))
nomes = list(resultados.keys())
acuracia = list(resultados.values())
colors = ['#6366f1', '#f97316', '#22c55e']

bars = plt.bar(nomes, acuracia, color=colors, width=0.5)
plt.ylabel('Acurácia Global')
plt.title('Desempenho Geral dos Descritores sob Ruído Realista', fontsize=12, fontweight='bold')
plt.ylim(0.5, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Aggiungere i valori sopra le barre usando il round standard per la visualizzazione
for bar, val in zip(bars, acuracia):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print("Analisi dei risultati:")
print("- Pixel Grezzi: Sensibili a variazioni locali di illuminazione e rumore.")
print("- LBP: Buona tolleranza per variazioni monotoniche dell'illuminazione globale.")
print("- HOG: Eccellente per contorni e forme stabili sotto piccole fluttuazioni geometriche.")

<Figure size 1800x1050 with 1 Axes>

**Figura 7.9:** Confronto dettagliato dell


Analisi dei risultati:
- Pixel Grezzi: Sensibili a variazioni locali di illuminazione e rumore.
- LBP: Buona tolleranza per variazioni monotoniche dell'illuminazione globale.
- HOG: Eccellente per contorni e forme stabili sotto piccole fluttuazioni geometriche.


## 7.9 Un Problema Concreto: Simulando Descrittori di Frutta

Riprendendo il problema di classificazione della frutta presentato all'inizio del capitolo, ogni immagine può essere rappresentata da un vettore di caratteristiche (*feature vector*) ottenuto tramite l'estrazione di descrittori di colore, texture e forma. La [Tabela 7.1](#tbl-descritores-frutas) presenta alcuni descrittori frequentemente utilizzati nelle applicazioni di Visione Computazionale, incluse tecniche introdotte nel Capitolo 3 e in questo capitolo.

<a id="tbl-descritores-frutas"></a>

**Tabela 7.1:** Insieme di descrittori cromatici, testurali e geometrici utilizzati per rappresentare immagini di frutta.

| Caratteristica       | Descrizione                                                |
|----------------------|------------------------------------------------------------|
| R, G, B              | intensità media dei canali rosso, verde e blu              |
| NC                   | intensità media in livelli di grigio (*grayscale*)         |
| LBP                  | descrittore di texture (*Local Binary Pattern*)            |
| HOG                  | descrittore di forma (*Histogram of Oriented Gradients*)   |
| Area                 | numero di pixel dell'oggetto                               |
| Perimetro            | lunghezza del contorno                                     |
| Circolarità          | misura di quanto è circolare l'oggetto                     |
| Rapporto larghezza/altezza | proporzione tra larghezza e altezza della regione      |


In questo esempio, ogni immagine è rappresentata dal vettore

$$
X=(R,G,B,NC,\text{LBP},\text{HOG},\text{Area},\text{Perimetro},\text{Circolarità},\text{Rapporto}).
$$

> ### 📝 Semplificazione adottata in questa tabella
>
> Nella pratica, LBP e HOG non sono valori scalari, ma istogrammi con decine o centinaia di componenti. In questa sezione, ciascuno di essi è rappresentato da un unico valore solo per semplificare la presentazione. Nelle applicazioni reali, queste posizioni sarebbero sostituite dalle componenti complete dei rispettivi istogrammi.

Nelle applicazioni reali, non tutti i descrittori contribuiscono in egual misura a distinguere le classi. Alcuni forniscono informazioni più rilevanti, mentre altri possono essere ridondanti o poco discriminativi.

Per riprodurre questo scenario in modo controllato, si utilizzerà `make_classification()`, dalla libreria `scikit-learn`. La funzione genera un insieme di dati sintetico le cui caratteristiche possono essere interpretate come descrittori di immagini, consentendo di definire quante di esse saranno informative per la classificazione.

In questo esempio, vengono generate dieci caratteristiche sintetiche, delle quali solo sette (`n_informative=7`) partecipano alla separazione tra le tre classi. Le restanti simulano attributi poco informativi o ridondanti. La [Figura 7.10](#fig-07-make-classification-ilustracao) presenta una rappresentazione concettuale di tale processo.

Prima di introdurre l'algoritmo che sarà studiato in dettaglio in questo capitolo, è opportuna un'osservazione: per identificare, tra le dieci caratteristiche sintetiche, quali siano più discriminative — e quindi selezionarne due per la visualizzazione 2D —, si utilizza un *Random Forest* solo come strumento ausiliario di diagnosi. Il KNN, focus di questo capitolo, viene presentato di seguito.

In [9]:
# Configurazione per la riproduzione
np.random.seed(42)

# Generazione di dati sintetici con caratteristiche controllate
X, y = make_classification(
    n_samples=300,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    n_repeated=1,        # Una caratteristica è copia di un'altra
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42,
)

# Creare figura con due sottotrame
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sottotrama 1: Visualizzazione delle classi in 2D (usando due caratteristiche informative)
# Identificare quali caratteristiche sono più informative
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importancias = rf.feature_importances_
caracteristicas_informativas = np.argsort(importancias)[-2:]  # Le due più importanti

cores = {0: "#e74c3c", 1: "#f1c40f", 2: "#e67e22"}  # Mela, Banana, Arancia
rotulos = {0: "Maçã", 1: "Banana", 2: "Laranja"}

for classe in range(3):
    idx = y == classe
    ax1.scatter(X[idx, caracteristicas_informativas[0]], 
               X[idx, caracteristicas_informativas[1]],
               c=cores[classe], label=rotulos[classe], 
               alpha=0.6, s=50, edgecolors="white", linewidth=0.5)

ax1.set_xlabel(f"Característica {caracteristicas_informativas[0]+1} (informativa)", fontsize=11)
ax1.set_ylabel(f"Característica {caracteristicas_informativas[1]+1} (informativa)", fontsize=11)
ax1.set_title("Classes no Espaço de Características\n(2 características informativas)", 
             fontsize=12, fontweight="bold")
ax1.legend(loc="upper right")
ax1.grid(alpha=0.3)

# Sottotrama 2: Importanza delle caratteristiche
bars = ax2.bar(range(1, 11), importancias, color="#4a90d9", alpha=0.7)
ax2.set_xlabel("Índice da Característica", fontsize=11)
ax2.set_ylabel("Importância", fontsize=11)
ax2.set_title("Importância de cada Característica\npara a Classificação", 
             fontsize=12, fontweight="bold")
ax2.set_xticks(range(1, 11))
ax2.grid(axis="y", alpha=0.3)

# Colorare le barre per evidenziare le caratteristiche
cores_barras = ["#e74c3c" if i < 7 else "#95a5a6" for i in range(10)]
for bar, cor in zip(bars, cores_barras):
    bar.set_color(cor)

# Aggiungere legenda
from matplotlib.patches import Patch
legenda_elements = [
    Patch(facecolor="#e74c3c", label="Características Informativas (7)"),
    Patch(facecolor="#95a5a6", label="Características Redundantes (3)")
]
ax2.legend(handles=legenda_elements, loc="upper right")

# Annotare il numero di caratteristiche informative
ax2.axhline(y=0.15, color="red", linestyle="--", alpha=0.3)
ax2.text(0.5, 0.17, "Limiar de importância", fontsize=9, color="red", alpha=0.7)

plt.tight_layout()
plt.show()

print("\n🔍 Analisi dei dati generati:")
print(f"  • Totale dei campioni: {X.shape[0]}")
print(f"  • Numero di caratteristiche: {X.shape[1]}")
print(f"  • Caratteristiche informative: 7 (colonne da 1 a 7 del grafico)")
print(f"  • Caratteristiche ridondanti: 2 (colonne 8 e 9)")
print(f"  • Caratteristiche ripetute: 1 (colonna 10)")
print(f"  • Distribuzione delle classi: {np.bincount(y)}")

<Figure size 4200x1500 with 2 Axes>

**Figura 7.10:** Illustrazione del processo di generazione di dati sintetici con *make_classification*. A sinistra, visualizzazione delle tre classi in uno spazio bidimensionale formato da due caratteristiche informative. A destra, importanza relativa di ciascuna caratteristica per la classificazione, evidenziando che solo 7 delle 10 caratteristiche sono effettivamente discriminative, mentre le altre sono ridondanti (2) o ripetute (1).



🔍 Analisi dei dati generati:
  • Totale dei campioni: 300
  • Numero di caratteristiche: 10
  • Caratteristiche informative: 7 (colonne da 1 a 7 del grafico)
  • Caratteristiche ridondanti: 2 (colonne 8 e 9)
  • Caratteristiche ripetute: 1 (colonna 10)
  • Distribuzione delle classi: [101  98 101]


## 7.10 Classificatore k-NN: Come Funziona Internamente

Il **k-*Nearest Neighbors* (k-NN)** è uno degli algoritmi di classificazione più semplici e intuitivi dell'Apprendimento Automatico. A differenza di molti classificatori, non costruisce esplicitamente un modello durante la fase di addestramento. Invece, memorizza i campioni etichettati e, quando un nuovo campione deve essere classificato, cerca quelli che più gli somigliano.

Il principio dell'algoritmo si basa sull'ipotesi che campioni con caratteristiche simili tendano ad appartenere alla stessa classe. Per quantificare questa vicinanza, il k-NN utilizza una misura di distanza tra i vettori di caratteristiche.

Come esempio, si consideri la [Tabela 7.2](#tbl-knn-frutas), che presenta una versione semplificata del problema di classificazione della frutta utilizzando solo due caratteristiche: intensità del colore e circolarità, entrambe normalizzate nell'intervallo da 0 a 1.

<a id="tbl-knn-frutas"></a>

**Tabela 7.2:** Esempio semplificato di classificazione della frutta utilizzando due caratteristiche normalizzate.

| Campione         | Colore | Circolarità | Classe |
|-----------------|----:|--------------:|--------|
| Frutta 1         | 0,82 | 0,88 | Mela |
| Frutta 2         | 0,30 | 0,20 | Banana |
| Frutta 3         | 0,88 | 0,85 | Mela |
| Frutta ? (test) | 0,80 | 0,90 | ? |


Osservando solo queste due caratteristiche, si nota che la frutta di test è molto più vicina ai campioni etichettati come **Mela** che al campione etichettato come **Banana**. Nella sezione successiva, questa nozione intuitiva di vicinanza sarà formalizzata tramite una metrica di distanza, utilizzata dall'algoritmo per identificare i vicini più prossimi e decidere la classe del nuovo campione.

### 7.10.1 Metrica di Distanza

La prossimità tra due campioni è normalmente quantificata dalla
**distanza euclidea**, definita da

$$
d(x,x_i)=\|x-x_i\|_2=
\sqrt{\sum_{j=1}^{n}(x_j-x_{i,j})^2},
$$

dove:

-   $x$ è il campione di test;
-   $x_i$ è un campione del set di addestramento;
-   $n$ è il numero di caratteristiche;
-   $x_j$ e $x_{i,j}$ rappresentano la $j$-esima caratteristica.

Nell'implementazione di questo capitolo, $x$ corrisponde a una riga di `X_test`
e $x_i$ a una riga di `X_train`. Il metodo `predict()` calcola
automaticamente la distanza tra $x$ e tutti i campioni di
addestramento.

Nell'esempio della [Tabela 7.2](#tbl-knn-frutas):

$$
d(\text{teste}, \text{Frutta 1}) \approx 0{,}028,\qquad
d(\text{teste}, \text{Frutta 2}) \approx 0{,}860,\qquad
d(\text{teste}, \text{Frutta 3}) \approx 0{,}094.
$$

Poiché le distanze minori corrispondono alla Frutta 1 e alla Frutta 3, questi campioni
saranno utilizzati nella fase di decisione.

### 7.10.2 Regola di Decisione

Dopo aver ordinato le distanze, l'algoritmo seleziona i $k$ vicini più
prossimi. Sia $N_k(x)$ questo insieme. La classe prevista è data da

$$
\hat y=\operatorname{moda}\{\,y_i:x_i\in N_k(x)\,\},
$$

dove $y_i$ è l'etichetta del campione $x_i$ e $\hat y$ è la classe assegnata
al campione di test.

Nell'esempio, per $k=3$, i vicini sono Frutto 1 (Mela), Frutto 3 (Mela) e
Frutto 2 (Banana). Poiché **Mela** riceve due voti, questa è la classe
prevista.

::: callout-tip
### Classe `KNeighborsClassifier`

In questo capitolo, l'algoritmo viene implementato con la classe
`KNeighborsClassifier`, della libreria `scikit-learn`:

``` python
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
```

dove:

-   `KNeighborsClassifier(n_neighbors=3)`: definisce il valore di $k$;
-   `fit(X_train, y_train)`: memorizza i campioni di addestramento
    (`X_train`) e le loro etichette (`y_train`);
-   `predict(X_test)`: restituisce le classi previste per i campioni di
    `X_test`.

Internamente, `predict()` esegue le fasi descritte in precedenza:
calcola le distanze, identifica i $k$ vicini più prossimi e
determina la classe per votazione a maggioranza.
:::

La [Figura 7.11](#fig-07-knn-passo-passo) illustra questa procedura in un insieme
bidimensionale. La figura evidenzia i vicini utilizzati nella classificazione,
mentre la console presenta le fasi dell'algoritmo: calcolo delle
distanze, ordinamento, selezione dei vicini, votazione e previsione della
classe.

In [10]:
def knn_passo_a_passo(X, y, x, k=3):
    """Esegue le cinque fasi dell'algoritmo k-NN.
    Parametri: X (addestramento), y (etichette), x (test) e k (numero di vicini).
    """
    # 1. Distanze
    dist = [(np.linalg.norm(x-xi), yi, i) for i, (xi, yi) in enumerate(zip(X, y))]
    print(f"1. Distanze calcolate: {len(dist)}")

    # 2. Ordinamento
    dist.sort(key=lambda t: t[0])
    print("2. Distanze ordinate")

    # 3. Selezione
    vizinhos = dist[:k]
    print(f"3. {k} vicini più prossimi:")
    for d, c, _ in vizinhos:
        print(f"   {d:.4f} → {c}")

    # 4. Votazione
    votos = {}
    for _, c, _ in vizinhos:
        votos[c] = votos.get(c, 0) + 1
    print("4. Voti:", votos)

    # 5. Decisione
    classe = max(votos, key=votos.get)
    print("5. Classe prevista:", classe)

    return classe, vizinhos


# Dati di esempio
np.random.seed(4)
X = np.r_[np.random.randn(15,2)+[2,2],
          np.random.randn(15,2)+[-2,-2]]
y = np.array(["Classe A"]*15 + ["Classe B"]*15)
x = np.array([0.5,0.5])

classe, vizinhos = knn_passo_a_passo(X, y, x)

# Visualizzazione
plt.figure(figsize=(5,5))

for c, rotulo in [("Classe A","Classe A"), ("Classe B","Classe B")]:
    P = X[y==c]
    plt.scatter(P[:,0], P[:,1], s=80, label=rotulo)

plt.scatter(*x, marker="*", s=220, edgecolors="black", label="Teste")

for _, _, i in vizinhos:
    plt.scatter(*X[i], s=220, facecolors="none", edgecolors="black", linewidths=2)
    plt.plot([x[0], X[i,0]], [x[1], X[i,1]], "--", lw=1)

plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title(f"k-NN ($k=3$): classe predita = {classe}")
plt.legend()
plt.grid(alpha=.3)
plt.axis("equal")
plt.tight_layout()
plt.show()

1. Distanze calcolate: 30
2. Distanze ordinate
3. 3 vicini più prossimi:
   1.0850 → Classe A
   1.6379 → Classe B
   1.6382 → Classe A
4. Voti: {np.str_('Classe A'): 2, np.str_('Classe B'): 1}
5. Classe prevista: Classe A


<Figure size 1500x1500 with 1 Axes>

**Figura 7.11:** Classificazione di un nuovo campione tramite l


### 7.10.3 Il Ruolo del Parametro $k$

Il parametro $k$ determina quanti vicini partecipano alla decisione di classificazione.

- **Valori piccoli di $k$** (ad esempio, $k=1$) rendono il classificatore più sensibile a rumori e variazioni locali, producendo frontiere di decisione più irregolari e favorendo l'*overfitting*.
- **Valori maggiori di $k$** producono frontiere di decisione più morbide, ma possono ridurre la sensibilità a strutture locali, favorendo l'*underfitting*.

In problemi con due classi, è comune utilizzare valori dispari di $k$ per ridurre l'occorrenza di pareggi.

Un altro aspetto importante è la **maledizione della dimensionalità** (*curse of dimensionality*). Man mano che il numero di caratteristiche aumenta, le distanze tra i campioni tendono a diventare più simili, rendendo difficile l'identificazione di vicini realmente rappresentativi.

::: callout-note
### Riepilogo

L'algoritmo k-NN può essere riassunto in tre fasi:

1. estrarre il vettore di caratteristiche del nuovo campione;
2. identificare i $k$ vicini più prossimi;
3. classificare il campione tramite la classe più frequente tra questi vicini.

Il simulatore della [Figura 7.12](#fig-07-sim-07-knn) consente di esplorare visivamente l'effetto del parametro $k$ sulla frontiera di decisione.
:::

In [11]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-07-knn" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-07-knn * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-07-knn canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; cursor: crosshair; margin: 0 auto; }
  #sim-07-knn button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-07-knn button:hover { background: #e8dfcf; }
  #sim-07-knn button.sim-07-knn_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-07-knn input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-07-knn_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-07-knn_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-07-knn_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-07-knn_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-07-knn_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-07-knn_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-07-knn_legend { display: flex; align-items: center; gap: 6px; font-size: 11px; font-weight: 600; color: #5e5a4a; }
  .sim-07-knn_dot { width: 10px; height: 10px; border-radius: 50%; display: inline-block; border: 1px solid rgba(38,36,29,0.2); }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📌 Simulatore: Frontiera di Decisione del k-NN</span>
  <span class="sim-07-knn_pill">Clicca sul canvas per aggiungere punti</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles e Estatísticas -->
  <div class="sim-07-knn_panel" style="margin-bottom:14px;">
    
    <div class="sim-07-knn_grid_stats">
      <div class="sim-07-knn_stat_box">
        <div class="sim-07-knn_stat_label">Vicini (k)</div>
        <div id="sim-07-knn_kVal" class="sim-07-knn_stat_value" style="color:#2980b9;">3</div>
      </div>
      <div class="sim-07-knn_stat_box">
        <div class="sim-07-knn_stat_label">Classe Blu</div>
        <div id="sim-07-knn_nAzul" class="sim-07-knn_stat_value" style="color:#2980b9;">0</div>
      </div>
      <div class="sim-07-knn_stat_box">
        <div class="sim-07-knn_stat_label">Classe Rossa</div>
        <div id="sim-07-knn_nVerm" class="sim-07-knn_stat_value" style="color:#c0392b;">0</div>
      </div>
    </div>

    <!-- Botões de Ação -->
    <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:12px; justify-content:center;">
      <button id="sim-07-knn_btnAzul" class="sim-07-knn_active">🔵 Aggiungi Blu</button>
      <button id="sim-07-knn_btnVerm">🔴 Aggiungi Rosso</button>
      <button id="sim-07-knn_btnLimpar" style="border-color:#f5b7b1; color:#c0392b; background:#fdecea;">🗑️ Pulisci</button>
      <button id="sim-07-knn_btnReset">↺ Ripristina</button>
    </div>

    <!-- Slider k -->
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Valore di k: <span id="sim-07-knn_slVal" style="font-family:monospace; color:#26241d;">3</span>
      </label>
    </div>
    <input type="range" id="sim-07-knn_slider" min="1" max="15" step="1" value="3">

  </div>

  <!-- Canvas -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim-07-knn_canvas" width="640" height="360"></canvas>
  </div>

  <!-- Legenda e Rodapé -->
  <div class="sim-07-knn_panel">
    <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
      <div style="display:flex; gap:16px;">
        <div class="sim-07-knn_legend"><span class="sim-07-knn_dot" style="background:#2980b9;"></span> Classe Blu</div>
        <div class="sim-07-knn_legend"><span class="sim-07-knn_dot" style="background:#c0392b;"></span> Classe Rossa</div>
      </div>
      <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
        La regione colorata di sfondo rappresenta la classe assegnata dall'algoritmo a ogni punto dello spazio.
      </div>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Knn(root){
    if (!root || root.dataset.sim07KnnInit) return;
    root.dataset.sim07KnnInit = "1";

    const canvas = root.querySelector("#sim-07-knn_canvas");
    const ctx = canvas.getContext("2d");
    const W = canvas.width, H = canvas.height;
    const CELL = 10;

    let currentLabel = 0; // 0 = azul, 1 = vermelho
    let kVal = 3;
    let points = [];

    function defaultPoints(){
      return [
        {x:120,y:260,l:0},{x:150,y:230,l:0},{x:100,y:200,l:0},{x:160,y:290,l:0},
        {x:90,y:250,l:0},{x:140,y:190,l:0},{x:175,y:250,l:0},{x:110,y:150,l:0},
        {x:200,y:230,l:0},{x:130,y:310,l:0},
        {x:480,y:100,l:1},{x:450,y:130,l:1},{x:500,y:160,l:1},{x:440,y:80,l:1},
        {x:510,y:110,l:1},{x:470,y:60,l:1},{x:430,y:150,l:1},{x:520,y:190,l:1},
        {x:490,y:220,l:1},{x:460,y:190,l:1},
        {x:300,y:170,l:0},{x:320,y:190,l:1},{x:280,y:200,l:1},{x:310,y:150,l:0}
      ];
    }

    function classify(x, y, k, pontos){
      if (pontos.length === 0) return null;
      const dists = pontos.map(p => ({
        d: (p.x - x) * (p.x - x) + (p.y - y) * (p.y - y),
        l: p.l
      }));
      dists.sort((a, b) => a.d - b.d);
      const vizinhos = dists.slice(0, Math.min(k, dists.length));
      let votos = [0, 0];
      vizinhos.forEach(v => votos[v.l]++);
      return votos[1] > votos[0] ? 1 : 0;
    }

    function render(){
      ctx.clearRect(0, 0, W, H);

      // Região de decisão em grade
      for (let gy = 0; gy < H; gy += CELL){
        for (let gx = 0; gx < W; gx += CELL){
          const cx = gx + CELL / 2, cy = gy + CELL / 2;
          const classe = points.length > 0 ? classify(cx, cy, kVal, points) : null;
          if (classe === 0) ctx.fillStyle = "rgba(41, 128, 185, 0.15)";
          else if (classe === 1) ctx.fillStyle = "rgba(192, 57, 43, 0.15)";
          else ctx.fillStyle = "#fafaf7";
          ctx.fillRect(gx, gy, CELL, CELL);
        }
      }

      // Desenhar pontos de treinamento
      points.forEach(p => {
        ctx.beginPath();
        ctx.arc(p.x, p.y, 6.5, 0, 2 * Math.PI);
        ctx.fillStyle = p.l === 0 ? "#2980b9" : "#c0392b";
        ctx.fill();
        ctx.lineWidth = 1.5;
        ctx.strokeStyle = "#ffffff";
        ctx.stroke();
      });

      root.querySelector("#sim-07-knn_nAzul").textContent = points.filter(p => p.l === 0).length;
      root.querySelector("#sim-07-knn_nVerm").textContent = points.filter(p => p.l === 1).length;
      root.querySelector("#sim-07-knn_kVal").textContent = kVal;
    }

    canvas.addEventListener("click", function(ev){
      const rect = canvas.getBoundingClientRect();
      const scaleX = canvas.width / rect.width;
      const scaleY = canvas.height / rect.height;
      const x = (ev.clientX - rect.left) * scaleX;
      const y = (ev.clientY - rect.top) * scaleY;
      points.push({x: x, y: y, l: currentLabel});
      render();
    });

    root.querySelector("#sim-07-knn_slider").addEventListener("input", function(ev){
      kVal = parseInt(ev.target.value, 10);
      root.querySelector("#sim-07-knn_slVal").textContent = kVal;
      render();
    });

    const btnAzul = root.querySelector("#sim-07-knn_btnAzul");
    const btnVerm = root.querySelector("#sim-07-knn_btnVerm");

    function setLabel(l){
      currentLabel = l;
      btnAzul.classList.toggle("sim-07-knn_active", l === 0);
      btnVerm.classList.toggle("sim-07-knn_active", l === 1);
    }

    btnAzul.addEventListener("click", function(){ setLabel(0); });
    btnVerm.addEventListener("click", function(){ setLabel(1); });

    root.querySelector("#sim-07-knn_btnLimpar").addEventListener("click", function(){
      points = [];
      render();
    });

    root.querySelector("#sim-07-knn_btnReset").addEventListener("click", function(){
      points = defaultPoints();
      render();
    });

    setLabel(0);
    points = defaultPoints();
    render();
  }

  function tryInitSim07Knn(){
    var root = document.getElementById("sim-07-knn");
    if (root) initSim07Knn(root); else setTimeout(tryInitSim07Knn, 200);
  }
  tryInitSim07Knn();
})();
</script>
</div>
""")

**Figura 7.12:** Simulatore interattivo della frontiera di decisione del k-NN: aggiungi punti di addestramento e regola il valore di k per osservare l


<figure id="fig-07-sim-07-knn">
  <img src="imagens/fig-07-sim-07-knn.png" alt=" Simulatore interattivo della frontiera di decisione del k-NN: aggiungi punti di addestramento e regola il valore di k per osservare l'effetto sulla regione di decisione. " style="max-width:80%" />
  <figcaption><strong>Figura 7.12:</strong>  Simulatore interattivo della frontiera di decisione del k-NN: aggiungi punti di addestramento e regola il valore di k per osservare l'effetto sulla regione di decisione. </figcaption>
</figure>

## 7.11 Progetto Pratico 1: Classificazione di Cifre Scritte a Mano con k-NN

Le sezioni precedenti hanno presentato l'algoritmo k-NN attraverso un esempio semplificato di classificazione della frutta, utilizzando solo due caratteristiche. Di seguito, lo stesso algoritmo viene applicato a un insieme di dati di immagini, in cui ciascun campione è rappresentato da un vettore di dimensione maggiore.

Come caso di studio, si utilizza la base pubblica `load_digits`, messa a disposizione dalla libreria `scikit-learn`. Questo insieme di dati contiene 1797 immagini di cifre scritte a mano appartenenti alle classi da 0 a 9, ciascuna con una risoluzione di $8 \times 8$ pixel in scala di grigi. Ogni immagine è rappresentata da un vettore con 64 caratteristiche, corrispondenti alle intensità dei pixel, e ogni vettore possiede un'etichetta che indica la cifra corrispondente.

La base `load_digits` è messa a disposizione dalla libreria `scikit-learn` e viene utilizzata in questo capitolo per illustrare l'applicazione dell'algoritmo k-NN. Oltre a essere disponibile direttamente in `scikit-learn`, essa non richiede fasi aggiuntive di acquisizione e preparazione dei dati, consentendo di concentrare l'attenzione sull'implementazione e sulla valutazione del classificatore.

La [Figura 7.13](#fig-07-digits-amostra) presenta un campione delle immagini della base di dati.

In [12]:
digits = load_digits()
print(f"Totale dei campioni: {digits.data.shape[0]}, dimensione del vettore: {digits.data.shape[1]}")
print(f"Classi: {[int(i) for i in sorted(set(digits.target))]}")

n_amostras = 16
imgs = list(digits.images[:n_amostras])
imgs_titles = [str(label) for label in digits.target[:n_amostras]]
mm.show(imgs, titles=imgs_titles, cols=8, figsize=(12, 4))


Totale dei campioni: 1797, dimensione del vettore: 64
Classi: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


<Figure size 1800x600 with 16 Axes>

**Figura 7.13:** Campione di cifre scritte a mano dalla base *load_digits*, utilizzato come caso di studio di classificazione.


### 7.11.1 Classificazione con Vettori di Intensità

In questo primo esperimento, ogni immagine di dimensione $8 \times 8$ è rappresentata direttamente dalle intensità dei suoi 64 pixel, senza l'estrazione di descrittori aggiuntivi. Pertanto, ogni campione corrisponde a un vettore di 64 caratteristiche, utilizzato come input del classificatore k-NN.

Successivamente, il set di dati viene suddiviso in sottoinsiemi di training e test, preservando la proporzione delle dieci classi tramite il parametro `stratify=y`. Il classificatore viene addestrato con $k=3$ e valutato sul set di test utilizzando l'accuratezza e la matrice di confusione presentata nella [Figura 7.14](#fig-07-knn-pixels)..

In [13]:
X, y = digits.data, digits.target

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

n_neighbors = 3
knn_pixels = KNeighborsClassifier(n_neighbors=n_neighbors)
knn_pixels.fit(X_treino, y_treino)
pred_pixels = knn_pixels.predict(X_teste)

acc_pixels = accuracy_score(y_teste, pred_pixels)
print(f"Accuratezza (vettori di intensità, k={n_neighbors}): {acc_pixels:.4f}")

cm = confusion_matrix(y_teste, pred_pixels)

plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Blues")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, cm[i, j],
            ha="center", va="center",
            color="white" if cm[i, j] > cm.max()/2 else "black",
            fontsize=9
        )

plt.title("Matriz de Confusão — Pixels Brutos")
plt.xlabel("Classe Predita")
plt.ylabel("Classe Real")
plt.xticks(range(10))
plt.yticks(range(10))
plt.colorbar(fraction=0.046)
plt.tight_layout()
plt.show()


Accuratezza (vettori di intensità, k=3): 0.9870


<Figure size 1500x1200 with 2 Axes>

**Figura 7.14:** Matrice di confusione del classificatore k-NN addestrato con vettori di intensità grezzi (pixel).


### 7.11.2 Classificazione con descrittori HOG

Nell'esperimento precedente, ogni immagine è stata rappresentata direttamente dalle intensità dei suoi pixel. In questa sezione, tale rappresentazione viene sostituita da descrittori HOG (*Histogram of Oriented Gradients*), che codificano informazioni sulla distribuzione delle orientazioni dei gradienti dell'immagine.

Si mantengono la stessa suddivisione dei dati, lo stesso classificatore k-NN e lo stesso protocollo di valutazione, modificando solo la rappresentazione delle immagini. La [Figura 7.15](#fig-07-comparativo-descritores) confronta i risultati ottenuti con vettori di intensità e con descrittori HOG.

In [14]:
descritores_hog = np.array([
    hog(img, orientations=8, pixels_per_cell=(4, 4), cells_per_block=(1, 1))
    for img in digits.images
])
print(f"Dimensione del vettore HOG: {descritores_hog.shape[1]}")

Xh_treino, Xh_teste, yh_treino, yh_teste = train_test_split(
    descritores_hog, y, test_size=0.3, random_state=42, stratify=y
)

n_neighbors = 3
knn_hog = KNeighborsClassifier(n_neighbors=n_neighbors)
knn_hog.fit(Xh_treino, yh_treino)
pred_hog = knn_hog.predict(Xh_teste)
acc_hog = accuracy_score(yh_teste, pred_hog)
print(f"Accuratezza (descrittore HOG, k={n_neighbors}): {acc_hog:.4f}")

plt.figure(figsize=(4, 3))
plt.bar(["Pixels brutos", "HOG"], [acc_pixels, acc_hog], color=["#6366f1", "#f97316"])
plt.ylim(0, 1.1)  # Aumenta il limite superiore per dare spazio
plt.ylabel("Acurácia")
plt.title("Comparação de Descritores")
for i, v in enumerate([acc_pixels, acc_hog]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")  # Aumenta lo spostamento verticale
plt.tight_layout()

Dimensione del vettore HOG: 32
Accuratezza (descrittore HOG, k=3): 0.7593


<Figure size 1200x900 with 1 Axes>

**Figura 7.15:** Confronto di accuratezza tra descrittori di pixel grezzi e HOG per il classificatore k-NN (k=3) sulla base di cifre.


> ### 📝 Perché accade questo?
>
> Nella [Figura 7.15](#fig-07-comparativo-descritores), il classificatore addestrato con vettori di **intensità dei pixel** raggiunge una maggiore accuratezza ($0.987$) rispetto a quello basato su descrittori **HOG** ($0.759$). Questo risultato è legato alle caratteristiche della base `load_digits`.
>
> Le immagini hanno una risoluzione di soli $8\times8$ pixel, sono approssimativamente centrate e presentano poca variazione di illuminazione, scala e orientamento. In questo scenario, le intensità dei pixel preservano praticamente tutta l'informazione necessaria per distinguere le classi. Al contrario, l'HOG riassume l'immagine in istogrammi di orientamenti dei gradienti, riducendo parte del dettaglio spaziale disponibile nei pixel originali.
>
> Questa riduzione delle informazioni può rendere difficile la separazione di cifre visivamente simili, come 3 e 8 oppure 4 e 9, specialmente quando la risoluzione dell'immagine è bassa.
>
> In problemi con immagini a più alta risoluzione o soggette a variazioni di illuminazione, posizione, scala o piccole deformazioni, descrittori come l'HOG tendono a rappresentare meglio la struttura locale dell'immagine rispetto ai valori individuali dei pixel. Pertanto, questo esperimento illustra un principio importante dell'Apprendimento Automatico: **la rappresentazione dei dati deve essere scelta in base alle caratteristiche del problema e non alla complessità del descrittore.**

## 7.12 Valutazione dei Classificatori

Nelle sezioni precedenti, la qualità del classificatore è stata analizzata
mediante l'accuratezza e la matrice di confusione. In questa sezione, questi strumenti
vengono integrati con metriche utilizzate nella valutazione dei modelli e con
una procedura per selezionare il valore del parametro $k$.

L'accuratezza corrisponde alla proporzione di campioni classificati
correttamente. Sebbene sia una misura semplice e ampiamente utilizzata, essa
può risultare insufficiente quando le classi presentano distribuzioni molto
sbilanciate.

A partire dalla matrice di confusione — introdotta nel **Capitolo 1** e
utilizzata nel corso di questo capitolo — possono essere calcolate metriche per
classe, come precisione e richiamo:

$$
\text{Precisione}=\frac{VP}{VP+FP},
\qquad
\text{Richiamo}=\frac{VP}{VP+FN},
$$

dove $VP$, $FP$ e $FN$ rappresentano, rispettivamente, il numero di
veri positivi, falsi positivi e falsi negativi della classe
analizzata. La precisione quantifica la proporzione di predizioni positive
corrette, mentre il richiamo misura la capacità del classificatore di
identificare gli esempi appartenenti alla classe.

### 7.12.1 Scelta di $k$ tramite Validazione Incrociata

Negli esperimenti precedenti, si è adottato $k=3$ per illustrare il funzionamento dell'algoritmo. Tuttavia, questo parametro influenza direttamente le prestazioni del classificatore e, nella pratica, deve essere selezionato a partire dai dati.

Un approccio ampiamente utilizzato è la **validazione incrociata** (*cross-validation*), in cui il set di addestramento viene suddiviso in partizioni successive per stimare le prestazioni del modello su dati non utilizzati durante l'addestramento.

Il codice seguente calcola l'accuratezza media ottenuta tramite validazione incrociata di cinque partizioni (*5-fold cross-validation*) per diversi valori di $k$. La [Figura 7.16](#fig-07-elbow-k) presenta i risultati, consentendo di identificare la regione in cui il classificatore raggiunge le migliori prestazioni.

In [15]:
valores_k = range(1, 16)
acuracias_medias = []

for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(modelo, X, y, cv=5)
    acuracias_medias.append(scores.mean())

melhor_k = list(valores_k)[int(np.argmax(acuracias_medias))]
print(f"Miglior valore di k trovato: {melhor_k} (precisione media={max(acuracias_medias):.4f})")

plt.figure(figsize=(6, 4))
plt.plot(list(valores_k), acuracias_medias, marker="o", color="#4f46e5")
plt.axvline(melhor_k, color="#f97316", linestyle="--", label=f"melhor k = {melhor_k}")
plt.xlabel("k")
plt.ylabel("Acurácia média (validação cruzada)")
plt.title("Seleção de k por Validação Cruzada")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

Miglior valore di k trovato: 2 (precisione media=0.9672)


<Figure size 1800x1200 with 1 Axes>

**Figura 7.16:** Precisione media per cross-validazione (5 pieghe) in funzione del parametro k, per il dataset di cifre con vettori di intensità grezzi.


### 7.12.2 Il Compromesso tra Bias e Varianza: Diagnosi di *Overfitting* e *Underfitting*

L'iperparametro $k$ influenza la complessità del confine di decisione del classificatore k-NN e, di conseguenza, la sua capacità di generalizzazione. In termini generali, valori piccoli di $k$ rendono il modello più sensibile ai campioni di addestramento, mentre valori più grandi producono confini di decisione più morbidi.

Questi comportamenti sono associati al compromesso tra **bias** (*bias*) e **varianza** (*variance*). Valori molto piccoli di $k$ tendono ad aumentare il rischio di **sovradattamento** (*overfitting*), soprattutto in insiemi di dati rumorosi, mentre valori molto grandi possono portare al **sottodattamento** (*underfitting*), riducendo la capacità del modello di catturare le strutture locali dei dati.

Mentre la [Figura 7.16](#fig-07-elbow-k) ha presentato solo l'accuratezza media ottenuta tramite validazione incrociata, la [Figura 7.17](#fig-07-overfitting-analysis) confronta le accuratezze di addestramento e di test per diversi valori di $k$. Le regioni evidenziate nel grafico rappresentano il comportamento atteso dell'algoritmo: maggiore rischio di sovradattamento per valori piccoli di $k$, una regione intermedia che spesso produce un buon equilibrio tra bias e varianza, e maggiore rischio di sottodattamento per valori elevati di $k$.

Tuttavia, queste regioni devono essere interpretate solo come un riferimento concettuale. Il comportamento osservato dipende dalle caratteristiche dell'insieme di dati. Nella base `load_digits`, ad esempio, le immagini presentano poca variabilità e una buona separazione tra le classi, così che valori piccoli di $k$ possono mostrare prestazioni simili — o addirittura superiori — agli altri, senza evidenziare un sovradattamento significativo.

In [16]:
k_values = range(1, 16)
train_acc, test_acc = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_treino, y_treino)
    train_acc.append(accuracy_score(y_treino, knn.predict(X_treino)))
    test_acc.append(accuracy_score(y_teste, knn.predict(X_teste)))

plt.figure(figsize=(9,5))
plt.plot(k_values, train_acc, "o-", lw=2, label="Treinamento")
plt.plot(k_values, test_acc,  "s-", lw=2, label="Teste")

plt.axvspan(1, 3,  color="#fca5a5", alpha=.25, label="Maior risco de overfitting")
plt.axvspan(3,11,  color="#86efac", alpha=.25, label="Compromisso entre viés e variância")
plt.axvspan(11,15, color="#93c5fd", alpha=.25, label="Maior risco de underfitting")

plt.xlabel("Número de vizinhos ($k$)")
plt.ylabel("Acurácia")
plt.xticks(k_values)
plt.grid(alpha=.3)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

maior_acc = max(test_acc)
melhores_k = [k for k, a in zip(k_values, test_acc) if np.isclose(a, maior_acc)]

print("Interpretazione")
print("- Valori piccoli di k: maggiore rischio di overfitting.")
print("- Valori intermedi: migliore compromesso tra bias e varianza.")
print("- Valori grandi di k: maggiore rischio di underfitting.")
print("\nLe regioni colorate rappresentano tendenze generali;")
print("il comportamento osservato dipende dal set di dati.")
print(f"\nMaggiore precisione sul test: {maior_acc:.3f}")
print(f"Valori di k che hanno raggiunto questa precisione: {melhores_k}")

<Figure size 2700x1500 with 1 Axes>

**Figura 7.17:** Precisione sui set di training e test per diversi valori di $k$. Le regioni colorate rappresentano, in modo concettuale, tendenze di comportamento del classificatore: maggiore rischio di overfitting (rosso), compromesso tra bias e varianza (verde) e maggiore rischio di underfitting (blu).


Interpretazione
- Valori piccoli di k: maggiore rischio di overfitting.
- Valori intermedi: migliore compromesso tra bias e varianza.
- Valori grandi di k: maggiore rischio di underfitting.

Le regioni colorate rappresentano tendenze generali;
il comportamento osservato dipende dal set di dati.

Maggiore precisione sul test: 0.987
Valori di k che hanno raggiunto questa precisione: [1, 2, 3, 5]


## 7.13 Progetto Pratico 2: Confronto di Descrittori per la Classificazione di Trame

Nel **Capitolo 6**, la varianza locale è stata utilizzata come descrittore di trama per **rilevare** anomalie su superfici industriali, distinguendo campioni **conformi** e **difettosi**. In questo progetto, il problema viene riformulato come un compito di **classificazione multiclasse**, in cui diverse rappresentazioni dell'immagine vengono utilizzate come input per un classificatore.

Verranno considerati tre tipi di descrittori: le **intensità dei pixel**, il **Local Binary Patterns (LBP)** e l'**Histogram of Oriented Gradients (HOG)**. Per ogni rappresentazione, verrà estratto un vettore di caratteristiche che servirà da input per l'algoritmo dei $k$ vicini più prossimi ($k$-NN). Alla fine, verranno confrontate le accuratezze ottenute da ciascun descrittore nelle condizioni definite per questo esperimento.

La valutazione sarà effettuata mediante **validazione incrociata stratificata a cinque partizioni (*5-fold stratified cross-validation*)**. In questa procedura, il set di dati viene suddiviso in cinque sottoinsiemi preservando la proporzione tra le classi. In ogni iterazione, una partizione viene utilizzata per il test e le quattro rimanenti per l'addestramento, ripetendo il processo fino a quando tutte le partizioni non siano state utilizzate come set di test. Al termine delle cinque esecuzioni, vengono calcolate l'accuratezza media e la deviazione standard per ciascun descrittore.

Il set di dati è composto da tre classi di trame sintetiche: **granulare**, ottenuta da rumore gaussiano smussato; **a strisce**, formata da pattern sinusoidali periodici; e **macchiata**, composta da regioni circolari sovrapposte. Per introdurre variabilità tra i campioni, tutte le immagini ricevono una perturbazione tramite rumore gaussiano di bassa intensità. La [Figura 7.18](#fig-07-texturas-amostra) presenta esempi delle tre classi utilizzate nell'esperimento.

In [17]:
rng = np.random.default_rng(42)

def gerar_textura(classe, tamanho=64, ruido=0.10):
    """Genera una texture sintetica 64x64 appartenente a una delle tre classi."""
    if classe == "granular":
        escala = rng.uniform(0.14, 0.22)
        img = rng.normal(0.5, escala, (tamanho, tamanho))
        img = cv2.GaussianBlur(img.astype(np.float32), (3, 3), 0)

    elif classe == "listrada":
        n_periodos = rng.uniform(4, 8)
        amplitude = rng.uniform(0.22, 0.38)
        eixo_x = np.linspace(0, n_periodos * np.pi, tamanho)
        base = 0.5 + amplitude * np.sin(eixo_x)
        img = np.tile(base, (tamanho, 1)).astype(np.float32)
        img += rng.normal(0, 0.09, (tamanho, tamanho)).astype(np.float32)

    elif classe == "manchada":
        img = np.full((tamanho, tamanho), 0.5, dtype=np.float32)
        n_manchas = rng.integers(5, 11)
        for _ in range(n_manchas):
            cx, cy = rng.integers(0, tamanho, 2)
            raio = int(rng.integers(3, 11))
            intensidade = float(rng.uniform(0.15, 0.9))
            cv2.circle(img, (int(cx), int(cy)), raio, intensidade, -1)
        img = cv2.GaussianBlur(img, (5, 5), 0)

    else:
        raise ValueError(f"Classe desconhecida: {classe}")

    img = img + rng.normal(0, ruido, (tamanho, tamanho)).astype(np.float32)
    img = np.clip(img, 0, 1)
    return (img * 255).astype(np.uint8)

classes_textura = ["granular", "listrada", "manchada"]
amostras = [gerar_textura(c) for c in classes_textura]

mm.show(amostras, titles=classes_textura, cols=3, figsize=(9, 3))

<Figure size 1350x450 with 3 Axes>

**Figura 7.18:** Campioni sintetici delle tre classi di texture utilizzate nell


### 7.13.1 *Pipeline* di estrazione delle caratteristiche e valutazione comparativa

Per confrontare le prestazioni di diverse forme di rappresentazione delle immagini, è stato generato un dataset bilanciato contenente 60 campioni per ciascuna classe. Da questo dataset sono stati estratti tre tipi di vettori di caratteristiche, ciascuno rappresentante aspetti distinti dell'informazione visiva:

1. **Pixel grezzi**: vettore ottenuto dall'appiattimento (*flattening*) della matrice delle intensità dell'immagine, risultante in un vettore di $64 \times 64 = 4096$ attributi;
2. **Istogramma LBP uniforme**: istogramma normalizzato delle frequenze dei pattern locali prodotti dall'operatore LBP uniforme, composto da 10 attributi;
3. **Descrittore HOG**: vettore formato da istogrammi di gradienti orientati, che rappresentano la distribuzione spaziale delle orientazioni dei bordi, per un totale di 128 attributi.

Poiché questi descrittori presentano scale e dimensionalità distinte, i vettori di caratteristiche vengono standardizzati utilizzando lo `StandardScaler`, in modo che ciascun attributo abbia media zero e deviazione standard unitaria. Questa fase evita che attributi con maggiore ampiezza influenzino in modo sproporzionato il calcolo delle distanze euclidee impiegato dal classificatore.

La valutazione viene effettuata utilizzando l'algoritmo $k$-NN con $k=5$, sotto lo stesso protocollo di validazione incrociata stratificata in cinque partizioni descritto nella sezione precedente. L'accuratezza media ottenuta lungo le cinque esecuzioni, vedi [Figura 7.19](#fig-07-comparativo-kfolds), fornisce una stima più stabile delle prestazioni del classificatore, riducendo la dipendenza da una singola suddivisione tra addestramento e test.

In [18]:
rng = np.random.default_rng(42)

# 1. Generazione del database
X_bruto, X_lbp, X_hog, y_textura = [], [], [], []

for classe in classes_textura:
    for _ in range(60):
        img = gerar_textura(classe, ruido=0.10)
        
        # Estrazione 1: Pixel grezzi
        X_bruto.append(img.ravel())
        
        # Estrazione 2: Istogramma LBP uniforme
        lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
        hist_lbp, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
        X_lbp.append(hist_lbp)
        
        # Estrazione 3: Descrittore HOG
        feat_hog = hog(img, orientations=8, pixels_per_cell=(16, 16), cells_per_block=(1, 1))
        X_hog.append(feat_hog)
        
        y_textura.append(classe)

y_textura = np.array(y_textura)
descritores = {
    "Pixels Brutos": np.array(X_bruto),
    "LBP (Textura)": np.array(X_lbp),
    "HOG (Forma)": np.array(X_hog)
}

# 2. Valutazione statistica tramite 5-fold cross-validation
resultados_media = {}
resultados_desvio = {}

knn = KNeighborsClassifier(n_neighbors=5)
scaler = StandardScaler()

for nome, X_dados in descritores.items():
    X_norm = scaler.fit_transform(X_dados)
    scores = cross_val_score(knn, X_norm, y_textura, cv=5, scoring="accuracy")
    resultados_media[nome] = scores.mean()
    resultados_desvio[nome] = scores.std()
    print(f"{nome:15s} -> Accuratezza Media: {scores.mean():.4f} (± {scores.std():.4f})")

# 3. Tracciamento del grafico comparativo formale
plt.figure(figsize=(7, 4.5))
nomes_desc = list(resultados_media.keys())
medias = list(resultados_media.values())
desvios = list(resultados_desvio.values())

bars = plt.bar(nomes_desc, medias, yerr=desvios, capsize=6, 
               color=["#6366f1", "#9333ea", "#f97316"], 
               width=0.45, edgecolor="black", alpha=0.85)
plt.ylabel("Acurácia Média (5-Fold CV)", fontsize=11)
plt.title("Análise Comparativa de Descritores para Classificação de Texturas", 
          fontsize=12, fontweight="bold")
plt.ylim(0.3, 1.1)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, 
             h + 0.03, f"{h:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

Pixels Brutos   -> Accuratezza Media: 0.6889 (± 0.0478)
LBP (Textura)   -> Accuratezza Media: 0.7611 (± 0.0648)
HOG (Forma)     -> Accuratezza Media: 0.4889 (± 0.0624)


<Figure size 2100x1350 with 1 Axes>

**Figura 7.19:** Acurácia média obtida via validação cruzada (5-fold) para os descritores de Pixels Brutos, LBP e HOG aplicados à base de texturas sintéticas.


> ### 📝 Perché funziona? — LBP come rappresentazione delle trame
>
> Il descrittore LBP rappresenta la trama di un'immagine tramite un istogramma normalizzato che conta la frequenza dei pattern locali di intensità. Invece di memorizzare direttamente i valori dei pixel o le loro posizioni, questa rappresentazione riassume la distribuzione delle microstrutture presenti nell'immagine, producendo un vettore di caratteristiche compatto.
>
> In questo esperimento sono stati confrontati tre tipi di descrittori: pixel grezzi, LBP e HOG. I vettori formati dai **pixel grezzi** preservano tutte le intensità dell'immagine, ma incorporano anche variazioni derivanti da rumore e piccoli spostamenti spaziali, il che può rendere difficile il confronto tra campioni tramite la distanza euclidea.
>
> Il descrittore **HOG** rappresenta la distribuzione delle orientazioni dei gradienti, essendo adatto a descrivere forme e contorni. Poiché le immagini utilizzate in questo progetto differiscono principalmente per le proprietà di trama, e non per la presenza di contorni ben definiti, questa rappresentazione tende a catturare meno informazioni discriminative rispetto all'LBP.
>
> L'**LBP**, invece, è stato sviluppato specificamente per caratterizzare pattern locali di trama. Il suo istogramma descrive la frequenza delle microstrutture presenti nell'immagine, indipendentemente dalla loro posizione esatta, rendendo la rappresentazione meno sensibile a piccole variazioni spaziali e a cambiamenti monotonici dell'illuminazione.
>
> Sebbene gli istogrammi delle diverse classi presentino distribuzioni distinte, il rumore gaussiano introdotto nella generazione delle immagini aumenta la variabilità tra campioni della stessa classe e può produrre regioni di sovrapposizione nello spazio delle caratteristiche. Di conseguenza, alcune trame possono essere confuse dal classificatore. Ciò nonostante, quando le caratteristiche rilevanti per distinguere le classi sono associate ai pattern locali di trama, ci si aspetta che descrittori progettati per questo scopo, come l'LBP, producano rappresentazioni più informative rispetto a quelle basate solo sulle intensità dei pixel o sulle orientazioni dei gradienti.

### 7.13.2 Diagnóstico Fino del Classificatore: Precisione, Recall e F1-Score

L'accuratezza riassume le prestazioni del classificatore in un unico valore, ma
non indica come queste prestazioni siano distribuite tra le diverse classi.
Per un'analisi più dettagliata, si utilizzano metriche calcolate
individualmente per ciascuna classe.

La **precisione** (*precision*) misura la proporzione di campioni classificati
come appartenenti a una classe che vi appartengono effettivamente. Il
**recall** (*recall*) misura la proporzione di campioni della classe che sono
stati correttamente identificati dal classificatore. Il **F1-score**
corrisponde alla media armonica tra precisione e recall, fornendo un
indicatore che bilancia entrambe le misure.

Il report riporta anche il **supporto** (*support*), ovvero il numero di
campioni di ciascuna classe presenti nel set di test. Questa informazione
è importante per contestualizzare le metriche, poiché i risultati ottenuti
su pochi campioni tendono a presentare una maggiore variabilità.

La [Figura 7.20](#fig-07-metricas-avaliacao-detalhadas) presenta queste metriche per le
tre classi di texture. Insieme, esse consentono di identificare differenze
di prestazioni che non sono evidenti solo attraverso l'accuratezza. Ad esempio,
una classe può presentare alta precisione e un recall inferiore, indicando
che il classificatore commette pochi falsi positivi, ma non riesce a
identificare parte dei campioni che appartengono effettivamente a quella classe.
Questo tipo di analisi aiuta a comprendere i limiti del modello e a
individuare possibili strategie per il suo miglioramento.

In [19]:
X_lbp_data = np.array(X_lbp)
y_textura_data = np.array(y_textura)

# Eseguire uno split train/test per il report dettagliato
Xt_treino, Xt_teste, yt_treino, yt_teste = train_test_split(
    X_lbp_data, y_textura_data, test_size=0.3, random_state=42, stratify=y_textura_data
)

# Scalare i dati
scaler = StandardScaler()
Xt_treino_scaled = scaler.fit_transform(Xt_treino)
Xt_teste_scaled = scaler.transform(Xt_teste)

# Addestrare il classificatore k-NN
knn_textura = KNeighborsClassifier(n_neighbors=5)
knn_textura.fit(Xt_treino_scaled, yt_treino)

y_pred = knn_textura.predict(Xt_teste_scaled)

report = classification_report(
    yt_teste,
    y_pred,
    target_names=classes_textura,
    output_dict=True
)

print("=== REPORT DI CLASSIFICAZIONE DETTAGLIATO ===")
print(f"{'Classe':<12} {'Precisão':>10} {'Revocação':>12} {'F1-score':>10} {'Suporte':>10}")
for classe in classes_textura:
    r = report[classe]
    print(f"{classe:<12} {r['precision']:>10.2f} {r['recall']:>12.2f} "
          f"{r['f1-score']:>10.2f} {r['support']:>10.0f}")

precision = precision_score(yt_teste, y_pred, average=None, labels=classes_textura)
recall = recall_score(yt_teste, y_pred, average=None, labels=classes_textura)
f1 = f1_score(yt_teste, y_pred, average=None, labels=classes_textura)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(classes_textura))
width = 0.25

bars1 = ax.bar(x - width, precision, width, label='Precisão', color='#6366f1', alpha=0.8)
bars2 = ax.bar(x, recall, width, label='Revocação', color='#f97316', alpha=0.8)
bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#22c55e', alpha=0.8)

ax.set_xlabel('Classe', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Métricas por Classe - Classificação de Texturas', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes_textura)
ax.legend(loc='upper right')
ax.set_ylim(0, 1.35)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("Interpretazione delle metriche:")
print("- Precisione: tra i campioni classificati come appartenenti alla classe,",
      "quanti erano corretti?")
print("- Recall: tra i campioni che appartengono realmente alla classe, quanti",
      "sono stati identificati?")
print("- F1-Score: media armonica tra precisione e recall.")
print("- Supporto: numero di campioni reali di ciascuna classe presenti nel set di test.")
print("\nIl supporto non misura le prestazioni; informa solo su quanti esempi di ciascuna",
      "classe sono \nstati utilizzati nella valutazione.")

=== REPORT DI CLASSIFICAZIONE DETTAGLIATO ===
Classe         Precisão    Revocação   F1-score    Suporte
granular           0.67         0.78       0.72         18
listrada           0.55         0.61       0.58         18
manchada           1.00         0.72       0.84         18


<Figure size 3000x1500 with 1 Axes>

**Figura 7.20:** Métricas di valutazione dettagliate per il classificatore k-NN con descrittori LBP


Interpretazione delle metriche:
- Precisione: tra i campioni classificati come appartenenti alla classe, quanti erano corretti?
- Recall: tra i campioni che appartengono realmente alla classe, quanti sono stati identificati?
- F1-Score: media armonica tra precisione e recall.
- Supporto: numero di campioni reali di ciascuna classe presenti nel set di test.

Il supporto non misura le prestazioni; informa solo su quanti esempi di ciascuna classe sono 
stati utilizzati nella valutazione.


## 7.14 Limiti dei Descrittori Artigianali

Gli esperimenti di questo capitolo mostrano che i descrittori classici possono
essere piuttosto efficaci in compiti di classificazione, ma presentano anche
importanti limitazioni:

- **Specificità:** ogni descrittore è stato sviluppato per rappresentare un
  determinato tipo di informazione, come colore, tessitura o forma. Pertanto, un
  descrittore adeguato per un compito potrebbe non essere il più appropriato per
  un altro.
- **Dipendenza dagli iperparametri:** le prestazioni di descrittori come
  LBP e HOG dipendono dalla scelta di parametri, come il raggio di vicinanza,
  il numero di punti campionati, la dimensione della cella e il numero di orientamenti,
  che devono essere regolati in base all'applicazione.
- **Rappresentazione limitata:** i descrittori di colore, tessitura e gradiente
  catturano proprietà di basso livello dell'immagine, ma non rappresentano
  direttamente concetti semantici più complessi, come oggetti o scene.
- **Maledizione della dimensionalità:** descrittori molto estesi possono
  ridurre l'efficacia di classificatori basati sulla distanza, come il
  *k*-NN.

Queste limitazioni motivano l'evoluzione delle tecniche studiate nei prossimi
capitoli. Il **Capitolo 8** presenta metodi classici per il rilevamento e la
corrispondenza di caratteristiche nelle immagini, mentre il **Capitolo 9**
introduce le **Reti Neurali Convoluzionali**, capaci di apprendere
automaticamente rappresentazioni adeguate per ciascun compito a partire dai
dati.

## 7.15 Riassunto

In questo capitolo sono stati presentati i fondamenti del riconoscimento di pattern applicato alle immagini. I principali concetti studiati sono stati:

- ***Pipeline* di riconoscimento di pattern:** acquisizione, pre-processing, estrazione di descrittori, classificazione e valutazione.
- **Descrittori classici:** descrittori di colore, LBP per la texture e HOG per la forma, utilizzati per rappresentare diverse caratteristiche delle immagini.
- **Normalizzazione delle caratteristiche:** standardizzazione (*Z-score*) per evitare che attributi di maggiore magnitudine dominino il calcolo delle distanze.
- **Classificatore *k*-NN:** classificazione basata sui $k$ vicini più prossimi nello spazio delle caratteristiche.
- **Scelta del parametro $k$:** influenza del valore di $k$ sulle prestazioni del classificatore e uso della validazione incrociata per la sua selezione.
- **Valutazione dei classificatori:** accuratezza, matrice di confusione, precisione, recall e F1-score come metriche complementari di prestazione.
- **Limiti dei descrittori artigianali:** specificità, dipendenza dagli iperparametri e difficoltà nel rappresentare informazioni di alto livello.

I concetti sono stati illustrati mediante esperimenti con la base pubblica `load_digits`, texture sintetiche generate a scopo didattico e dati simulati di descrittori di frutta.

## 7.16 🤖 Uso del Gemini Notebook come Tutor

In questa edizione, il **Gemini Notebook** viene presentato come uno strumento
di supporto allo studio. Il sistema utilizza esclusivamente i documenti
messi a disposizione dall'autore come fonte di conoscenza, consentendo
di esplorare i concetti del capitolo tramite domande, riepiloghi e
spiegazioni correlate al materiale studiato.

> ### ❗ 🎓 Studia con il Tutor Intelligente
>
> [🚀 ACCEDI AL GEMINI NOTEBOOK: CAPITOLO 07](https://notebooklm.google.com/notebook/494d06c6-ca01-4cd2-aa00-dbe30469308c)
>
> #### 🌐 Lingua e Linguaggio di Programmazione
>
> Il progetto di questo capitolo nel Gemini Notebook è stato realizzato
> esclusivamente con il testo in **portoghese** e gli esempi di codice in
> **Python**. Se stai studiando dall'edizione in inglese o francese, oppure
> segui il percorso in C++, le risposte del tutor potrebbero non corrispondere
> esattamente alla versione che stai leggendo.
>
> #### ⚠️ Avviso sui Contenuti Generati dall'IA
>
> Le risposte fornite dal Gemini Notebook possono contenere imprecisioni o
> omissioni. Quando necessario, conferma le informazioni utilizzando il
> materiale di questo capitolo e altre fonti accademiche affidabili.
> L'esecuzione degli esempi pratici presentati lungo il testo rimane il
> modo migliore per consolidare i concetti studiati.

## 7.17 Lista di Esercizi

Gli esercizi seguenti esplorano ed estendono i concetti presentati in questo capitolo tramite adattamenti degli algoritmi implementati, analisi sperimentali e confronti tra diversi approcci.

1. **(10%)** Implementare un descrittore di colore (istogramma RGB o HSV, con almeno 16 *bin* per canale) per le tre classi di frutti simulati in [Figura 7.1](#fig-07-frutas-motivacao).. Addestrare un classificatore *k*-NN con questo descrittore, confrontarne l'accuratezza con quella ottenuta dai descrittori LBP e HOG ([Figura 7.9](#fig-07-comparacao-descritores-detalhada)) e discutere in quali situazioni l'informazione cromatica è maggiormente discriminante.

2. **(15%)** Studiare l'effetto della normalizzazione delle caratteristiche (*Z-score*) sulle prestazioni del *k*-NN in uno spazio di attributi eterogeneo, combinando descrittori di colore, LBP e HOG in un unico vettore. Confrontare i risultati ottenuti con e senza normalizzazione per almeno tre valori di $k$.

3. **(15%)** Riprodurre l'analisi di *overfitting* e *underfitting* della [Figura 7.17](#fig-07-overfitting-analysis) variando la dimensione del set di addestramento (ad esempio, 20%, 50% e 80% del database `load_digits`). Discutere come la quantità di esempi influenzi la scelta del valore di $k$.

4. **(15%)** Estendere il Progetto Pratico 2 aggiungendo una quarta classe sintetica di trama. Valutare precisione, richiamo e F1-score per ciascuna classe, seguendo lo schema della [Figura 7.20](#fig-07-metricas-avaliacao-detalhadas), e analizzare l'impatto della nuova classe sulla matrice di confusione.

5. **(15%)** Implementare manualmente il classificatore *k*-NN, senza utilizzare `sklearn`:

   `sklearn.neighbors.KNeighborsClassifier`,

   completando la funzione `knn_passo_a_passo` presentata nel capitolo. Confrontare l'accuratezza e il tempo di esecuzione dell'implementazione manuale con quella del `scikit-learn` su dataset di dimensioni crescenti e collegare i risultati alla maledizione della dimensionalità.

6. **(15%)** Studiare l'influenza dei parametri `orientations`, `pixels_per_cell` e `cells_per_block` del descrittore HOG sul database `load_digits`. Valutare almeno quattro combinazioni di parametri e discutere il compromesso tra dimensionalità del descrittore e prestazioni del classificatore.

7. **(15%)** Valutare l'influenza dei parametri $P$ (numero di vicini) e $R$ (raggio) del descrittore LBP nella classificazione delle trame sintetiche del Progetto Pratico 2, considerando $P \in \{4,8,16\}$ e $R \in \{1,2,3\}$. Analizzare come tali parametri influenzino la capacità discriminativa del descrittore.

8. **(Bonus – 10%)** Implementare manualmente la validazione incrociata *k-fold* per il classificatore *k*-NN sul database `load_digits`, senza utilizzare `cross_val_score`, e confrontare i risultati con quelli ottenuti dall'implementazione del `scikit-learn` presentata in [Figura 7.16](#fig-07-elbow-k)..

## Riferimenti del Capitolo

La base teorica e gli esperimenti presentati in questo capitolo
si fondano sui seguenti riferimenti:

- Gonzalez (2018), per i fondamenti degli descrittori statistici di texture e delle operazioni di preprocessing applicate all'estrazione delle caratteristiche.
- Szeliski (2022), per la presentazione del pipeline classico di riconoscimento di pattern, dell'estrazione di descrittori e della valutazione dei classificatori in Visione Computazionale.
- Duda (2001), per i fondamenti teorici del riconoscimento di pattern, del classificatore *k*-NN e della relazione tra bias e varianza.
- Cover (1967), per la formulazione originale dell'algoritmo dei *k* vicini più prossimi.
- Ojala (2002), per la formulazione del descrittore *Local Binary Patterns* (LBP) e della sua variante uniforme, utilizzata in questo capitolo.
- Dalal (2005), per la formulazione del descrittore *Histogram of Oriented Gradients* (HOG), impiegato nella rappresentazione di forma e contorno.
- Pedregosa (2011), per l'implementazione del classificatore *k*-NN, delle metriche di valutazione e della validazione incrociata nella libreria `scikit-learn`.
- Quilici-gonzalez (2014), per la presentazione didattica di classificatori tradizionali di Riconoscimento di Pattern, come Alberi di Decisione, Regole di Classificazione e Macchine a Vettori di Supporto (SVM), complementari al classificatore k-NN esplorato in questo capitolo.
- Quilici-gonzalez (2026), per l'aggiornamento e l'ampliamento di questi contenuti nella sua 2ª edizione, attualmente in produzione.

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.it/cap07/cap07.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 7.18 💻 **Parte Pratica con Esercizi di Programmazione**

La presente lista di esercizi di programmazione (EP) consolida le formulazioni teoriche presentate nel corso del Capitolo 7 — Classificazione di Immagini e Riconoscimento di Pattern — attraverso un percorso pratico applicato. Diversamente dalla manipolazione diretta dei pixel dei capitoli precedenti, gli EP di questo capitolo lavorano con le **grandezze intermedie** di una *pipeline* reale di riconoscimento di pattern — vettori di caratteristiche, distanze, etichette previste e reali, codici binari locali e istogrammi di orientamento — consentendo di validare manualmente ogni fase del ragionamento senza dipendere da librerie esterne di apprendimento automatico.

L'incatenamento degli esercizi riproduce il flusso concettuale del capitolo: si inizia con l'implementazione manuale della regola di decisione del classificatore **k-NN** su un piccolo spazio di caratteristiche; successivamente, si rivisita, sotto l'ottica della normalizzazione delle caratteristiche, il classificatore implementato nel primo esercizio della lista; si prosegue con il calcolo delle metriche di **valutazione** (matrice di confusione, precisione e richiamo) a partire da etichette previste e reali; si continua con la codifica manuale del descrittore di texture **LBP** da un intorno $3\times3$; si approfondisce il calcolo dell'istogramma delle orientazioni del descrittore **HOG** per una singola cella; si avanza, quindi, verso l'integrazione di **estrazione di descrittori**, **classificazione k-NN** e **valutazione multi-classe** in una *pipeline* completa di riconoscimento di texture; e si conclude con l'applicazione di questa stessa *pipeline* su un'**immagine reale** (formato PGM), in cui il descrittore LBP viene calcolato direttamente sui pixel di un mosaico di texture.

### 🎯 Obiettivo di questo Quaderno

Il quaderno consente di sviluppare, validare, organizzare e testare soluzioni di **Esercizi di Programmazione (EPs)** in ambienti interattivi, come Colab, con gli stessi casi di test di Moodle, copiandoli lì solo al momento di registrare il voto ufficiale.

### *Download*

Scarica `morph.py` e `testsuite.py` eseguendo la cella qui sotto:

In [20]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

✅ Ambiente pronto. Morph: 1.1.9 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Esecuzione dei Test
Per valutare i test, esegui `TestSuite("EP07_01.estensione").run()` in una nuova cella, sostituendo l'estensione con quella del linguaggio utilizzato (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). Il sistema scarica i casi di test da GitHub, esegue il programma e calcola il voto automaticamente.

Per testare il codice Python direttamente, senza salvare un file, usa `run_code(codice)` passando il codice come *stringa* in una variabile `codice`:

```python
codice = """
# 7 ... il tuo codice qui ...
"""
TestSuite("EP07_01").run_code(codice)
```

### 🛠️ Riepilogo dei Metodi di `morph.py` (Cap. 7)

La libreria `morph.py` mette a disposizione due versioni per la maggior parte degli algoritmi: una **didattica** (metodi che terminano in `0`), implementata passo dopo passo in NumPy, e un'altra **classica**, basata sulle librerie `scikit-learn` e `scikit-image`. Le implementazioni didattiche vengono utilizzate negli **Esercizi di Programmazione (EP)**, poiché non dipendono da librerie esterne e vengono eseguite entro il limite di memoria dell'ambiente **VPL** di Moodle. Le versioni classiche, invece, sono più efficienti e indicate per esperimenti in ambienti come Colab e Jupyter Notebook, ma normalmente **non possono essere utilizzate negli EP** di Moodle, poiché la libreria `scikit-learn` supera la memoria disponibile nel VPL.

1. **Lettura dei dati (`readClasses`, `readDataset`, `readTrain`, `readTest`)**  
   Standardizzano l'input dei set di addestramento e di test, restituendo le matrici delle caratteristiche ($X$) e i vettori delle etichette ($y$).

2. **Classificazione (`knn0` / `knn`)**  
   Implementano l'algoritmo dei **k-nearest neighbors (k-NN)** per la classificazione binaria e multiclasse, utilizzando la distanza Euclidea o di Manhattan.

3. **Normalizzazione (`zscore0` / `zscore`)**  
   Applicano la normalizzazione *z-score* agli attributi, riducendo le differenze di scala prima della classificazione.

4. **Valutazione (`confusion0` / `confusion`)**  
   Calcolano la matrice di confusione e metriche come accuratezza, precisione e richiamo, sia per problemi binari che multiclasse.

5. **Descrittore di texture (`lbp0` / `lbp`)**  
   Calcolano il ***Local Binary Pattern* (LBP)**, consentendo di ottenere la mappa LBP, il codice di un pixel o l'istogramma di una regione dell'immagine.

6. **Descrittore di forma (`hog0` / `hog`)**  
   Calcolano l'***Histogram of Oriented Gradients* (HOG)**, producendo istogrammi delle orientazioni dei gradienti per rappresentare informazioni su forma e contorno.

### 7.0.1 EP07_01 🟢 Classificatore k-NN Passo dopo Passo

Il `KNeighborsClassifier` di `scikit-learn`, utilizzato nel corso del capitolo, nasconde dietro una singola chiamata (`.fit` / `.predict`) una regola decisionale piuttosto semplice: per ogni nuova osservazione, calcolare la distanza rispetto a tutti gli esempi di addestramento, selezionare i $k$ più vicini e votare per la classe di maggioranza tra di essi.

Prima di fare affidamento sulla libreria, ti è stato affidato il compito di implementare questa regola da zero, per uno spazio delle caratteristiche bidimensionale, esattamente come fa internamente il simulatore interattivo della frontiera decisionale del capitolo a ogni clic dell'utente.

#### 7.0.1.1 📋 Linee Guida di Implementazione

1. **Quantità e parametro:** Leggere l'intero $N$ (numero di esempi di addestramento) e l'intero dispari $k$ (numero di vicini).
2. **Esempi di addestramento:** Per ciascuno degli $N$ esempi, leggere tre valori: le coordinate $x$ e $y$ (reali) e l'etichetta $r$ (intero, $0$ o $1$).
3. **Query:** Leggere l'intero $Q$ (numero di punti di query) e successivamente le coordinate $x_q$, $y_q$ (reali) di ciascuna query.
4. **Distanza:** Per ogni query, calcolare la distanza euclidea rispetto a **tutti** gli esempi di addestramento:
$$
d(x_q, x_i) = \sqrt{(x_q - x_i)^2 + (y_q - y_i)^2}.
$$
5. **Selezione dei vicini:** Ordinare gli esempi per distanza crescente e selezionare i primi $k$. In caso di **parità di distanza** al confine del k-esimo vicino, risolvere a favore dell'esempio letto **per primo** nell'input (ordine di lettura stabile).
6. **Votazione di maggioranza:** Contare i voti di ciascuna classe tra i $k$ vicini selezionati. In caso di **pareggio nella votazione** (possibile solo quando $k$ è pari, cosa che non dovrebbe verificarsi per la direttiva del punto 1, ma gestire in modo difensivo), assegnare la classe del vicino più prossimo tra le classi in parità.
7. **Output:** Per ogni query, nell'ordine di input, stampare la classe prevista. Alla fine, stampare il totale delle query classificate come classe `1`.

#### 7.0.1.2 📌 Vincoli Computazionali

* **Metrica fissa:** utilizzare esclusivamente la distanza euclidea (non la *distanza al quadrato*) per l'ordinamento, sebbene il risultato del confronto sia lo stesso.
* **k sempre dispari:** l'input garantisce $k$ dispari e $k \le N$; ciononostante, implementare il pareggio del punto 6 per robustezza.
* **Stabilità:** nell'ordinamento per distanza, preservare l'ordine relativo degli esempi con la stessa distanza (ordinamento stabile).

#### 7.0.1.3 🧠 Fondamenti Teorici

| Elemento | Ruolo nel k-NN |
|---|---|
| Spazio delle caratteristiche | Insieme di tutti i vettori $(x, y)$ possibili |
| Distanza euclidea | Misura di similarità tra osservazioni |
| $k$ piccolo | Frontiera irregolare, alta varianza |
| $k$ grande | Frontiera regolare, alto bias |
| Votazione di maggioranza | Regola decisionale $\hat y = \operatorname{moda}\{y_i : x_i \in N_k(x)\}$ |

#### 7.0.1.4 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $N$ e $k$, separati da spazio.
* Prossime $N$ righe: tre valori per riga — $x$, $y$ (reali) e $r$ (intero $\in \{0,1\}$), separati da spazio.
* Riga successiva: intero $Q$.
* Prossime $Q$ righe: due valori per riga — $x_q$, $y_q$ (reali), separati da spazio.

**Output:**

* $Q$ righe, ciascuna con la classe prevista (`0` o `1`) per la rispettiva query, nell'ordine di input.
* Ultima riga: `Totale classe 1: X`.

#### 7.0.1.5 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4 3<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>1<br>1 1 | 0<br>Totale classe 1: 0 | Query vicina al gruppo di classe 0. |
| 4 1<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>2<br>0.9 0.1<br>5.5 5.1 | 0<br>1<br>Totale classe 1: 1 | Con $k=1$, ogni query eredita la classe del vicino più prossimo. |

In [21]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0701" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0701 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0701 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0701 button:hover { background: #e8dfcf; }
  #sim-ep0701 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0701_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0701_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_01: Classificatore k-NN Passo dopo Passo</span>
  <span class="sim-ep0701_pill">Voto di Maggioranza</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0701_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Numero di Vicini (k): <span id="sim-ep0701_vl" style="font-family:monospace; color:#26241d;">3</span>
      </label>
    </div>
    
    <input id="sim-ep0701_sl" type="range" min="1" max="7" step="2" value="3">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Regola k e osserva quali esempi di addestramento (ordinati per distanza) partecipano al voto per la query fissa (&starf; in x = 3, y = 3).
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0701_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0701_debug" class="sim-ep0701_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep01(root){
    if (!root || root.dataset.sim07Ep01Init) return;
    root.dataset.sim07Ep01Init = "1";

    var query = {x: 3, y: 3};
    var pontos = [
      {nome: "A", x: 0, y: 0, r: 0},
      {nome: "B", x: 1, y: 0, r: 0},
      {nome: "C", x: 5, y: 5, r: 1},
      {nome: "D", x: 6, y: 5, r: 1},
      {nome: "E", x: 2, y: 2, r: 0},
      {nome: "F", x: 4, y: 4, r: 1},
      {nome: "G", x: 0, y: 2, r: 0},
      {nome: "H", x: 6, y: 3, r: 1}
    ];

    pontos.forEach(function(p, i){
      p.d = Math.sqrt(Math.pow(p.x - query.x, 2) + Math.pow(p.y - query.y, 2));
      p.idx = i;
    });

    pontos.sort(function(a, b){
      return (a.d - b.d) || (a.idx - b.idx);
    });

    var slEl  = root.querySelector('#sim-ep0701_sl');
    var vlEl  = root.querySelector('#sim-ep0701_vl');
    var cards = root.querySelector('#sim-ep0701_cards');
    var dbg   = root.querySelector('#sim-ep0701_debug');

    function render(){
      var k = parseInt(slEl.value, 10);
      vlEl.textContent = k;
      cards.innerHTML = '';
      var votos = [0, 0];

      pontos.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">d = ' + p.d.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : pontos[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] + '  |  Classe prevista: ' + previsto;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim07Ep01(){
    var root = document.getElementById('sim-ep0701');
    if (root) initSim07Ep01(root); else setTimeout(tryInitSim07Ep01, 200);
  }
  tryInitSim07Ep01();
})();
</script>
""")

**Figura 7.21:** Simulatore EP07_01: Classificatore k-NN Passo a Passo


<figure id="fig-07-sim-ep0701">
  <img src="imagens/fig-07-sim-ep0701.png" alt=" Simulatore EP07_01: Classificatore k-NN Passo a Passo " style="max-width:80%" />
  <figcaption><strong>Figura 7.21:</strong>  Simulatore EP07_01: Classificatore k-NN Passo a Passo </figcaption>
</figure>

In [22]:
%%writefile EP07_01.py
# Codice Python

Overwriting EP07_01.py


In [23]:
TestSuite("EP07_01.py").run()

### 7.0.2 EP07_02 🟡 Normalizzazione *Z-score* e Robustezza del k-NN a Scale Diverse

Questo esercizio riprende il classificatore implementato nell'**EP07_01**, questa volta sotto la prospettiva discussa nella sezione *L'Impatto della Scala e la Normalizzazione delle Caratteristiche* del capitolo: il k-NN decide in base alla distanza tra vettori, per cui una caratteristica misurata su una scala molto più ampia rispetto alle altre tende a **dominare** il calcolo della distanza, anche quando non è la più rilevante per separare le classi.

Un sistema di ispezione registra, per ogni pezzo, la sua **area** (in pixel, che può arrivare a centinaia o migliaia) e la sua **circolarità** (sempre tra $0$ e $1$). Ti è stato affidato il compito di classificare nuovi pezzi tramite k-NN in due modi — con e senza la standardizzazione *Z-score* presentata nel capitolo — e di riportare in quali casi i due approcci **divergono**.

#### 7.0.2.1 📋 Linee Guida di Implementazione

1. **Quantità e parametro:** Leggere l'intero $N$ (numero di esempi di addestramento) e l'intero dispari $k$.
2. **Esempi di addestramento:** Per ciascuno degli $N$ esempi, leggere tre valori: l'area $x_1$ (reale), la circolarità $x_2$ (reale) e l'etichetta $r$ (intero, $0$ o $1$).
3. **Query:** Leggere l'intero $Q$ e, successivamente, le coordinate $x_1, x_2$ di ciascuna query.
4. **Classificazione senza normalizzazione:** Per ogni query, classificarla tramite k-NN direttamente su $(x_1, x_2)$, con distanza euclidea e le stesse regole di pareggio dell'EP07_01 (ordine di lettura per distanze a pari merito; vicino più prossimo tra classi a pari merito nella votazione).
5. **Parametri di normalizzazione:** Calcolare la media $\mu_j$ e la deviazione standard **popolazionale** $\sigma_j$ (divisione per $N$, non per $N-1$ — la stessa convenzione adottata dalla classe `StandardScaler`) di ciascuna caratteristica $j \in \{1,2\}$, **esclusivamente sul set di addestramento**.
6. **Standardizzazione:** Trasformare ciascuna caratteristica di addestramento e di query tramite
$$
z_j = \frac{x_j - \mu_j}{\sigma_j}.
$$
Se $\sigma_j = 0$ (caratteristica costante nell'addestramento), definire $z_j = 0$ per tutti i campioni di quella caratteristica, evitando la divisione per zero.
7. **Classificazione con normalizzazione:** Ripetere la classificazione k-NN del punto 4, ora sui vettori standardizzati $(z_1, z_2)$, con le stesse regole di pareggio.
8. **Output:** Per ogni query, nell'ordine di input, stampare le due classi previste. Alla fine, stampare il numero di query in cui le due classificazioni **divergono**.

#### 7.0.2.2 📌 Vincoli Computazionali

* **Adattamento solo sul training:** $\mu_j$ e $\sigma_j$ sono calcolati unicamente a partire dal set di addestramento e riapplicati alle query — mai ricalcolati a partire da esse. Questa pratica evita la **dispersione dei dati** (*data leakage*), menzionata nella sezione sulla normalizzazione del capitolo.
* **Deviazione standard popolazionale:** utilizzare $\sigma_j = \sqrt{\frac{1}{N}\sum_i (x_{i,j}-\mu_j)^2}$, e non la versione campionaria (divisione per $N-1$).
* **Caratteristica costante:** trattare $\sigma_j = 0$ come caso speciale (punto 6); non deve verificarsi un errore di divisione per zero.
* **Regole di pareggio:** riutilizzare esattamente le convenzioni dell'EP07_01, sia nella selezione dei $k$ vicini che nella votazione a maggioranza.

#### 7.0.2.3 🧠 Fondamenti Teorici

| Elemento | Ruolo |
|---|---|
| Standardizzazione *Z-score* | Riscalare ogni caratteristica a media $0$ e deviazione standard $1$, rendendo scale eterogenee comparabili |
| Adattamento (*fit*) solo sul training | Garantisce che la valutazione sulle query rifletta solo ciò che il modello ha appreso nell'addestramento |
| Distanza euclidea senza normalizzazione | Dominata dalla caratteristica con maggiore ampiezza — qui, l'area |
| Predizione divergente | Evidenzia che la scala delle caratteristiche, e non solo l'algoritmo o i dati, può determinare il confine decisionale del k-NN |

Questo esercizio sottolinea, in modo controllato, la ragione per cui lo `StandardScaler` viene applicato prima del k-NN nel corso del capitolo: senza questo passaggio, caratteristiche come la circolarità — anche se altamente discriminative — possono essere praticamente ignorate dal classificatore di fronte a una caratteristica come l'area con ampiezza centinaia di volte maggiore.

#### 7.0.2.4 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Interi $N$ e $k$, separati da spazio.
* Prossime $N$ righe: tre valori per riga — $x_1$, $x_2$ (reali) e $r$ (intero $\in \{0,1\}$), separati da spazio.
* Prossima riga: intero $Q$.
* Prossime $Q$ righe: due valori per riga — $x_1$, $x_2$ (reali) della query, separati da spazio.

**Output:**

* $Q$ righe, nel formato `SemNorm=<0|1> ComNorm=<0|1>`, nell'ordine di input delle query.
* Ultima riga: `Divergiu: <int>`.

#### 7.0.2.5 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4 3<br>10 0.9 0<br>12 0.85 0<br>900 0.2 1<br>950 0.25 1<br>1<br>500 0.88 | SemNorm=1 ComNorm=0<br>Divergiu: 1 | Senza normalizzazione, l'area (scala di centinaia) domina la distanza e la query viene classificata come classe `1`. Dopo la standardizzazione, la circolarità — molto più vicina ai campioni di classe `0` — inizia a pesare in modo confrontabile, e la predizione cambia a `0`. |
| 2 1<br>0 0.5 0<br>100 0.5 1<br>1<br>60 0.5 | SemNorm=1 ComNorm=1<br>Divergiu: 0 | La circolarità è costante nell'addestramento ($\sigma_2=0$); secondo la regola del punto 6, $z_2=0$ per tutti i campioni, e la classificazione dipende solo dall'area in entrambi i casi. |

In [24]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0702" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0702 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0702 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0702 button:hover { background: #e8dfcf; }
  #sim-ep0702 button.sim-ep0702_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0702_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0702_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_02: Normalizzazione Z-score e Distanza k-NN</span>
  <span class="sim-ep0702_pill">Standardizzazione delle Caratteristiche</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Seleção de Modo -->
  <div class="sim-ep0702_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:8px;">
      Ogni esempio ha due caratteristiche: area (px) e circolarità [0, 1]. Alterna la normalizzazione e osserva il cambiamento nella classe prevista.
    </div>
    
    <div id="sim-ep0702_query" style="font-size:11px; color:#26241d; text-align:center; font-family:monospace; font-weight:700; margin-bottom:10px;"></div>

    <div style="display:flex; justify-content:center; gap:8px; flex-wrap:wrap;">
      <button id="sim-ep0702_btn_raw" class="sim-ep0702_active">Senza Normalizzazione</button>
      <button id="sim-ep0702_btn_norm">Con Normalizzazione (Z-score)</button>
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0702_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0702_debug" class="sim05_ep01_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep02(root){
    if (!root || root.dataset.sim07Ep02Init) return;
    root.dataset.sim07Ep02Init = "1";

    var pontos = [
      {nome: "P1", x1: 10,  x2: 0.90, r: 0},
      {nome: "P2", x1: 12,  x2: 0.85, r: 0},
      {nome: "P3", x1: 900, x2: 0.20, r: 1},
      {nome: "P4", x1: 950, x2: 0.25, r: 1}
    ];

    pontos.forEach(function(p, i){ p.idx = i; });
    var query = {x1: 500, x2: 0.88};
    var k = 3;

    function stats(vals){
      var m = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
      var v = vals.reduce(function(a, s){ return a + (s - m) * (s - m); }, 0) / vals.length;
      return {mean: m, std: Math.sqrt(v)};
    }

    var s1 = stats(pontos.map(function(p){ return p.x1; }));
    var s2 = stats(pontos.map(function(p){ return p.x2; }));

    function z(x, s){ return s.std === 0 ? 0 : (x - s.mean) / s.std; }

    var cards   = root.querySelector('#sim-ep0702_cards');
    var dbg     = root.querySelector('#sim-ep0702_debug');
    var qEl     = root.querySelector('#sim-ep0702_query');
    var btnRaw  = root.querySelector('#sim-ep0702_btn_raw');
    var btnNorm = root.querySelector('#sim-ep0702_btn_norm');
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle('sim-ep0702_active', !modoNorm);
      btnNorm.classList.toggle('sim-ep0702_active', modoNorm);

      qEl.textContent = '★ Query: Area = ' + query.x1 + ', Circularidade = ' + query.x2 +
        (modoNorm ? ' → z_área = ' + z(query.x1, s1).toFixed(3) + ', z_circ = ' + z(query.x2, s2).toFixed(3) : '');

      var qx1 = modoNorm ? z(query.x1, s1) : query.x1;
      var qx2 = modoNorm ? z(query.x2, s2) : query.x2;

      var lista = pontos.map(function(p){
        var px1 = modoNorm ? z(p.x1, s1) : p.x1;
        var px2 = modoNorm ? z(p.x2, s2) : p.x2;
        var d = Math.sqrt((px1 - qx1) * (px1 - qx1) + (px2 - qx2) * (px2 - qx2));
        return {nome: p.nome, r: p.r, d: d, idx: p.idx, area: p.x1, circ: p.x2, va: px1, vc: px2};
      });

      lista.sort(function(a, b){ return (a.d - b.d) || (a.idx - b.idx); });

      cards.innerHTML = '';
      var votos = [0, 0];

      lista.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        var valorUsado = modoNorm
          ? ('z = (' + p.va.toFixed(2) + ', ' + p.vc.toFixed(2) + ')')
          : ('área = ' + p.area + ', circ = ' + p.circ);

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">' + valorUsado + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px;">d = ' + p.d.toFixed(3) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : lista[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = (modoNorm ? 'COM Normalização' : 'SEM Normalização') +
        '  |  k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] +
        '  |  Classe prevista: ' + previsto;
    }

    btnRaw.addEventListener('click', function(){ modoNorm = false; render(); });
    btnNorm.addEventListener('click', function(){ modoNorm = true; render(); });
    render();
  }

  function tryInitSim07Ep02(){
    var root = document.getElementById('sim-ep0702');
    if (root) initSim07Ep02(root); else setTimeout(tryInitSim07Ep02, 200);
  }
  tryInitSim07Ep02();
})();
</script>
""")

**Figura 7.22:** Simulatore EP07_02: Effetto della Normalizzazione *Z-score* sulla Distanza k-NN


<figure id="fig-07-sim-ep0702">
  <img src="imagens/fig-07-sim-ep0702.png" alt=" Simulatore EP07_02: Effetto della Normalizzazione *Z-score* sulla Distanza k-NN " style="max-width:80%" />
  <figcaption><strong>Figura 7.22:</strong>  Simulatore EP07_02: Effetto della Normalizzazione *Z-score* sulla Distanza k-NN </figcaption>
</figure>

In [25]:
%%writefile EP07_02.py
# Codice Python

Overwriting EP07_02.py


In [26]:
TestSuite("EP07_02.py").run()

### 7.0.3 EP07_03 🟡 Valutazione tramite Matrice di Confusione

Un classificatore binario della qualità della saldatura è stato addestrato e testato su una linea di produzione. Per ogni pezzo ispezionato, il sistema ha registrato l'etichetta **reale** (ottenuta da un esperto) e l'etichetta **prevista** dal classificatore, dove `1` rappresenta "difettoso" e `0` rappresenta "conforme".

La direzione qualità vuole conoscere non solo l'accuratezza del sistema, ma anche la sua **precisione** (quando il sistema segnala un difetto, con quale frequenza ha ragione?) e il suo **richiamo** (di tutti i pezzi realmente difettosi, quanti il sistema è riuscito a identificare?) — la distinzione discussa nella sezione sulla valutazione dei classificatori del capitolo.

#### 7.0.3.1 📋 Linee Guida di Implementazione

1. **Quantità:** Leggere l'intero $N$ (numero di pezzi ispezionati).
2. **Dati di ciascun pezzo:** Per ciascuno degli $N$ pezzi, leggere due interi — l'etichetta reale $y$ e l'etichetta prevista $\hat y$ (entrambe $\in \{0, 1\}$).
3. **Matrice di confusione:** Considerando la classe `1` (difettoso) come **positiva**, contare:
   - $VP$ (Vero Positivo): $y=1$ e $\hat y=1$;
   - $FP$ (Falso Positivo): $y=0$ e $\hat y=1$;
   - $FN$ (Falso Negativo): $y=1$ e $\hat y=0$;
   - $VN$ (Vero Negativo): $y=0$ e $\hat y=0$.
4. **Metriche:** Calcolare
$$
\text{Accuratezza} = \frac{VP+VN}{N}, \quad
\text{Precisione} = \frac{VP}{VP+FP}, \quad
\text{Richiamo} = \frac{VP}{VP+FN}.
$$
5. **Casi degeneri:** Se $VP+FP=0$ (nessuna previsione positiva), stampare `Precisao: indefinida`. Se $VP+FN=0$ (nessun caso positivo reale), stampare `Revocacao: indefinida`.
6. **Arrotondamento:** Tutte le metriche numeriche devono essere arrotondate a 4 cifre decimali (*round half away from zero*) solo nella visualizzazione.

#### 7.0.3.2 📌 Vincoli Computazionali

* **Convenzione di classe positiva fissa:** la classe `1` è sempre la classe positiva in questo esercizio, indipendentemente dalla sua frequenza relativa.
* **Protezione dalla divisione per zero:** implementare i casi degeneri del punto 5 prima di eseguire la divisione.
* **Ordine di uscita:** seguire esattamente l'ordine specificato nella sezione di uscita, anche nei casi degeneri.

#### 7.0.3.3 🧠 Fondamento Teorico

| Metrica | Domanda a cui risponde | Sensibile allo squilibrio? |
|---|---|---|
| Accuratezza | Quale frazione di pezzi è stata classificata correttamente? | Sì — può mascherare errori nella classe minoritaria |
| Precisione | Dei pezzi segnalati come difettosi, quanti lo sono realmente? | Penalizza i falsi positivi |
| Richiamo | Dei pezzi realmente difettosi, quanti sono stati rilevati? | Penalizza i falsi negativi |

In un contesto industriale, un **richiamo** basso è spesso più grave di una **precisione** bassa: lasciar passare un pezzo difettoso (falso negativo) tende a essere più costoso che ispezionare manualmente un pezzo buono segnalato per errore (falso positivo).

#### 7.0.3.4 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: Intero $N$.
* Successive $N$ righe: due interi per riga — $y$ e $\hat y$, separati da spazio.

**Output (in questo ordine esatto):**

```
VP=<int> FP=<int> FN=<int> VN=<int>
Acuracia: <valore o metrica indefinita>
Precisao: <valore o indefinita>
Revocacao: <valore o indefinita>
```

#### 7.0.3.5 📌 Esempi

| Input | Output | Osservazione |
|---|---|---|
| 4<br>1 1<br>0 1<br>1 0<br>0 0 | VP=1 FP=1 FN=1 VN=1<br>Acuracia: 0.5000<br>Precisao: 0.5000<br>Revocacao: 0.5000 | Un errore di ogni tipo. |
| 3<br>0 0<br>0 0<br>0 0 | VP=0 FP=0 FN=0 VN=3<br>Acuracia: 1.0000<br>Precisao: indefinida<br>Revocacao: indefinida | Nessun caso positivo reale né previsto. |

In [27]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0703" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0703 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0703 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0703 button:hover { background: #e8dfcf; }
  #sim-ep0703 button.sim-ep0703_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0703_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0703_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_03: Precisione x Richiamo</span>
  <span class="sim-ep0703_pill">Linea di Produzione</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Cenário -->
  <div class="sim-ep0703_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Scegli uno scenario di ispezione e osserva come Accuratezza, Precisione e Richiamo reagiscono in modo diverso.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0703_b1" class="sim-ep0703_active">Scenario A: Errori Bilanciati</button>
      <button id="sim-ep0703_b2">Scenario B: Falsi Negativi</button>
      <button id="sim-ep0703_b3">Scenario C: Nessun Difetto Reale</button>
      <button id="sim-ep0703_b4">Scenario D: Falsi Positivi</button>
    </div>
  </div>

  <!-- Cards das Peças do Cenário -->
  <div id="sim-ep0703_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0703_debug" class="sim-ep0703_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep03(root){
    if (!root || root.dataset.sim07Ep03Init) return;
    root.dataset.sim07Ep03Init = "1";

    var cenarios = {
      A: [{y:1, p:1}, {y:0, p:1}, {y:1, p:0}, {y:0, p:0}],
      B: [{y:1, p:0}, {y:1, p:0}, {y:1, p:1}, {y:0, p:0}],
      C: [{y:0, p:0}, {y:0, p:0}, {y:0, p:0}],
      D: [{y:0, p:1}, {y:0, p:1}, {y:1, p:1}, {y:0, p:0}]
    };

    var cards = root.querySelector('#sim-ep0703_cards');
    var dbg   = root.querySelector('#sim-ep0703_debug');

    var botoes = {
      A: root.querySelector('#sim-ep0703_b1'),
      B: root.querySelector('#sim-ep0703_b2'),
      C: root.querySelector('#sim-ep0703_b3'),
      D: root.querySelector('#sim-ep0703_b4')
    };

    function render(key){
      Object.keys(botoes).forEach(function(k){
        botoes[k].classList.toggle('sim-ep0703_active', k === key);
      });

      var dados = cenarios[key];
      var VP = 0, FP = 0, FN = 0, VN = 0;
      cards.innerHTML = '';

      dados.forEach(function(d, i){
        if (d.y === 1 && d.p === 1) VP++;
        else if (d.y === 0 && d.p === 1) FP++;
        else if (d.y === 1 && d.p === 0) FN++;
        else VN++;

        var statusCor = '';
        var statusTxt = '';

        if (d.y === d.p) {
          statusCor = 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
          statusTxt = d.y === 1 ? 'VP (Acerto)' : 'VN (Acerto)';
        } else {
          statusCor = 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;';
          statusTxt = d.p === 1 ? 'FP (Alarme Falso)' : 'FN (Escapou)';
        }

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' + statusCor;

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">Peça ' + (i + 1) + '</div>' +
          '<div style="font-family:monospace; font-size:10px; margin-bottom:4px;">Real = ' + d.y + ' | Prev = ' + d.p + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + statusTxt + '</div>';

        cards.appendChild(div);
      });

      var N = dados.length;
      var acc = ((VP + VN) / N).toFixed(4);
      var prec = (VP + FP) > 0 ? (VP / (VP + FP)).toFixed(4) : 'Indefinida';
      var rev = (VP + FN) > 0 ? (VP / (VP + FN)).toFixed(4) : 'Indefinida';

      if (prec === 'Indefinida' || parseFloat(prec) < 0.5) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'VP = ' + VP + ' | FP = ' + FP + ' | FN = ' + FN + ' | VN = ' + VN +
        '  |  Acurácia = ' + acc + '  |  Precisão = ' + prec + '  |  Revocação = ' + rev;
    }

    botoes.A.addEventListener('click', function(){ render('A'); });
    botoes.B.addEventListener('click', function(){ render('B'); });
    botoes.C.addEventListener('click', function(){ render('C'); });
    botoes.D.addEventListener('click', function(){ render('D'); });

    render('A');
  }

  function tryInitSim07Ep03(){
    var root = document.getElementById('sim-ep0703');
    if (root) initSim07Ep03(root); else setTimeout(tryInitSim07Ep03, 200);
  }
  tryInitSim07Ep03();
})();
</script>
""")

**Figura 7.23:** Simulatore EP07_03: Precisione x Richiamo


<figure id="fig-07-sim-ep0703">
  <img src="imagens/fig-07-sim-ep0703.png" alt=" Simulatore EP07_03: Precisione x Richiamo " style="max-width:80%" />
  <figcaption><strong>Figura 7.23:</strong>  Simulatore EP07_03: Precisione x Richiamo </figcaption>
</figure>

In [28]:
%%writefile EP07_03.py
# Codice Python

Overwriting EP07_03.py


In [29]:
TestSuite("EP07_03.py").run()

### 7.0.4 EP07_04 🟠 Codifica Manuale del Descrittore LBP

La funzione `local_binary_pattern` di `scikit-image`, utilizzata nel progetto di classificazione delle trame, calcola automaticamente il codice LBP di ogni pixel di un'immagine. Prima di utilizzarla come una scatola nera, ti è stato affidato il compito di implementare manualmente il calcolo del codice LBP classico ($P=8$, $R=1$) per il pixel centrale di un intorno $3\times3$, esattamente come definito nell'equazione del capitolo.

Oltre al codice, il sistema di ispezione delle trame deve anche sapere se quel pattern è **uniforme** — un pattern è uniforme quando il numero di transizioni ($0\to1$ o $1\to0$) percorrendo gli 8 bit **circolarmente** (tornando dall'ultimo bit al primo) è **al massimo 2**, proprietà sfruttata dalla variante *uniforme* del LBP menzionata nel capitolo.

#### 7.0.4.1 📋 Linee Guida di Implementazione

1. **Quantità:** Leggere l'intero $T$ (numero di intorni da elaborare).
2. **Dati di ogni intorno:** Per ciascuno dei $T$ intorni, leggere una matrice $3\times3$ di interi (intensità), fornita in 3 righe di 3 valori ciascuna. Il pixel centrale è la posizione `[1][1]`.
3. **Ordine dei vicini:** Percorrere gli 8 vicini in senso **orario**, partendo dall'angolo in alto a sinistra, nel seguente ordine di posizioni `[riga][colonna]`: `[0][0]`, `[0][1]`, `[0][2]`, `[1][2]`, `[2][2]`, `[2][1]`, `[2][0]`, `[1][0]`. Questo è l'indice $p = 0, 1, \ldots, 7$ dell'equazione del LBP.
4. **Funzione soglia:** Per ogni vicino $p$ con intensità $g_p$ e centro $g_c$, calcolare $s(g_p - g_c)$, che vale `1` se $g_p \geq g_c$ e `0` altrimenti.
5. **Codice LBP:** Calcolare
$$
\mathrm{LBP} = \sum_{p=0}^{7} s(g_p - g_c)\, 2^p.
$$
6. **Transizioni:** Considerando la sequenza circolare di bit $s_0, s_1, \ldots, s_7$ (nell'ordine del punto 3), contare quante coppie consecutive **adiacenti nella sequenza circolare** (inclusa la coppia $s_7, s_0$) differiscono tra loro.
7. **Classificazione:** Se il numero di transizioni è $\le 2$, classificare come `UNIFORME`; altrimenti, `NAO_UNIFORME`.
8. **Uscita:** Per ogni intorno, nell'ordine di ingresso, stampare il codice LBP (intero decimale, $0$–$255$), il numero di transizioni e la classificazione.

#### 7.0.4.2 📌 Vincoli Computazionali

* **Ordine fisso dei vicini:** l'ordine del punto 3 è obbligatorio — invertirlo produce un codice numericamente diverso, anche se rappresenta lo stesso pattern visivo.
* **Confronto non stretto:** $s(z) = 1$ quando $z \ge 0$ (il capitolo stesso definisce l'uguaglianza come inclusa nel caso `1`).
* **Conteggio circolare:** non dimenticare la coppia che chiude il ciclo ($s_7$ con $s_0$); ignorare questa coppia è un errore comune che classifica erroneamente i pattern uniformi.

#### 7.0.4.3 🧠 Fondamento Teorico

| Pattern (bit $s_0\ldots s_7$) | Transizioni | Interpretazione |
|---|---|---|
| `00000000` o `11111111` | 0 | Regione omogenea (macchia chiara o scura) |
| `00001111` | 2 | Bordo semplice tra due regioni |
| `01010101` | 8 | Trama a contrasto alternato — non uniforme |

I pattern uniformi si concentrano in regioni di trama liscia o bordi semplici; i pattern non uniformi tendono a corrispondere a rumore ad alta frequenza. Per questo motivo, l'istogramma LBP *uniforme*, utilizzato nel progetto di classificazione delle trame, raggruppa tutti i pattern non uniformi in un unico contenitore, riducendo la dimensionalità del descrittore.

#### 7.0.4.4 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Intero $T$.
* Per ogni intorno: 3 righe con 3 interi ciascuna (matrice $3\times3$).

**Uscita:**

* $T$ righe, nel formato `LBP=<int> transicoes=<int> <UNIFORME|NAO_UNIFORME>`.

#### 7.0.4.5 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 1<br>10 10 10<br>10 50 10<br>10 10 10 | LBP=0 transicoes=0 UNIFORME | Centro è il più chiaro; tutti i vicini generano bit 0. |
| 1<br>90 90 90<br>10 50 10<br>90 90 90 | LBP=119 transicoes=4 NAO_UNIFORME | Vicini chiari e scuri alternati nell'intorno. |

In [30]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0704" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0704 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0704 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0704 button:hover { background: #e8dfcf; }
  #sim-ep0704 button.sim-ep0704_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0704_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0704_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_04: Codice LBP di un Intorno 3&times;3</span>
  <span class="sim-ep0704_pill">P = 8, R = 1</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Exemplo -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Fai clic su una cella dell'intorno per alternare tra chiaro e scuro (il centro è fisso) e osserva il codice LBP risultante. L'etichetta p indica l'indice dell'equazione.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0704_b1">Esempio 1: Macchia Omogenea</button>
      <button id="sim-ep0704_b2" class="sim-ep0704_active">Esempio 2: Pattern Alternato</button>
    </div>
  </div>

  <!-- Grid Vizinhança 3x3 -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px; text-align:center;">
    <div id="sim-ep0704_grid" style="display:grid; grid-template-columns:repeat(3, 60px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0704_debug" class="sim-ep0704_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep04(root){
    if (!root || root.dataset.sim07Ep04Init) return;
    root.dataset.sim07Ep04Init = "1";

    var exemplos = {
      1: [[10, 10, 10], [10, 50, 10], [10, 10, 10]],
      2: [[90, 90, 90], [10, 50, 10], [90, 90, 90]]
    };

    var valores = exemplos[2].map(function(row){ return row.slice(); });
    var grid = root.querySelector('#sim-ep0704_grid');
    var dbg  = root.querySelector('#sim-ep0704_debug');
    var btn1 = root.querySelector('#sim-ep0704_b1');
    var btn2 = root.querySelector('#sim-ep0704_b2');

    var ordem = [[0, 0], [0, 1], [0, 2], [1, 2], [2, 2], [2, 1], [2, 0], [1, 0]];
    var pIndex = {};
    ordem.forEach(function(pos, p){ pIndex[pos[0] + ',' + pos[1]] = p; });
    var cenarioAtivo = 2;

    function marcarBotaoAtivo(n){
      cenarioAtivo = n;
      btn1.classList.toggle('sim-ep0704_active', n === 1);
      btn2.classList.toggle('sim-ep0704_active', n === 2);
    }

    function render(){
      grid.innerHTML = '';
      for (var r = 0; r < 3; r++){
        for (var c = 0; c < 3; c++){
          (function(r, c){
            var v = valores[r][c];
            var central = (r === 1 && c === 1);
            var div = document.createElement('div');

            var bordaCor = central ? '#26241d' : '#e4dcc8';
            var textoCor = v > 128 ? '#26241d' : '#ffffff';

            div.style.cssText = 'position:relative; height:60px; display:flex; align-items:center; justify-content:center; font-family:monospace; font-weight:700; border-radius:6px; cursor:' + (central ? 'default' : 'pointer') + '; border:2px solid ' + bordaCor + '; background:rgb(' + v + ',' + v + ',' + v + '); color:' + textoCor + '; transition:all 0.15s ease;';
            div.textContent = v;

            if (!central){
              var pLabel = document.createElement('span');
              pLabel.textContent = 'p' + pIndex[r + ',' + c];
              pLabel.style.cssText = 'position:absolute; top:2px; left:4px; font-size:9px; font-weight:400; opacity:0.8;';
              div.appendChild(pLabel);

              div.addEventListener('click', function(){
                valores[r][c] = valores[r][c] >= 128 ? 10 : 200;
                cenarioAtivo = null;
                btn1.classList.remove('sim-ep0704_active');
                btn2.classList.remove('sim-ep0704_active');
                render();
              });
            }
            grid.appendChild(div);
          })(r, c);
        }
      }

      var gc = valores[1][1];
      var bits = ordem.map(function(pos){ return valores[pos[0]][pos[1]] >= gc ? 1 : 0; });
      var lbp = 0;
      bits.forEach(function(b, p){ lbp += b * Math.pow(2, p); });

      var trans = 0;
      for (var i = 0; i < 8; i++){
        if (bits[i] !== bits[(i + 1) % 8]) trans++;
      }

      var classe = trans <= 2 ? 'UNIFORME' : 'NÃO-UNIFORME';

      if (trans <= 2) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'bit (p0..p7) = ' + bits.join('') + '  |  LBP = ' + lbp + '  |  transições = ' + trans + '  |  ' + classe;
    }

    btn1.addEventListener('click', function(){
      valores = exemplos[1].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(1);
      render();
    });

    btn2.addEventListener('click', function(){
      valores = exemplos[2].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(2);
      render();
    });

    marcarBotaoAtivo(2);
    render();
  }

  function tryInitSim07Ep04(){
    var root = document.getElementById('sim-ep0704');
    if (root) initSim07Ep04(root); else setTimeout(tryInitSim07Ep04, 200);
  }
  tryInitSim07Ep04();
})();
</script>
""")

**Figura 7.24:** Simulatore EP07_04: Codice LBP di un vicinato 3×3


<figure id="fig-07-sim-ep0704">
  <img src="imagens/fig-07-sim-ep0704.png" alt=" Simulatore EP07_04: Codice LBP di un vicinato 3×3 " style="max-width:80%" />
  <figcaption><strong>Figura 7.24:</strong>  Simulatore EP07_04: Codice LBP di un vicinato 3×3 </figcaption>
</figure>

In [31]:
%%writefile EP07_04.py
# Codice Python

Overwriting EP07_04.py


In [32]:
TestSuite("EP07_04.py").run()

### 7.0.5 EP07_05 🔴 Istogramma delle Orientazioni di una Cella HOG

La funzione `hog` di `scikit-image`, impiegata nel progetto di classificazione delle cifre, divide l'immagine in piccole **celle** e, per ciascuna, costruisce un istogramma delle orientazioni del gradiente ponderato per la magnitudine — esattamente il passaggio centrale descritto nella sezione sul descrittore HOG del capitolo.

Sei stato incaricato di implementare questo calcolo per una singola cella, a partire dai valori di magnitudine e orientazione del gradiente **già calcolati** per ogni pixel della cella (tralasciando il calcolo delle derivate parziali).

#### 7.0.5.1 📋 Linee Guida di Implementazione

1. **Dimensioni:** Leggere gli interi $n$ (la cella ha $n \times n$ pixel) e $B$ (numero di contenitori dell'istogramma).
2. **Magnitudini:** Leggere $n$ righe con $n$ valori reali ciascuna, che rappresentano $|\nabla f(x,y)|$ per ogni pixel della cella.
3. **Orientazioni:** Leggere altre $n$ righe con $n$ valori reali ciascuna, che rappresentano $\theta(x,y)$ in **gradi**, già convertiti nell'intervallo **non orientato** $[0^\circ, 180^\circ)$, come convenzionalmente utilizzato dal HOG.
4. **Contenitori:** I $B$ contenitori coprono $[0^\circ, 180^\circ)$ in fasce uguali di larghezza $180/B$ gradi. Un pixel con orientazione $\theta$ appartiene al contenitore $\lfloor \theta / (180/B) \rfloor$; se questo indice è uguale a $B$ (possibile solo quando $\theta$ è esattamente $180^\circ$, cosa che non dovrebbe verificarsi secondo la direttiva del punto 3), utilizzare il contenitore $B-1$.
5. **Istogramma grezzo:** Per ogni pixel, accumulare la sua **magnitudine** (non il suo conteggio) nel contenitore corrispondente:
$$
H[b] = \sum_{(x,y)\, :\, \text{bin}(\theta(x,y)) = b} |\nabla f(x,y)|.
$$
6. **Normalizzazione L2:** Dopo aver costruito $H$, normalizzarlo per ottenere $\hat H$:
$$
\hat H[b] = \frac{H[b]}{\sqrt{\sum_{j=0}^{B-1} H[j]^2 + \epsilon}}, \qquad \epsilon = 10^{-6}.
$$
7. **Output:** Stampare l'istogramma grezzo $H$ (arrotondato a 2 cifre decimali) su una riga, seguito dall'istogramma normalizzato $\hat H$ (arrotondato a 4 cifre decimali) su un'altra riga, entrambi con i $B$ valori separati da spazi, nell'ordine dei contenitori.

#### 7.0.5.2 📌 Vincoli Computazionali

* ***Binning* non orientato:** l'intervallo delle orientazioni è $[0,180)$, non $[0,360)$ — i gradienti in direzioni opposte (differenza di $180^\circ$) contribuiscono allo **stesso** contenitore, convenzione standard del HOG per il rilevamento di oggetti.
* **Accumulo per magnitudine, non per conteggio:** l'istogramma pondera ogni pixel per la sua magnitudine del gradiente, non conta semplicemente quanti pixel cadono in ciascun contenitore.
* **Costante di stabilizzazione:** l'$\epsilon = 10^{-6}$ al denominatore della normalizzazione evita la divisione per zero quando la cella è completamente omogenea (tutte le magnitudini nulle).

#### 7.0.5.3 📐 Da dove provengono le matrici di ingresso

Prima di questo EP, ogni pixel $(x,y)$ dell'immagine passa attraverso:

$$
G_x = f(x+1,y)-f(x-1,y), \qquad G_y = f(x,y+1)-f(x,y-1)
$$

$$
|\nabla f| = \sqrt{G_x^2+G_y^2}, \qquad \theta_{\text{segnalato}} = \operatorname{atan2}(G_y,G_x)
$$

Poiché il HOG ignora la polarità del contrasto, l'angolo viene raddoppiato nell'intervallo non orientato:

$$
\theta = \theta_{\text{segnalato}} \bmod 180°
$$

Ripetendo questo per tutti i pixel di una cella $n\times n$, si ottengono le due matrici di ingresso di questo esercizio: **magnitudini** $|\nabla f|$ e **orientazioni** $\theta \in [0°,180°)$.

#### 7.0.5.4 🧠 Fondamenti Teorici

| Fase | Ruolo |
|---|---|
| Magnitudine del gradiente | Pondera il contributo di ogni pixel — i bordi forti pesano più del rumore debole |
| Orientazione non orientata | Rende il descrittore invariante alla polarità del contrasto (chiaro→scuro vs. scuro→chiaro) |
| Istogramma per cella | Riassume la distribuzione locale dei bordi in un vettore compatto |
| Normalizzazione L2 | Riduce la sensibilità del descrittore alle variazioni globali di illuminazione e contrasto |

La concatenazione degli istogrammi normalizzati di tutte le celle dell'immagine — non implementata in questo esercizio — forma il vettore di caratteristiche HOG completo, utilizzato come ingresso del classificatore k-NN nel progetto del capitolo.

#### 7.0.5.5 📦 Specifica di Ingresso e Uscita (VPL)

**Ingresso:**

* Riga 1: Interi $n$ e $B$.
* Prossime $n$ righe: $n$ magnitudini reali ciascuna.
* Prossime $n$ righe: $n$ orientazioni reali (gradi, $[0,180)$) ciascuna.

**Uscita:**

* Riga 1: i $B$ valori dell'istogramma grezzo, arrotondati a 2 cifre decimali.
* Riga 2: i $B$ valori dell'istogramma normalizzato, arrotondati a 4 cifre decimali.

#### 7.0.5.6 📌 Esempi

| Ingresso | Uscita | Osservazione |
|---|---|---|
| 2 2<br>1.0 2.0<br>3.0 4.0<br>10 100<br>170 20 | 5.00 5.00<br>0.7071 0.7071 | Contenitore di larghezza 90°: $[0,90)$ e $[90,180)$; le magnitudini 1 e 4 cadono nel contenitore 0, 2 e 3 nel contenitore 1. |
| 2 4<br>0.0 0.0<br>0.0 0.0<br>0 0<br>0 0 | 0.00 0.00 0.00 0.00<br>0.0000 0.0000 0.0000 0.0000 | Cella omogenea: $\epsilon$ evita la divisione per zero. |

In [33]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0705" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0705 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0705 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0705 button:hover { background: #e8dfcf; }
  #sim-ep0705 button.sim-ep0705_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0705_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0705_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_05: Istogramma delle Orientazioni di una Cella</span>
  <span class="sim-ep0705_pill">🔴 cella 3×3 fissa</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Regola B e osserva come la <b>matrice delle orientazioni</b> (indipendente da quella delle magnitudini) viene mappata
      nei compartimenti tramite <code>bin = floor(θ / (180/B))</code>, e come le magnitudini vengono sommate in ciascun bin.
    </p>

    <!-- Controle B -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Numero di compartimenti (B)</label>
        <span id="ep0705_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">2</span>
      </div>
      <input id="ep0705_sl" style="width:100%;accent-color:#2980b9;" max="6" min="2" step="1" type="range" value="2">
    </div>

    <!-- Entrada bruta (formato VPL) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📄 Input (esattamente come il programma legge da stdin)</div>
      <pre id="ep0705_stdin" style="background:#1e1e1e;color:#d4d4d4;border-radius:8px;padding:12px 14px;font-size:12px;line-height:1.5;overflow-x:auto;margin:0;"></pre>
    </div>

    <!-- Duas matrizes separadas -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;margin-bottom:20px;">
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🔢 Matrice delle magnitudini |∇f|</div>
        <div id="ep0705_mag_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📐 Matrice delle orientazioni θ (gradi) — colorata per bin</div>
        <div id="ep0705_ang_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
    </div>

    <!-- Regua 0-180 -->
    <div style="margin-bottom:22px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:10px;">📏 Dove ciascun θ cade sul righello [0°, 180°) — <code>bin = floor(θ / larghezza)</code></div>
      <div style="position:relative;height:70px;margin:0 6px;">
        <div id="ep0705_regua" style="position:absolute;top:28px;left:0;right:0;height:14px;border-radius:7px;overflow:hidden;display:flex;border:1px solid #d1d5db;"></div>
        <div id="ep0705_regua_ticks" style="position:absolute;top:44px;left:0;right:0;height:14px;"></div>
        <div id="ep0705_regua_marcas" style="position:absolute;top:0;left:0;right:0;height:26px;"></div>
      </div>
    </div>

    <!-- Faixas dos compartimentos -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📊 Intervalli di ciascun compartimento (larghezza = 180° / B)</div>
      <div id="ep0705_faixas" style="display:flex;flex-wrap:wrap;gap:6px;"></div>
    </div>

    <!-- Grade de pixels colorida por bin (mag + ang juntos) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧩 Ogni pixel: magnitudine + orientazione → bin</div>
      <div id="ep0705_pixels" style="display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <!-- Botões -->
    <div style="display:flex;gap:8px;justify-content:center;margin-bottom:14px;">
      <button id="ep0705_btn_raw" class="ep0705_btn">Istogramma grezzo (H)</button>
      <button id="ep0705_btn_norm" class="ep0705_btn">Istogramma normalizzato (Ĥ)</button>
    </div>

    <!-- Barras -->
    <div id="ep0705_bars" style="display:flex;gap:6px;align-items:flex-end;height:120px;justify-content:center;margin-bottom:14px;"></div>

    <!-- Passo a passo -->
    <div style="margin-bottom:6px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧮 Calcolo passo passo (floor della divisione + somma delle magnitudini per bin)</div>
      <div id="ep0705_passos" style="background:#f3f4f6;border-radius:8px;padding:10px 12px;font-family:monospace;font-size:11px;color:#374151;line-height:1.8;"></div>
    </div>

    <div id="ep0705_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0705 .ep0705_btn { font-size:11px;padding:6px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0705 .ep0705_btn.ativo { background:#2980b9;color:#fff;border-color:#2980b9; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var n = 3;
    var mags = [[1.0,2.0,0.5],[3.0,4.0,1.5],[0.8,2.5,3.2]];
    var angs = [[10,100,45],[170,20,95],[60,150,5]];
    var CORES = ["#6366f1","#0ea5e9","#10b981","#f59e0b","#ef4444","#a855f7"];

    var slEl = root.querySelector("#ep0705_sl");
    var vlEl = root.querySelector("#ep0705_vl");
    var stdinEl = root.querySelector("#ep0705_stdin");
    var magGridEl = root.querySelector("#ep0705_mag_grid");
    var angGridEl = root.querySelector("#ep0705_ang_grid");
    var reguaEl = root.querySelector("#ep0705_regua");
    var reguaTicksEl = root.querySelector("#ep0705_regua_ticks");
    var reguaMarcasEl = root.querySelector("#ep0705_regua_marcas");
    var faixasEl = root.querySelector("#ep0705_faixas");
    var pxEl = root.querySelector("#ep0705_pixels");
    var bars = root.querySelector("#ep0705_bars");
    var passosEl = root.querySelector("#ep0705_passos");
    var dbg = root.querySelector("#ep0705_debug");
    var btnRaw = root.querySelector("#ep0705_btn_raw");
    var btnNorm = root.querySelector("#ep0705_btn_norm");
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle("ativo", !modoNorm);
      btnNorm.classList.toggle("ativo", modoNorm);

      var B = parseInt(slEl.value);
      vlEl.textContent = B;
      var largura = 180/B;

      // ---- Entrada bruta (stdin) ----
      var linhas = [];
      linhas.push(n + " " + B);
      mags.forEach(function(row){ linhas.push(row.map(function(v){return v.toFixed(1);}).join(" ")); });
      angs.forEach(function(row){ linhas.push(row.join(" ")); });
      stdinEl.textContent = linhas.join("\\n");

      // ---- bin de cada pixel (floor(theta/largura), clip) ----
      var binsMat = [];
      for(var i=0;i<n;i++){
        binsMat.push([]);
        for(var j=0;j<n;j++){
          var raw = angs[i][j]/largura;
          var b = Math.floor(raw);
          if(b > B-1) b = B-1;
          if(b < 0) b = 0;
          binsMat[i].push(b);
        }
      }

      // ---- Matriz de magnitudes (grid simples) ----
      magGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var d = document.createElement("div");
          d.style.cssText = "text-align:center;border-radius:8px;padding:8px 4px;font-size:12px;font-family:monospace;background:#f9fafb;border:1px solid #e5e7eb;color:#374151;";
          d.textContent = mags[i][j].toFixed(1);
          magGridEl.appendChild(d);
        }
      }

      // ---- Matriz de orientações (colorida por bin, com floor explícito) ----
      angGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var b2 = binsMat[i][j];
          var cor2 = CORES[b2];
          var raw2 = angs[i][j]/largura;
          var d2 = document.createElement("div");
          d2.style.cssText = "text-align:center;border-radius:8px;padding:6px 4px;font-size:11px;font-family:monospace;background:"+cor2+"22;border:2px solid "+cor2+";color:#374151;";
          d2.innerHTML = "<div style=\\"font-weight:700;\\">"+angs[i][j]+"°</div>"+
            "<div style=\\"font-size:9px;color:#6b7280;\\">÷"+largura.toFixed(1)+"="+raw2.toFixed(2)+"</div>"+
            "<div style=\\"font-size:9px;font-weight:700;color:"+cor2+";\\">⌊·⌋=bin "+b2+"</div>";
          angGridEl.appendChild(d2);
        }
      }

      // ---- Régua 0-180 com faixas coloridas ----
      reguaEl.innerHTML = "";
      for(var b3=0;b3<B;b3++){
        var seg = document.createElement("div");
        seg.style.cssText = "flex:1;background:"+CORES[b3]+";opacity:0.35;border-right:1px solid rgba(255,255,255,0.6);";
        reguaEl.appendChild(seg);
      }
      // ticks (limites dos bins)
      reguaTicksEl.innerHTML = "";
      for(var b4=0;b4<=B;b4++){
        var pct = (b4*largura/180*100);
        var tick = document.createElement("div");
        tick.style.cssText = "position:absolute;left:"+pct+"%;top:0;font-size:9px;color:#6b7280;transform:translateX(-50%);white-space:nowrap;";
        tick.textContent = (b4*largura).toFixed(0)+"°";
        reguaTicksEl.appendChild(tick);
      }
      // marcadores dos angulos de cada pixel
      reguaMarcasEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var ang = angs[i][j];
          var b5 = binsMat[i][j];
          var pctm = (ang/180*100);
          var marker = document.createElement("div");
          marker.style.cssText = "position:absolute;left:"+pctm+"%;top:0;transform:translateX(-50%);display:flex;flex-direction:column;align-items:center;";
          marker.innerHTML = "<div style=\\"font-size:9px;color:"+CORES[b5]+";font-weight:700;\\">("+i+","+j+")</div>"+
            "<div style=\\"width:0;height:0;border-left:5px solid transparent;border-right:5px solid transparent;border-top:8px solid "+CORES[b5]+";\\"></div>";
          reguaMarcasEl.appendChild(marker);
        }
      }

      // ---- Faixas dos bins (legenda) ----
      faixasEl.innerHTML = "";
      for(var b=0;b<B;b++){
        var lo = (b*largura).toFixed(1);
        var hi = ((b+1)*largura).toFixed(1);
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:6px;background:#f9fafb;border:1px solid #e5e7eb;border-radius:20px;padding:4px 10px;font-size:11px;color:#374151;";
        chip.innerHTML = "<span style=\\"width:10px;height:10px;border-radius:50%;background:"+CORES[b]+";display:inline-block;\\"></span>bin "+b+": ["+lo+"°, "+hi+"°)";
        faixasEl.appendChild(chip);
      }

      // ---- Atribuição por pixel + histograma bruto ----
      var H = new Array(B).fill(0);
      var binsPorPixel = [];
      pxEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var mag = mags[i][j], ang = angs[i][j];
          var bin = binsMat[i][j];
          binsPorPixel.push({i:i, j:j, mag:mag, ang:ang, bin:bin});
          H[bin] += mag;

          var div = document.createElement("div");
          var cor = CORES[bin];
          div.style.cssText = "text-align:center;border-radius:10px;padding:8px 6px;font-size:11px;background:"+cor+"22;border:2px solid "+cor+";color:#374151;";
          div.innerHTML = "<div style=\\"font-weight:700;\\">mag="+mag.toFixed(1)+"</div>"+
            "<div style=\\"font-family:monospace;\\">θ="+ang+"°</div>"+
            "<div style=\\"font-weight:700;color:"+cor+";\\">→ bin "+bin+"</div>";
          pxEl.appendChild(div);
        }
      }

      var denom = Math.sqrt(H.reduce(function(s,v){return s+v*v;},0) + 1e-6);
      var Hn = H.map(function(v){ return v/denom; });

      // ---- Barras (coloridas por bin) ----
      var dados = modoNorm ? Hn : H;
      var maxD = Math.max.apply(null, dados.concat([0.001]));
      bars.innerHTML = "";
      dados.forEach(function(v, b){
        var col = document.createElement("div");
        col.style.cssText = "display:flex;flex-direction:column;align-items:center;gap:4px;";
        var barra = document.createElement("div");
        var altura = Math.round((v/maxD)*90) + 4;
        barra.style.cssText = "width:34px;height:"+altura+"px;background:"+CORES[b]+";border-radius:4px 4px 0 0;";
        var label = document.createElement("div");
        label.style.cssText = "font-family:monospace;font-size:10px;color:#4b5563;";
        label.textContent = modoNorm ? v.toFixed(4) : v.toFixed(2);
        var binLabel = document.createElement("div");
        binLabel.style.cssText = "font-size:9px;color:#9ca3af;";
        binLabel.textContent = "bin "+b;
        col.appendChild(barra);
        col.appendChild(label);
        col.appendChild(binLabel);
        bars.appendChild(col);
      });

      // ---- Passo a passo (floor + soma) ----
      var passos = [];
      for(var b=0;b<B;b++){
        var contribs = binsPorPixel.filter(function(p){ return p.bin===b; });
        var termos = contribs.map(function(p){ return p.mag.toFixed(2)+" (θ="+p.ang+"°→⌊"+(p.ang/largura).toFixed(2)+"⌋="+p.bin+")"; }).join(" + ");
        if(termos === "") termos = "(nenhum pixel)";
        passos.push("<span style=\\"color:"+CORES[b]+";font-weight:700;\\">H["+b+"]</span> = "+termos+" = <b>"+H[b].toFixed(2)+"</b>");
      }
      passosEl.innerHTML = passos.join("<br>");

      dbg.textContent = "H=[" + H.map(function(v){return v.toFixed(2);}).join(", ") + "]  |  Ĥ=[" +
        Hn.map(function(v){return v.toFixed(4);}).join(", ") + "]";
    }

    slEl.addEventListener("input", render);
    btnRaw.addEventListener("click", function(){ modoNorm = false; render(); });
    btnNorm.addEventListener("click", function(){ modoNorm = true; render(); });
    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0705");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.25:** Simulatore EP07_05: Istogramma HOG di una Cella (mappatura degli angoli ai bins)


<figure id="fig-07-sim-ep0705">
  <img src="imagens/fig-07-sim-ep0705.png" alt=" Simulatore EP07_05: Istogramma HOG di una Cella (mappatura degli angoli ai bins) " style="max-width:80%" />
  <figcaption><strong>Figura 7.25:</strong>  Simulatore EP07_05: Istogramma HOG di una Cella (mappatura degli angoli ai bins) </figcaption>
</figure>

In [34]:
%%writefile EP07_05.py
# Codice Python

Overwriting EP07_05.py


In [35]:
TestSuite("EP07_05.py").run()

### 7.0.6 EP07_06 🟣 *Pipeline* Completo: Descrittori + k-NN + Valutazione Multi-Classe

Questo esercizio integra le tre fasi centrali del capitolo in un unico *pipeline*, riproducendo in miniatura il **Progetto Pratico 2** (classificazione di trame sintetiche tramite LBP): un insieme di istogrammi di descrittori **già estratti** (come se fossero istogrammi LBP) viene utilizzato per addestrare un classificatore k-NN, che a sua volta viene valutato su un insieme di test indipendente mediante una matrice di confusione multi-classe.

A differenza dell'EP07_01, qui lo spazio delle caratteristiche ha dimensione arbitraria $H$ (la dimensione dell'istogramma), esistono più di due classi e la metrica di distanza è un parametro di input — consentendo di riprodurre l'esperimento di confronto delle metriche discusso nel capitolo.

#### 7.0.6.1 📋 Linee Guida di Implementazione

1. **Classi:** Leggere l'intero $C$ (numero di classi) seguito da $C$ nomi di classe (*stringhe* senza spazi), nell'ordine in cui devono apparire nella matrice di confusione.
2. **Configurazione:** Leggere l'intero $H$ (dimensione degli istogrammi), la *stringa* $M$ (metrica: `euclidiana` o `manhattan`) e l'intero dispari $k$.
3. **Addestramento:** Leggere l'intero $N$ e, successivamente, $N$ righe, ciascuna contenente il nome della classe seguito da $H$ valori reali (l'istogramma del descrittore).
4. **Test:** Leggere l'intero $Q$ e, successivamente, $Q$ righe, ciascuna contenente il nome della classe **reale** seguito da $H$ valori reali (l'istogramma del descrittore del campione di test).
5. **Distanza:** Per ogni campione di test, calcolare la distanza da ogni esempio di addestramento utilizzando la metrica $M$:
$$
d_{\text{euclidiana}}(u,v) = \sqrt{\sum_{j=1}^{H}(u_j-v_j)^2}, \qquad
d_{\text{manhattan}}(u,v) = \sum_{j=1}^{H} |u_j - v_j|.
$$
6. **Classificazione k-NN:** Selezionare i $k$ esempi di addestramento più vicini (in caso di parità di distanza, dare la precedenza all'ordine di lettura, come nell'EP07_01) e classificare in base alla classe maggioritaria tra di essi. In caso di **pareggio di voti** tra due o più classi, scegliere quella che appare **per prima** nella lista di classi del punto 1.
7. **Matrice di confusione:** Costruire una matrice $C \times C$ in cui la riga corrisponde alla classe reale e la colonna alla classe prevista, seguendo l'ordine delle classi del punto 1.
8. **Accuratezza:** Calcolare l'accuratezza globale come rapporto tra i successi e $Q$.
9. **Output:** Per ogni campione di test, nell'ordine di input, stampare la classe prevista. Successivamente, stampare la matrice di confusione (una riga per classe reale, valori separati da spazi, nell'ordine delle classi). Infine, stampare l'accuratezza arrotondata a 4 cifre decimali.

#### 7.0.6.2 📌 Vincoli Computazionali

* **Metrica selezionabile:** implementare entrambe le distanze; la metrica $M$ definisce quale viene utilizzata per l'intera esecuzione (non è possibile mescolare metriche nella stessa chiamata).
* **Pareggio di voti deterministico:** il criterio del punto 6 (ordine della lista delle classi) deve essere seguito anche quando il pareggio coinvolge più di due classi.
* **Indipendenza tra addestramento e test:** non è necessario verificare che i campioni di test non appaiano nell'addestramento — si assume che l'input sia valido.

#### 7.0.6.3 🧠 Fondamenti Teorici

| Fase dell'esercizio | Fase corrispondente nel capitolo |
|---|---|
| Istogrammi di addestramento/test già estratti | `descritor_lbp` applicato alle trame sintetiche |
| Distanza euclidea o Manhattan | Parametro `metric` del `KNeighborsClassifier` |
| Votazione maggioritaria con $k$ vicini | `KNeighborsClassifier.predict` |
| Matrice di confusione $C\times C$ | `confusion_matrix` di `scikit-learn` |
| Accuratezza globale | `accuracy_score` di `scikit-learn` |

Questo esercizio evidenzia, in modo controllato, un risultato discusso nel capitolo: la **scelta della metrica di distanza** e del **valore di $k$** può modificare la classe prevista per lo stesso campione, anche mantenendo fisso il descrittore utilizzato — rafforzando l'idea che, nel riconoscimento di pattern classico, il descrittore, la metrica e il classificatore formano un sistema interdipendente, e non componenti isolate.

#### 7.0.6.4 📦 Specifica di Input e Output (VPL)

**Input:**

* Riga 1: intero $C$ seguito da $C$ nomi di classe.
* Riga 2: intero $H$, *stringa* $M$ e intero $k$.
* Riga 3: intero $N$.
* Prossime $N$ righe di addestramento: nome della classe seguito da $H$ reali.
* Riga successiva: intero $Q$.
* Prossime $Q$ righe di test: nome della classe reale seguito da $H$ reali.

**Output:**

* $Q$ righe con la classe prevista per ogni campione di test, nell'ordine di input.
* $C$ righe con la matrice di confusione (una riga per classe reale).
* Ultima riga: `Acuracia: <valore>`.

#### 7.0.6.5 📌 Esempi

| Input (riassunto) | Output | Osservazione |
|---|---|---|
| 2 granular listrada<br>2 euclidiana 1<br>4<br>granular 0.9 0.1<br>granular 0.8 0.2<br>listrada 0.1 0.9<br>listrada 0.2 0.8<br>2<br>granular 0.85 0.15<br>listrada 0.15 0.85 | granular<br>listrada<br>1 0<br>0 1<br>Acuracia: 1.0000 | Con $k=1$, ogni test viene classificato dal vicino di addestramento più prossimo. |

> ### 📝 Nota
>
> Questo simulatore utilizza un insieme semplificato di **3 classi** (`granulare`, `a strisce`, `maculata`) su punti 2D fittizi, unicamente per illustrare il *pipeline* di votazione, spareggio e matrice di confusione del k-NN. Nel **EP07_07**, applicherai questa stessa logica a un mosaico di un'immagine reale, che introduce una quarta classe (`a scacchi`) e sostituisce i punti 2D con istogrammi LBP estratti direttamente dai pixel dell'immagine.

In [36]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0706" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0706 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0706 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0706 button:hover { background: #e8dfcf; }
  #sim-ep0706 button.sim-ep0706_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0706_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0706_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_06: Pipeline k-NN Multi-classe</span>
  <span class="sim-ep0706_pill">6 Addestramento &middot; 3 Test &middot; 3 Classi</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Scegli la metrica, il valore di k e il campione di test (★). Osserva i k vicini più prossimi, la votazione,
      il pareggio quando necessario, e come ciò si propaga alla matrice di confusione e all'accuratezza dell'intero set.
    </p>

    <!-- Controles -->
    <div style="display:flex;flex-wrap:wrap;gap:18px;justify-content:center;margin-bottom:16px;">
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Metrica (M)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_be" class="ep0706_btn">Euclidea</button>
          <button id="ep0706_bm" class="ep0706_btn">Manhattan</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Vicini (k)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_k1" class="ep0706_btn">k=1</button>
          <button id="ep0706_k3" class="ep0706_btn">k=3</button>
          <button id="ep0706_k5" class="ep0706_btn">k=5</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Campione di test (★)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_t0" class="ep0706_btn">test 1</button>
          <button id="ep0706_t1" class="ep0706_btn">test 2</button>
          <button id="ep0706_t2" class="ep0706_btn">test 3</button>
        </div>
      </div>
    </div>

    <!-- Legenda -->
    <div id="ep0706_legenda" style="display:flex;gap:10px;justify-content:center;margin-bottom:10px;"></div>

    <!-- Dispersao 2D -->
    <div style="max-width:340px;margin:0 auto 16px auto;height:300px;border:1px solid #e5e7eb;border-radius:12px;background:#fafafa;">
      <div id="ep0706_svg_container" style="width:100%;height:100%;"></div>
    </div>

    <!-- Distancias ordenadas -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📏 Distanze fino al campione di test (ordinate) — <span style="font-weight:400;font-size:10px;color:#8a8672;">#i = ordine di lettura nella lista di addestramento (passa il mouse)</span></div>
      <div id="ep0706_dists" style="display:grid;grid-template-columns:1fr 1fr;gap:2px 10px;font-family:monospace;font-size:10px;"></div>
    </div>

    <!-- Votacao -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🗳️ Votazione tra i k vicini</div>
      <div id="ep0706_votos" style="display:flex;gap:10px;justify-content:center;margin-bottom:6px;"></div>
      <div id="ep0706_previsao" style="text-align:center;font-size:12px;font-weight:bold;"></div>
    </div>

    <!-- Matriz de confusao + acuracia (conjunto de teste inteiro) -->
    <div style="margin-bottom:8px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📋 Matrice di confusione e accuratezza — eseguendo il pipeline sui 3 campioni di test</div>
      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;justify-content:center;">
        <table id="ep0706_cm" style="border-collapse:collapse;font-size:11px;font-family:monospace;"></table>
        <div id="ep0706_acc" style="font-size:13px;font-weight:bold;color:#5e5a4a;"></div>
      </div>
    </div>

    <div id="ep0706_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0706 .ep0706_btn { font-size:11px;padding:5px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0706 .ep0706_btn.ativo { background:#7c3aed;color:#fff;border-color:#7c3aed; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var classes = ["granular","listrada","manchada"];
    var CORES = {granular:"#6366f1", listrada:"#f59e0b", manchada:"#10b981"};

    var trainPts = [
      {nome:"granular_1", cls:"granular", x:0.70, y:0.70},
      {nome:"granular_2", cls:"granular", x:0.25, y:0.85},
      {nome:"listrada_1", cls:"listrada", x:0.85, y:0.50},
      {nome:"listrada_2", cls:"listrada", x:0.60, y:0.15},
      {nome:"manchada_1", cls:"manchada", x:0.30, y:0.30},
      {nome:"manchada_2", cls:"manchada", x:0.15, y:0.55}
    ];
    var testPts = [
      {nome:"teste 1", cls:"granular", x:0.50, y:0.50},
      {nome:"teste 2", cls:"listrada", x:0.70, y:0.20},
      {nome:"teste 3", cls:"manchada", x:0.20, y:0.40}
    ];

    var svgContainer = root.querySelector("#ep0706_svg_container");
    var svg = svgNS("svg");
    svg.setAttribute("viewBox", "0 0 100 100");
    svg.setAttribute("style", "width:100%;height:100%;");
    svgContainer.appendChild(svg);
    var legendaEl = root.querySelector("#ep0706_legenda");
    var distsEl = root.querySelector("#ep0706_dists");
    var votosEl = root.querySelector("#ep0706_votos");
    var previsaoEl = root.querySelector("#ep0706_previsao");
    var cmEl = root.querySelector("#ep0706_cm");
    var accEl = root.querySelector("#ep0706_acc");
    var dbg = root.querySelector("#ep0706_debug");

    var be = root.querySelector("#ep0706_be"), bm = root.querySelector("#ep0706_bm");
    var bk1 = root.querySelector("#ep0706_k1"), bk3 = root.querySelector("#ep0706_k3"), bk5 = root.querySelector("#ep0706_k5");
    var bt0 = root.querySelector("#ep0706_t0"), bt1 = root.querySelector("#ep0706_t1"), bt2 = root.querySelector("#ep0706_t2");

    var metrica = "euclidiana", k = 1, testSel = 0;

    function dist(u, v){
      var dx = u.x-v.x, dy = u.y-v.y;
      if(metrica === "euclidiana") return Math.sqrt(dx*dx+dy*dy);
      return Math.abs(dx)+Math.abs(dy);
    }

    function knnPredict(xtest){
      var ds = trainPts.map(function(p, i){ return {i:i, p:p, d:dist(xtest, p)}; });
      ds.sort(function(a,b){ return a.d - b.d; }); // ordem estavel = desempate por ordem de leitura
      var viz = ds.slice(0, k);
      var votos = {}; classes.forEach(function(c){ votos[c]=0; });
      viz.forEach(function(v){ votos[v.p.cls]++; });
      var maxV = Math.max.apply(null, classes.map(function(c){return votos[c];}));
      var empatados = classes.filter(function(c){ return votos[c]===maxV; });
      var pred = empatados[0]; // primeira classe da lista entre as empatadas
      return {pred:pred, viz:viz, votos:votos, empatados:empatados, ordenados:ds};
    }

    function svgNS(tag){
      // Concatenado de propósito: evita que filtros de auto-link do Moodle
      // reconheçam "http://www.w3.org/2000/svg" como URL e insiram uma tag <a>
      // dentro desta string, o que quebraria a sintaxe do createElementNS.
      var SVG_NS = "http" + "://www.w3.org/2000/svg";
      return document.createElementNS(SVG_NS, tag);
    }

    function render(){
      be.classList.toggle("ativo", metrica==="euclidiana");
      bm.classList.toggle("ativo", metrica==="manhattan");
      bk1.classList.toggle("ativo", k===1);
      bk3.classList.toggle("ativo", k===3);
      bk5.classList.toggle("ativo", k===5);
      bt0.classList.toggle("ativo", testSel===0);
      bt1.classList.toggle("ativo", testSel===1);
      bt2.classList.toggle("ativo", testSel===2);

      // Legenda
      legendaEl.innerHTML = "";
      classes.forEach(function(c){
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:5px;font-size:11px;color:#374151;";
        chip.innerHTML = '<span style="width:10px;height:10px;border-radius:50%;background:'+CORES[c]+';display:inline-block;"></span>'+c;
        legendaEl.appendChild(chip);
      });

      var xt = testPts[testSel];
      var r = knnPredict(xt);
      var vizIdx = r.viz.map(function(v){ return v.i; });

      // ---- SVG: pontos de treino, linhas para vizinhos, estrela de teste ----
      svg.innerHTML = "";
      // grade leve
      for(var g=1; g<4; g++){
        var lineV = svgNS("line");
        lineV.setAttribute("x1", g*25); lineV.setAttribute("y1", 0);
        lineV.setAttribute("x2", g*25); lineV.setAttribute("y2", 100);
        lineV.setAttribute("stroke", "#eee"); lineV.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineV);
        var lineH = svgNS("line");
        lineH.setAttribute("x1", 0); lineH.setAttribute("y1", g*25);
        lineH.setAttribute("x2", 100); lineH.setAttribute("y2", g*25);
        lineH.setAttribute("stroke", "#eee"); lineH.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineH);
      }
      // linhas ate os vizinhos (desenhadas antes dos pontos, para ficarem por baixo)
      vizIdx.forEach(function(i){
        var p = trainPts[i];
        var line = svgNS("line");
        line.setAttribute("x1", xt.x*100); line.setAttribute("y1", (1-xt.y)*100);
        line.setAttribute("x2", p.x*100); line.setAttribute("y2", (1-p.y)*100);
        line.setAttribute("stroke", CORES[p.cls]); line.setAttribute("stroke-width", "0.6");
        line.setAttribute("stroke-dasharray", "1.5,1"); line.setAttribute("opacity", "0.7");
        svg.appendChild(line);
      });
      // pontos de treino
      trainPts.forEach(function(p, i){
        var isViz = vizIdx.indexOf(i) !== -1;
        if(isViz){
          var halo = svgNS("circle");
          halo.setAttribute("cx", p.x*100); halo.setAttribute("cy", (1-p.y)*100);
          halo.setAttribute("r", 5); halo.setAttribute("fill", "none");
          halo.setAttribute("stroke", CORES[p.cls]); halo.setAttribute("stroke-width", "0.8");
          svg.appendChild(halo);
        }
        var c = svgNS("circle");
        c.setAttribute("cx", p.x*100); c.setAttribute("cy", (1-p.y)*100);
        c.setAttribute("r", 3.2);
        c.setAttribute("fill", CORES[p.cls]);
        c.setAttribute("stroke", "#fff"); c.setAttribute("stroke-width", "0.6");
        c.setAttribute("opacity", isViz ? "1" : "0.55");
        svg.appendChild(c);
      });
      // estrela de teste
      var correto = (r.pred === xt.cls);
      var estCor = correto ? "#16a34a" : "#dc2626";
      var halo2 = svgNS("circle");
      halo2.setAttribute("cx", xt.x*100); halo2.setAttribute("cy", (1-xt.y)*100);
      halo2.setAttribute("r", 5.5); halo2.setAttribute("fill", "#fff");
      halo2.setAttribute("stroke", estCor); halo2.setAttribute("stroke-width", "0.8");
      svg.appendChild(halo2);
      var txt = svgNS("text");
      txt.setAttribute("x", xt.x*100); txt.setAttribute("y", (1-xt.y)*100+1.8);
      txt.setAttribute("text-anchor", "middle"); txt.setAttribute("font-size", "6.5");
      txt.setAttribute("fill", estCor);
      txt.textContent = "★";
      svg.appendChild(txt);

      // ---- Distancias ordenadas ----
      distsEl.innerHTML = "";
r.ordenados.forEach(function(v, ord){
  var dentroK = ord < k;
  var row = document.createElement("div");
  row.style.cssText = "display:flex;justify-content:space-between;align-items:center;padding:2px 6px;border-radius:6px;" +
    (dentroK ? "background:"+CORES[v.p.cls]+"22;border:1px solid "+CORES[v.p.cls]+";" : "background:#f9fafb;border:1px solid #f1f1f1;color:#9ca3af;");
  row.innerHTML =
    '<span style="display:flex;align-items:center;gap:4px;">' +
      (dentroK ? '✓' : '\u00A0') +
      '<span title="posizione di lettura nella lista originale di addestramento — usata per il pareggio quando due distanze sono uguali" ' +
        'style="background:#eee;color:#9ca3af;border-radius:3px;padding:0 3px;font-size:8.5px;cursor:help;">#' + (v.i+1) + '</span>' +
      ' ' + v.p.nome + ' <span style="color:'+CORES[v.p.cls]+';font-weight:700;">('+v.p.cls+')</span>' +
    '</span>' +
    '<span>d='+v.d.toFixed(4)+'</span>';
  distsEl.appendChild(row);
});

      // ---- Votacao ----
      votosEl.innerHTML = "";
      classes.forEach(function(c){
        var venceu = (c === r.pred);
        var empatou = r.empatados.length > 1 && r.empatados.indexOf(c) !== -1;
        var div = document.createElement("div");
        div.style.cssText = "text-align:center;border-radius:10px;padding:8px 14px;font-size:12px;" +
          (venceu ? "background:"+CORES[c]+"22;border:2px solid "+CORES[c]+";" : "background:#f9fafb;border:1px solid #e5e7eb;color:#9ca3af;");
        div.innerHTML = '<div style="font-weight:700;color:'+CORES[c]+';">'+c+'</div><div style="font-size:16px;font-weight:700;">'+r.votos[c]+'</div>' +
          (empatou ? '<div style="font-size:9px;color:#b91c1c;">empate</div>' : '');
        votosEl.appendChild(div);
      });
      var msgEmpate = r.empatados.length > 1 ? " (empate entre "+r.empatados.join(", ")+" — desempate pela ordem da lista de classes)" : "";
      previsaoEl.innerHTML = 'Classe prevista: <span style="color:'+CORES[r.pred]+';">'+r.pred+'</span>' + msgEmpate +
        ' &nbsp;|&nbsp; classe real: <span style="color:'+CORES[xt.cls]+';">'+xt.cls+'</span> ' + (correto ? '✅' : '❌');

      // ---- Matriz de confusao + acuracia sobre as 3 amostras de teste ----
      var cm = [[0,0,0],[0,0,0],[0,0,0]];
      var acertos = 0;
      var predsGlobais = [];
      testPts.forEach(function(tp){
        var rr = knnPredict(tp);
        predsGlobais.push(rr.pred);
        var iReal = classes.indexOf(tp.cls);
        var iPrev = classes.indexOf(rr.pred);
        cm[iReal][iPrev]++;
        if(rr.pred === tp.cls) acertos++;
      });
      var acc = acertos/testPts.length;

      var thead = '<tr><td></td>' + classes.map(function(c){ return '<td style="padding:4px 8px;color:'+CORES[c]+';font-weight:700;">'+c.slice(0,4)+'</td>'; }).join('') + '</tr>';
      var rows = classes.map(function(cReal, i){
        var cells = classes.map(function(cPrev, j){
          var v = cm[i][j];
          var diag = (i===j);
          var bg = v===0 ? '#fff' : (diag ? '#dcfce7' : '#fee2e2');
          return '<td style="padding:4px 10px;text-align:center;border:1px solid #e5e7eb;background:'+bg+';">'+v+'</td>';
        }).join('');
        return '<tr><td style="padding:4px 8px;color:'+CORES[cReal]+';font-weight:700;">'+cReal.slice(0,4)+'</td>'+cells+'</tr>';
      }).join('');
      cmEl.innerHTML = thead + rows;
      accEl.textContent = "Accuratezza: " + acc.toFixed(4) + " (" + acertos + "/" + testPts.length + ")";

      dbg.textContent = "M="+metrica+" k="+k+" | teste_sel="+xt.nome+" | y_pred(todas)=["+predsGlobais.join(", ")+"]";
    }

    be.addEventListener("click", function(){ metrica="euclidiana"; render(); });
    bm.addEventListener("click", function(){ metrica="manhattan"; render(); });
    bk1.addEventListener("click", function(){ k=1; render(); });
    bk3.addEventListener("click", function(){ k=3; render(); });
    bk5.addEventListener("click", function(){ k=5; render(); });
    bt0.addEventListener("click", function(){ testSel=0; render(); });
    bt1.addEventListener("click", function(){ testSel=1; render(); });
    bt2.addEventListener("click", function(){ testSel=2; render(); });

    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0706");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.26:** Simulatore EP07_06: *Pipeline* k-NN Multi-Classe (votazione, spareggio e matrice di confusione)


<figure id="fig-07-sim-ep0706">
  <img src="imagens/fig-07-sim-ep0706.png" alt=" Simulatore EP07_06: *Pipeline* k-NN Multi-Classe (votazione, spareggio e matrice di confusione) " style="max-width:80%" />
  <figcaption><strong>Figura 7.26:</strong>  Simulatore EP07_06: *Pipeline* k-NN Multi-Classe (votazione, spareggio e matrice di confusione) </figcaption>
</figure>

In [37]:
%%writefile EP07_06.py
# Codice Python

Overwriting EP07_06.py


In [38]:
TestSuite("EP07_06.py").run()

### 7.0.7 EP07_07 ⚫ Classificazione Reale di un Mosaico di Trame tramite LBP + k-NN

Negli esercizi precedenti, il descrittore LBP (**EP07_04**) e il classificatore k-NN multiclasse (**EP07_06**) sono stati studiati separatamente, sempre a partire da dati già forniti in input — vicinanze $3\times3$ isolate o istogrammi precedentemente estratti. In questo esercizio conclusivo del capitolo, il programma dovrà **leggere un'immagine reale**, nel formato **PGM ASCII (P2)**, calcolare il descrittore LBP direttamente dai pixel e, successivamente, classificare ogni regione tramite il k-NN, riproducendo, in scala ridotta, il flusso completo di un sistema di riconoscimento delle trame. Questo approccio anticipa anche l'idea di **classificazione per mosaico di regioni**, legata alla segmentazione semantica studiata in un capitolo successivo.

Il simulatore interattivo dell'**EP07_06** utilizzava solo tre classi (`granulare`, `a righe` e `maculata`) rappresentate da punti bidimensionali fittizi. In questo esercizio, si aggiunge una quarta classe, **a scacchi**, e i punti vengono sostituiti da istogrammi LBP estratti da un'immagine reale.

L'immagine di input è un **mosaico** formato da una griglia $G\times G$ di blocchi quadrati di $S\times S$ pixel. Ogni blocco contiene un campione di una delle quattro classi di trama sintetica del capitolo: **granulare**, **a righe**, **maculata** o **a scacchi** (motivo a scacchiera con intensità alternate). Come negli altri esercizi del libro, il caricamento dell'immagine viene eseguito dalla funzione didattica `mm.readImg`.

> ### 💡 Perché un mosaico unico, e non più immagini?
>
> L'input riunisce i $G \times G$ campioni di trama in un unico file **PGM**, solo per semplificare la lettura dei dati ed evitare l'apertura di più file. Per l'algoritmo, ciò non modifica l'elaborazione: ogni blocco viene trattato in modo indipendente, come se fosse un'immagine isolata.
> L'unica eccezione è l'**esclusione del bordo** (punto 4 di seguito).

#### 7.0.7.1 📋 Linee Guida di Implementazione

1. **Lettura delle dimensioni dell'immagine**

   Leggere, tramite l'input standard, due righe contenenti, rispettivamente, il numero di righe $L$ e il numero di colonne $C$ del mosaico (entrambi multipli della dimensione del blocco $S$, con $L=C$).

2. **Caricamento dell'immagine**

   Utilizzare la funzione didattica

   ```python
   f = mm.readImg(L, C)
   ```

   per leggere i valori di intensità $L \times C$ (toni di grigio, `uint8`) del mosaico.

3. **Parametri della griglia**

   Leggere l'intero $G$ (numero di blocchi per lato) e l'intero $S$ (dimensione del lato di ogni blocco, in pixel), soddisfacendo $L = C = G \times S$.

4. **Calcolo del codice LBP per pixel**

   Per ogni pixel **interno** dell'immagine (cioè che non si trova sul bordo globale di `f` — riga o colonna $0$ o $L-1$/$C-1$), calcolare il codice LBP con $P=8$ vicini e raggio $R=1$, percorrendo i vicini in senso **orario** a partire dall'angolo superiore sinistro, esattamente come nell'EP07_04: `[riga-1][colonna-1]`, `[riga-1][colonna]`, `[riga-1][colonna+1]`, `[riga][colonna+1]`, `[riga+1][colonna+1]`, `[riga+1][colonna]`, `[riga+1][colonna-1]`, `[riga][colonna-1]`.

   I pixel sul bordo globale dell'immagine **non** hanno una vicinanza completa e devono essere **ignorati** (non contribuiscono a nessun istogramma). Questo include i pixel di bordo che cadono all'interno di un blocco (l'esclusione è sempre relativa al bordo dell'intera immagine, non al bordo di ogni singolo blocco).

5. **Istogramma LBP uniforme per blocco (10 contenitori)**

   Per ogni blocco $(i,j)$ della griglia ($i,j = 0,\ldots,G-1$), accumulare, tra i suoi pixel validi (punto 4), un istogramma $H^{(i,j)}$ di $10$ contenitori:

   * Considerando la sequenza circolare di bit $s_0,\ldots,s_7$ del pixel (stessa regola di transizioni dell'EP07_04): se il numero di transizioni è $\le 2$ (pattern **uniforme**), il pixel contribuisce al contenitore $\operatorname{popcount}(s_0,\ldots,s_7) \in \{0,\ldots,8\}$ (numero di bit uguali a `1`);
   * In caso contrario (pattern **non uniforme**), il pixel contribuisce al contenitore $9$.

   Alla fine, normalizzare l'istogramma di ogni blocco dividendolo per il numero di pixel validi in esso contenuti, ottenendo $\hat H^{(i,j)}$, con $\sum_{b=0}^{9} \hat H^{(i,j)}[b] = 1$.

6. **Prototipi di addestramento**

   Leggere l'intero $Ncl$ (numero di classi) seguito da $Ncl$ nomi di classe (ordine che definisce la matrice di confusione e il criterio di parità per la votazione, come nell'EP07_06); successivamente, leggere la stringa $M$ (metrica: `euclidiana` o `manhattan`) e l'intero dispari $k$; infine, leggere l'intero $N$ (numero di prototipi) e, per ciascuno, il nome della classe seguito da $10$ valori reali (istogramma prototipo già normalizzato).

7. **Classificazione k-NN di ogni blocco**

   Per ogni blocco, calcolare la distanza di $\hat H^{(i,j)}$ da ciascuno degli $N$ prototipi, usando la metrica $M$ (stesse formule dell'EP07_06). Selezionare i $k$ prototipi più vicini (criterio di parità per la distanza basato sull'ordine di lettura dei prototipi) e classificare tramite la classe maggioritaria (criterio di parità per la votazione basato sull'ordine delle classi del punto 6).

8. **Etichette reali e valutazione**

   Leggere, in un'unica riga, i $G \times G$ nomi di classe **reali** di ogni blocco, in ordine di lettura per riga della griglia (blocco $(0,0)$, $(0,1)$, …, $(0,G-1)$, $(1,0)$, …). Costruire la matrice di confusione $Ncl \times Ncl$ (riga = classe reale, colonna = classe prevista) e calcolare l'accuratezza globale.

9. **Output**

   Stampare, per ogni blocco (nello stesso ordine di lettura delle etichette reali del punto 8), la classe prevista. Successivamente, stampare la matrice di confusione (una riga per classe reale, nell'ordine del punto 6). Infine, stampare l'accuratezza, arrotondata a 4 cifre decimali.

#### 7.0.7.2 📌 Vincoli Computazionali

* **Descrittore fisso:** $P=8$, $R=1$ e $10$ contenitori (come da punto 5) sono fissi in questo esercizio — non vengono letti dall'input.
* **Esclusione del bordo globale, non del blocco:** un pixel sul confine tra due blocchi, ma all'interno dell'immagine, è valido e contribuisce normalmente all'istogramma del blocco a cui appartiene.
* **Ordine di lettura come criterio di parità:** sia il criterio di parità per la distanza (punto 7) che quello per la votazione (punto 7) seguono esattamente le stesse convenzioni dell'EP07_01 e dell'EP07_06.
* **Prototipi come input, non appresi:** a differenza del Progetto Pratico 2, gli istogrammi di addestramento vengono forniti direttamente nell'input; il programma non deve generare trame sintetiche.

#### 7.0.7.3 🧠 Fondamenti Teorici

| Fase dell'esercizio | Fase corrispondente nel capitolo |
|---|---|
| Lettura dell'immagine tramite `mm.readImg` | Acquisizione dell'immagine nel *pipeline* di riconoscimento di pattern |
| Codice LBP per pixel (EP07_04) | `local_binary_pattern(immagine, P=8, R=1, method="uniform")` |
| Istogramma di 10 contenitori per blocco | Funzione `descrittor_lbp` del Progetto Pratico 2 (`bins=10`, `range=(0, P+2)`) |
| Classificazione k-NN con metrica selezionabile (EP07_06) | `KNeighborsClassifier` addestrato su `X_texture` |
| Matrice di confusione $Ncl\times Ncl$ e accuratezza | `confusion_matrix` e `accuracy_score` su `yt_test` |

Questo esercizio evidenzia, con pixel reali invece di valori sintetici, una limitazione discussa nella sezione finale del capitolo: classi di trama visivamente distinte per un osservatore umano — come **granulare** e **maculata** — possono produrre istogrammi LBP simili quando la vicinanza considerata è piccola ($R=1$), poiché entrambe presentano un'alta frequenza di pattern non uniformi alla scala di un singolo pixel. La classe **a scacchi**, invece, avendo bordi regolari e ripetitivi, tende a essere separata con maggiore facilità. Ci si aspetta che la matrice di confusione prodotta rifletta esattamente questo pattern di confusione parziale.

#### 7.0.7.4 📦 Specifica di Input e Output (VPL)

**Input:**

```
L
C
[matrice L x C dell'immagine]
G S
Ncl nome_classe_1 ... nome_classe_Ncl
M k
N
nome_classe h0 h1 ... h9      (ripetuta N volte)
etichetta(0,0) etichetta(0,1) ... etichetta(G-1,G-1)
```

**Output:**

* $G \times G$ righe con la classe prevista per ogni blocco, nell'ordine di lettura della griglia.
* $Ncl$ righe con la matrice di confusione (una riga per classe reale, valori separati da spazi).
* Ultima riga: `Acuracia: <valore>`.

#### 7.0.7.5 📌 Esempio (verifica manuale)

Per verificare l'implementazione del descrittore prima di testarla su un mosaico completo, si consideri un'immagine $6\times6$ **omogenea**, con tutti i pixel di intensità $100$, trattata come un unico blocco ($G=1$, $S=6$). Poiché ogni pixel interno ha gli 8 vicini con intensità uguale a quella del centro ($g_p \ge g_c$ in tutti i casi), tutti i bit $s_p$ valgono `1`, il numero di transizioni è $0$ (uniforme) e il contenitore è $\operatorname{popcount}(11111111)=8$. L'istogramma dell'unico blocco è, quindi, `0 0 0 0 0 0 0 0 1 0`.

| Input (riepilogo) | Output | Osservazione |
|---|---|---|
| 6<br>6<br>[36 valori uguali a 100]<br>1 6<br>2 uniforme altro<br>euclidiana 1<br>2<br>uniforme 0 0 0 0 0 0 0 0 1 0<br>altro 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1<br>uniforme | uniforme<br>1 0<br>0 0<br>Acuracia: 1.0000 | La distanza del blocco al prototipo `uniforme` è esattamente $0$; la classe `altro` non appare nell'etichetta reale, quindi la sua riga nella matrice di confusione è nulla. |

#### 7.0.7.6 📌 File di Riferimento (.pgm)

Per il debug locale, due mosaici di test nel formato ASCII P2 sono messi a disposizione (allegati a questa consegna; quando li si integra nel repository del capitolo, salvarli in `all/cap07/dati/EP07/`):

* 📥 **Caso 1 — Mosaico semplice (`Caso1_Mosaico_Simples.pgm`)**: griglia $2\times2$ di blocchi di $24\times24$ pixel, un campione di ciascuna delle quattro classi, con basso rumore — utile per validare la lettura dell'immagine e la logica di classificazione in uno scenario controllato.
* 📥 **Caso 2 — Mosaico misto (`Caso2_Mosaico_Misto.pgm`)**: griglia $3\times3$ di blocchi di $16\times16$ pixel, con classi ripetute e maggiore variabilità — scenario in cui la confusione tra **granulare** e **maculata** discussa nei Fondamenti Teorici tende a manifestarsi.

La [Figura 7.27](#fig-07-ep07) mostra i due mosaici, per un'ispezione visiva prima dell'implementazione.

In [39]:
import os
import urllib.request
import numpy as np

def garantir_e_baixar_arquivo(nome_arquivo):
    diretorio_local = "dados/EP07"
    caminho_local = os.path.join(diretorio_local, nome_arquivo)
    
    # Creare la directory locale se non esiste
    if not os.path.exists(diretorio_local):
        os.makedirs(diretorio_local)
        
    # Se il file non esiste localmente, lo scarica dal repository remoto
    if not os.path.exists(caminho_local):
        url_base = "https://raw.githubusercontent.com/fzampirolli/"
        url_base += "pdi-vc/master/all/cap07/dados/EP07"
        url_arquivo = f"{url_base}/{nome_arquivo}"
        print(f"Scaricamento {nome_arquivo} da GitHub...")
        try:
            urllib.request.urlretrieve(url_arquivo, caminho_local)
        except Exception as e:
            raise IOError(f"Erro ao baixar {nome_arquivo} do GitHub. ",
                          "Verifique a conexão ou a URL. Detalhes: {e}")
            
    return caminho_local

def ler_pgm_p2(caminho):
    with open(caminho) as f:
        linhas = [l for l in f.read().split() if l]
    assert linhas[0] == "P2"
    C, L = int(linhas[1]), int(linhas[2])
    maxv = int(linhas[3])
    valores = list(map(int, linhas[4:4 + L * C]))
    return np.array(valores, dtype=np.uint8).reshape(L, C)

# Garantisce il download e ottiene il percorso corretto
arq_caso1 = garantir_e_baixar_arquivo("Caso1_Mosaico_Simples.pgm")
arq_caso2 = garantir_e_baixar_arquivo("Caso2_Mosaico_Misto.pgm")

# Legge le matrici PGM
caso1 = ler_pgm_p2(arq_caso1)
caso2 = ler_pgm_p2(arq_caso2)

mm.show(
    [caso1, caso2],
    titles=[
        "Caso 1: Mosaico Semplice\n(blocchi 2x2, 1 campione/classe)",
        "Caso 2: Mosaico Misto\n(blocchi 3x3, classi ripetute)",
    ],
    cols=2,
    figsize=(8, 4),
)

<Figure size 1200x600 with 2 Axes>

**Figura 7.27:** Mosaici di riferimento (formato PGM ASCII) usati nell


In [40]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML('''
<div id="sim-ep0707" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">  
<style>
  #sim-ep0707 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0707 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #ede6d8; background: #f3efe6; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0707 button:hover { background: #e8e0cf; }
  #sim-ep0707 button.sim-ep0707_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0707_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #ede6d8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0707_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulatore EP07_07: Classificazione del Mosaico tramite LBP + k-NN</span>
  <span class="sim-ep0707_pill">⚫ pipeline completo</span>
</div>


  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
Mosaico 3x3 di blocchi 12x12 (L=C=36). LBP (P=8,R=1) calcolato pixel per pixel, con esclusione del bordo globale.      Regola k e la metrica e osserva la classificazione di ciascun blocco rispetto a 8 prototipi (2 per classe).
   
   </p>
     
<div style="background:#fff3cd;border:1px solid #ffe69c;border-radius:8px;padding:8px 12px;margin-bottom:12px;font-size:11px;color:#7a5c00;">
  ⚠️ Texture sintetiche generate da codice, non i file .pgm reali di EP07_07. Usa questo simulatore per capire il flusso dell'algoritmo, non come riferimento di difficoltà tra le classi.
</div>
     
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;display:flex;gap:24px;flex-wrap:wrap;align-items:center;">
      <div style="flex:1;min-width:180px;">
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
          <label style="font-size:12px;font-weight:bold;color:#2980b9;">k (numero di vicini)</label>
          <span id="ep0707_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">1</span>
        </div>
<input id="ep0707_sl" style="width:100%;accent-color:#2980b9;" max="5" min="1" step="2" type="range" value="1">
      </div>
      <div>
        <label style="font-size:12px;font-weight:bold;color:#2980b9;display:block;margin-bottom:6px;">Metrika</label>
        <select id="ep0707_metric" style="font-size:12px;padding:4px 8px;border-radius:6px;border:1px solid #ccc;">
          <option value="euclidiana">euclidea</option>
          <option value="manhattan">manhattan</option>
        </select>
      </div>
    </div>

    <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:flex-start;">
      <canvas id="ep0707_canvas" style="border-radius:8px;border:1px solid #ccc;"></canvas>
      <div id="ep0707_grid" style="flex:1;min-width:220px;display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <div id="ep0707_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;white-space:pre-line;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var S = 12, G = 3, L = G * S, SCALE = 5;   // bloco maior reduz o vazamento de borda; SCALE ajustado p/ manter o canvas ~180px
    var classesOrder = ["granular", "listrada", "manchada", "xadrez"];
    // Grade 3x3 com classes repetidas, análoga ao Caso 2 do enunciado
    var layout = [
      "granular", "listrada", "manchada",
      "xadrez",   "granular", "manchada",
      "listrada", "xadrez",   "granular"
    ];

    // --- Geração determinística de textura por pixel (didática, não os PGMs reais) ---
    function h(a, b, phase){
      var v = Math.sin((a + phase) * 12.9898 + (b + phase * 0.7) * 78.233 + phase * 3.1) * 43758.5453;
      return v - Math.floor(v);
    }
    function texturePixel(cls, r, c, phase){
      phase = phase || 0;
      switch(cls){
        case "granular": return h(r, c, phase) < 0.5 ? 220 : 30;
        case "listrada": return ((c + Math.floor(phase * 2)) % 4) < 2 ? 220 : 30;
        case "manchada": return h(Math.floor(r / 3), Math.floor(c / 3), phase) < 0.5 ? 200 : 60;
        case "xadrez":   return ((Math.floor(r / 2) + Math.floor(c / 2)) % 2 === 0) ? 230 : 20;
      }
    }

    function buildImage(){
      var img = [];
      for(var r = 0; r < L; r++){
        var row = [];
        for(var c = 0; c < L; c++){
          var bi = Math.floor(r / S), bj = Math.floor(c / S);
          row.push(texturePixel(layout[bi * G + bj], r, c, 0));
        }
        img.push(row);
      }
      return img;
    }

    // --- LBP: P=8, R=1, sentido horário, s_p = 1 se vizinho >= centro ---
    function lbpBin(patch, r, c){
      var center = patch[r][c];
      var neigh = [
        patch[r-1][c-1], patch[r-1][c], patch[r-1][c+1],
        patch[r][c+1],
        patch[r+1][c+1], patch[r+1][c], patch[r+1][c-1],
        patch[r][c-1]
      ];
      var bits = neigh.map(function(v){ return v >= center ? 1 : 0; });
      var trans = 0;
      for(var i = 0; i < 8; i++){ if(bits[i] !== bits[(i+1) % 8]) trans++; }
      if(trans <= 2) return bits.reduce(function(a,b){ return a+b; }, 0); // popcount 0..8
      return 9; // não uniforme
    }

    // Histograma de um patch isolado (usado para gerar protótipos), excluindo apenas a borda do patch
    function computeLBPHist(patch){
      var n = patch.length, m = patch[0].length;
      var hist = new Array(10).fill(0), count = 0;
      for(var r = 1; r < n - 1; r++){
        for(var c = 1; c < m - 1; c++){
          hist[lbpBin(patch, r, c)]++;
          count++;
        }
      }
      for(var k = 0; k < 10; k++) hist[k] = count > 0 ? hist[k] / count : 0;
      return hist;
    }

    // Histogramas por bloco da imagem completa, excluindo só a borda global (item 4/5 do enunciado)
    function computeMosaicHistograms(img){
      var hists = [], counts = [];
      for(var i = 0; i < G*G; i++){ hists.push(new Array(10).fill(0)); counts.push(0); }
      for(var r = 1; r < L - 1; r++){
        for(var c = 1; c < L - 1; c++){
          var bin = lbpBin(img, r, c);
          var idx = Math.floor(r/S) * G + Math.floor(c/S);
          hists[idx][bin]++;
          counts[idx]++;
        }
      }
      for(var b = 0; b < hists.length; b++){
        for(var k = 0; k < 10; k++) hists[b][k] = counts[b] > 0 ? hists[b][k] / counts[b] : 0;
      }
      return hists;
    }

    // --- Protótipos: 2 por classe (N=8), ordem de leitura fixa (usada no desempate) ---
    var prototypes = [];
    classesOrder.forEach(function(cls){
      [0, 5].forEach(function(phase){
        var Sp = S + 2, patch = [];
        for(var r = 0; r < Sp; r++){
          var row = [];
          for(var c = 0; c < Sp; c++) row.push(texturePixel(cls, r, c, phase));
          patch.push(row);
        }
        prototypes.push({ classe: cls, hist: computeLBPHist(patch) });
      });
    });

    function dist(u, v, metric){
      var s = 0;
      for(var i = 0; i < u.length; i++){
        s += metric === "euclidiana" ? (u[i]-v[i])*(u[i]-v[i]) : Math.abs(u[i]-v[i]);
      }
      return metric === "euclidiana" ? Math.sqrt(s) : s;
    }

    // Desempate de distância: ordem de leitura dos protótipos. Desempate de votação: ordem das classes.
    function classify(hist, k, metric){
      var cand = prototypes.map(function(p, idx){ return { classe: p.classe, d: dist(hist, p.hist, metric), idx: idx }; });
      cand.sort(function(a, b){ return a.d !== b.d ? a.d - b.d : a.idx - b.idx; });
      var viz = cand.slice(0, k);
      var votos = {};
      viz.forEach(function(v){ votos[v.classe] = (votos[v.classe] || 0) + 1; });
      var melhor = null, melhorN = -1;
      classesOrder.forEach(function(c){
        var n = votos[c] || 0;
        if(n > melhorN){ melhorN = n; melhor = c; }
      });
      return melhor;
    }

    var canvas = root.querySelector('#ep0707_canvas');
    canvas.width = L * SCALE; canvas.height = L * SCALE;
    var ctx = canvas.getContext('2d');
    var slK = root.querySelector('#ep0707_sl');
    var vlK = root.querySelector('#ep0707_vl');
    var selMetric = root.querySelector('#ep0707_metric');
    var gridEl = root.querySelector('#ep0707_grid');
    var dbg = root.querySelector('#ep0707_debug');

    var img = buildImage();
    var hists = computeMosaicHistograms(img);

    function render(){
      var k = parseInt(slK.value);
      var metric = selMetric.value;
      vlK.textContent = k;

      var preds = [];
      for(var idx = 0; idx < G*G; idx++) preds.push(classify(hists[idx], k, metric));

      var confusion = classesOrder.map(function(){ return new Array(classesOrder.length).fill(0); });
      var acertos = 0;
      for(var i2 = 0; i2 < G*G; i2++){
        var ri = classesOrder.indexOf(layout[i2]);
        var pi = classesOrder.indexOf(preds[i2]);
        confusion[ri][pi]++;
        if(layout[i2] === preds[i2]) acertos++;
      }
      var acc = acertos / (G*G);

      // Desenha a imagem real em tons de cinza
      for(var r = 0; r < L; r++){
        for(var c = 0; c < L; c++){
          var v = img[r][c];
          ctx.fillStyle = 'rgb(' + v + ',' + v + ',' + v + ')';
          ctx.fillRect(c*SCALE, r*SCALE, SCALE, SCALE);
        }
      }
      // Contorna cada bloco: verde = acerto, vermelho = erro
      for(var idx3 = 0; idx3 < G*G; idx3++){
        var bi = Math.floor(idx3 / G), bj = idx3 % G;
        ctx.strokeStyle = (preds[idx3] === layout[idx3]) ? '#10b981' : '#f43f5e';
        ctx.lineWidth = 2;
        ctx.strokeRect(bj*S*SCALE + 1, bi*S*SCALE + 1, S*SCALE - 2, S*SCALE - 2);
      }

      // Grade textual de apoio
      gridEl.innerHTML = '';
      for(var idx4 = 0; idx4 < G*G; idx4++){
        var ok = preds[idx4] === layout[idx4];
        var card = document.createElement('div');
        card.style.cssText = 'border-radius:8px;padding:6px;text-align:center;font-size:10px;border:2px solid ' + (ok ? '#10b981' : '#f43f5e') + ';';
        card.innerHTML = 'Real: ' + layout[idx4] + '<br><b style="color:' + (ok ? '#059669' : '#e11d48') + '">Pred: ' + preds[idx4] + (ok ? ' ✅' : ' ❌') + '</b>';
        gridEl.appendChild(card);
      }

      // Saída no mesmo formato do programa (itens 7-9 do enunciado)
      var linhas = [];
      linhas.push('Classes preditas (ordem de leitura da grade):');
      linhas.push(preds.join(' '));
      linhas.push('');
      linhas.push('Matriz de confusão (linhas=real, colunas=predita; ordem ' + classesOrder.join(',') + '):');
      confusion.forEach(function(lin){ linhas.push(lin.join(' ')); });
      linhas.push('');
      linhas.push('Acuracia: ' + acc.toFixed(4));
      dbg.textContent = linhas.join('\\n');
    }

    slK.addEventListener('input', render);
    selMetric.addEventListener('change', render);
    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0707');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.28:** Simulatore EP07_07: Classificazione di un Mosaico di Trame tramite LBP + k-NN


<figure id="fig-07-sim-ep0707">
  <img src="imagens/fig-07-sim-ep0707.png" alt=" Simulatore EP07_07: Classificazione di un Mosaico di Trame tramite LBP + k-NN " style="max-width:80%" />
  <figcaption><strong>Figura 7.28:</strong>  Simulatore EP07_07: Classificazione di un Mosaico di Trame tramite LBP + k-NN </figcaption>
</figure>

In [41]:
%%writefile EP07_07.py
# Codice Python

Overwriting EP07_07.py


In [42]:
TestSuite("EP07_07.py").run()

## Referências do Capítulo


COVER, T.; HART, P. **Nearest neighbor pattern classification**. 1967.

DALAL, N.; TRIGGS, B. **Histograms of oriented gradients for human detection**. 2005.

DUDA, Richard O.; HART, Peter E.; STORK, David G. **Pattern Classification**. New York, Wiley-Interscience, 2001.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

OJALA, Timo; PIETIK\"{A}INEN, Matti; M\"{A}ENP\"{A}\"{A}, Topi. **Multiresolution Gray-Scale and Rotation Invariant Texture Classification with Local Binary Patterns**. USA, IEEE Computer Society, 2002.

PEDREGOSA, Fabian *et al*. **Scikit-learn: Machine Learning in Python**. 2011.

QUILICI-GONZALEZ, José Artur; ZAMPIROLLI, Francisco de Assis. **Sistemas Inteligentes e Mineração de Dados**. Editora UFABC, 2014.

QUILICI-GONZALEZ, José Artur; ZAMPIROLLI, Francisco de Assis; SOUZA, Fábio Rezende de. **Sistemas Inteligentes e Mineração de Dados: Do Weka ao Python**. 2026.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

{GOOGLE}. **{NotebookLM}**. 2025.